# Create Works Base

## Overview

This notebook creates `openalex_works_base`, the foundational table for the works pipeline. It collects and deduplicates data from multiple provenances (crossref, pubmed, datacite, repo, etc.) into a single row per work.

## Data Flow

```
locations_mapped (multiple rows per work, one per provenance)
    |
    v
openalex_works_base (one row per work, deduplicated)
    |
    v
CreateWorksEnriched (adds citations, topics, authorships, etc.)
    |
    v
openalex_works (final output)
```

## Key Transformations

### Field Deduplication
Fields are deduplicated using priority-based selection via `get_highest_priority_value()`:
- Priority order defined in `openalex.system.priority_table`
- For each field, the value from the highest-priority provenance is selected
- Abstracts: one record gives both the text and the inverted index; a published version's Crossref or PubMed record
  outranks a preprint's (unless it is a fragment), then provenance priority as before (oxjob #2142)

### Locations Collection
All locations (URLs, sources) are collected into an array, deduplicated by landing_page_url, and sorted by:
1. Version (publishedVersion > acceptedVersion > submittedVersion)
2. URL quality score (europepmc > pmc > arxiv > .edu)
3. Provenance priority

### Authorships
Authors are deduplicated by selecting from the highest-priority provenance, then enriched with:
- Legacy affiliations from `works_legacy.raw_affiliation_strings`
- Legacy `is_corresponding` from `works_legacy.work_authors`

### Curation Requests
Location curations are laid over `locations_mapped` upstream, in the view `locations_mapped_curated`
(`Apply_Location_Curations`, oxjob #1170), which this notebook reads. The one curation rule here: a field in a
location's `curated_fields` is used as-is, so a curated `is_oa` or licence wins over the journal rules
(`source_is_oa`, `doaj_license`).

## References Handling

**Important:** This notebook only collects pre-resolved `referenced_works` (work IDs from legacy/MAG data).
It does NOT collect raw `references` data - that is handled separately by `parse_work_references.ipynb`.

See `qa/issues/open/references-pipeline-refactor-2026-01/` for planned improvements to this flow.

### Create `openalex_works_base` table

In [ ]:
-- oxjob #1910: a work title's control characters (865K titles: 803K MARC non-sort marks from the Bavarian library
-- catalogue feed, Windows-1252 bytes in publisher deposits, mojibake) are removed without losing what they stood for.
-- The API's sanitize_name already DROPS them at serve time, which deletes the apostrophe in Newton\x92s: this restores it.
-- Source of truth and tests: oxjobs #1910 work/title_controls.py (this body is generated from it by build_udf_sql.py).
-- Revert: remove this cell, the next one, and the three #1910 edits in the main statement (title, topics_key, join).
CREATE OR REPLACE FUNCTION openalex.common.clean_title_controls(s STRING)
RETURNS STRING
LANGUAGE PYTHON
DETERMINISTIC
COMMENT 'oxjob #1910: removes C0/C1 control characters from a work title without losing what they stood for (MARC non-sort marks dropped, exact mojibake repaired, a lone Windows-1252 byte read as its character where it fits, else dropped as the API already does). Source + tests: oxjobs #1910 work/title_controls.py'
AS $$
import re
import unicodedata

_MAX_ROUNDS = 4  # corpus has strings mojibaked up to ~5 levels deep, each round strictly shrinks

# unified latin-1 + cp1252 reverse map: chars <= U+00FF map to their own codepoint
# (latin-1), cp1252's specials (> U+00FF) map to the 0x80-0x9F byte cp1252 decoded them from.
_CP1252_REV = {}
for _b in range(0x80, 0xA0):
    try:
        _CP1252_REV[bytes([_b]).decode("cp1252")] = _b
    except UnicodeDecodeError:
        pass  # 0x81 0x8D 0x8F 0x90 0x9D are undefined in cp1252

# Core leads: chars whose latin-1 byte is a UTF-8 lead byte for scripts that occur in real text.
_CORE_LEADS_2 = set("ÃÂÄÅÌÎÏÐÑ")  # Ì (CC) covers NFD combining-mark mojibake ("MuÌnster")
# Peripheral leads: may participate in repair but never trigger it alone (kills the
# Ö/Ü + typographic-NBSP legit-text corruption risk).
_PERIPHERAL_LEADS_2 = set("ÆÇÒÓÔÕÖØÙÚÛÜ")


def _is_cont(ch):
    """Char maps (via unified map) to a UTF-8 continuation byte 0x80-0xBF?"""
    cp = ord(ch)
    if 0x80 <= cp <= 0xBF:
        return True
    return _CP1252_REV.get(ch, 0) >= 0x80  # cp1252 specials are all 0x80-0x9F


def _detect(s):
    """True if s contains at least one CORE mojibake sequence (latin-1/cp1252 read)."""
    if not s:
        return False
    n = len(s)
    for i, ch in enumerate(s):
        if ch in _CORE_LEADS_2 and i + 1 < n and _is_cont(s[i + 1]):
            return True
        if ch == "â" and i + 2 < n and _is_cont(s[i + 1]) and _is_cont(s[i + 2]):
            return True
        cp = ord(ch)
        # 3-byte leads à-á ã-ï (skip â handled above) need two continuations, at least one
        # not NBSP: legit "Umeå\xa0\xa0University" is byte-valid UTF-8 for a rare CJK char.
        if (0xE0 <= cp <= 0xEF and cp != 0xE2 and i + 2 < n
                and _is_cont(s[i + 1]) and _is_cont(s[i + 2])
                and not (s[i + 1] == "\xa0" and s[i + 2] == "\xa0")):
            return True
        # 4-byte leads ð-ô need three continuations (same NBSP rule)
        if (0xF0 <= cp <= 0xF4 and i + 3 < n
                and _is_cont(s[i + 1]) and _is_cont(s[i + 2]) and _is_cont(s[i + 3])
                and not (s[i + 1] == s[i + 2] == s[i + 3] == "\xa0")):
            return True
    return False


def _fallback_signature(s, codec):
    """True if s carries a DISTINCTIVE lead+continuation pair under a Central-European
    single-byte codec (ISO-8859-2 / cp1250): UTF-8 lead bytes C3/C4/C5 decode to Ă/Ä/Ĺ there.
    'Distinctive' = the lead char maps to a byte != its own codepoint, so only that codec's
    read produces it — without this, latin2's É (0xC9, == latin-1) reopens the
    'UNIVERSITÉ\\xa0PARIS' trap the core detector closed."""
    if not s:
        return False

    def byte_of(ch):
        try:
            return ch.encode(codec)[0]
        except (UnicodeEncodeError, IndexError):
            return None

    n = len(s)
    for i, ch in enumerate(s):
        b = byte_of(ch)
        if b is not None and 0xC2 <= b <= 0xDF and b != ord(ch) and i + 1 < n:
            b2 = byte_of(s[i + 1])
            if b2 is not None and 0x80 <= b2 <= 0xBF:
                return True
    return False


def _to_bytes_unified(s):
    out = bytearray()
    for ch in s:
        cp = ord(ch)
        if cp <= 0xFF:
            out.append(cp)
        else:
            b = _CP1252_REV.get(ch)
            if b is None:
                return None
            out.append(b)
    return bytes(out)


def _validate(s, raw):
    if raw is None:
        return None
    try:
        fixed = raw.decode("utf-8", errors="strict")
    except UnicodeDecodeError:
        return None
    if len(fixed) >= len(s) or "�" in fixed:
        return None
    return fixed


def _badness(s):
    """Count of chars implausible in repaired text (symbols, controls), picks among codec
    candidates — a wrong-codec 'repair' that still validates leaves stray symbols."""
    score = 0
    for ch in s:
        if ch in "\n\r\t":
            continue
        cat = unicodedata.category(ch)
        if cat.startswith("C"):
            score += 3
        elif cat.startswith("S"):
            score += 1
    return score


def _repair_once(s):
    """One round of repair, or None if not repairable. All candidate codecs are tried and the
    least-bad validated result wins (iso-8859-2/cp1250 signatures overlap, wrong codec can still
    yield valid-UTF-8 garbage, so first-match is unsafe)."""
    candidates = []
    if _detect(s):
        fixed = _validate(s, _to_bytes_unified(s))
        if fixed is not None:
            candidates.append(fixed)
    for codec in ("iso-8859-2", "cp1250"):
        if _fallback_signature(s, codec):
            try:
                raw = s.encode(codec, errors="strict")
            except UnicodeEncodeError:
                continue
            fixed = _validate(s, raw)
            if fixed is not None:
                candidates.append(fixed)
    if not candidates:
        return None
    return min(candidates, key=_badness)  # ties keep candidate order (min is stable)


def _fix_bare_c1(s):
    """Map surviving C1 controls (U+0080-U+009F) to their cp1252 chars."""
    if not any(0x80 <= ord(ch) <= 0x9F for ch in s):
        return s
    out = []
    for ch in s:
        if 0x80 <= ord(ch) <= 0x9F:
            try:
                ch = bytes([ord(ch)]).decode("cp1252")
            except UnicodeDecodeError:
                pass  # 0x81 0x8D 0x8F 0x90 0x9D undefined -> keep
        out.append(ch)
    return "".join(out)


def repair_mojibake(s):
    """Repair mojibake in s, returns s unchanged if not confidently repairable."""
    if not s:
        return s
    # oxjob #881: pure-ASCII text cannot contain mojibake -- skip the per-character detectors
    # (single C-level check, ~50% of abstracts, ~68% of titles). The broader trigger-lead regex
    # gate from b81da17e stays OUT pending a repro of the at-scale worker crash it shipped with.
    if s.isascii():
        return s
    cur = s
    for _ in range(_MAX_ROUNDS):
        nxt = _repair_once(cur)
        if nxt is None:
            break
        cur = nxt
    if cur is not s:
        cur = _fix_bare_c1(cur)
    return cur


CTRL = re.compile(r"[\x00-\x1f\x7f-\x9f]")
_MARC_PAIR = re.compile(r"\x98([^\x98\x9c]{0,40}?)\x9c")
_SEPARATORS = re.compile(r"[\t\n\v\f\r\x1c-\x1f]+")
_OTHER_CTRL = re.compile(r"[\x00-\x08\x0e-\x1b\x7f-\x9f]")
_CP1252_SPECIALS = set("€‚ƒ„…†‡ˆ‰Š‹ŒŽ"
                       "‘’“”•–—˜™š›œžŸ")
_PUNCT = {0x85, 0x91, 0x92, 0x93, 0x94, 0x95, 0x96, 0x97, 0x99}   # ... ' ' " " bullet en-dash em-dash TM
_LOW_QUOTES = {0x82, 0x84}                                      # single/double low-9 quotation marks
_ANGLE_QUOTES = {0x8B, 0x9B}                                    # single angle quotation marks
_DAGGERS = {0x86, 0x87}
_UPPER = {0x8A, 0x8C, 0x8E, 0x9F}                               # S-caron, OE, Z-caron, Y-diaeresis
_LOWER = {0x9A, 0x9C, 0x9E}                                     # s-caron, oe, z-caron
_QUOTES = {0x82, 0x84, 0x8B, 0x91, 0x92, 0x93, 0x94, 0x9B}
_ASCII_QUOTES = ("\"", "'")
_IN_WORD = {0x92, 0x96, 0x97, 0x9A, 0x9C, 0x9E}               # apostrophe, en/em dash, s-caron, oe, z-caron
# What follows a ligature slot (fi, fl, ff, ffi, ffl) in the PDF-extracted titles seen here
_LIGATURE_TAIL = re.compile(r"(?:cations?|cial|cien|nanc|nal|nit|nd|rst|eld|gur|xed|tness|ber|lm|uen|uid|ux|ow|ect|er\b|ers\b|ed\b)")
_CLOSING = re.compile("[\x91-\x94\u2018\u2019\u201c\u201d\"']")


def _ok_before(prev):
    """The char before a C1 char can't be a byte of mojibake: start, ASCII, or above U+00FF but not a cp1252 special."""
    if prev == "":
        return True
    cp = ord(prev)
    if 0x20 <= cp <= 0x7E:
        return True
    return cp > 0xFF and prev not in _CP1252_SPECIALS and not (0x80 <= cp <= 0x9F)


def _is_opening(prev):
    return prev == "" or prev.isspace() or prev in "([{/-"


def _cp1252_char(b, prev, nxt, nxt2, rest, closes_later):
    """Windows-1252 reading of C1 byte b in its context, or '' to drop it."""
    if not _ok_before(prev) or "\x80" <= nxt <= "\x9f":
        return ""  # only a lone C1 char is read: a run of them ('Parentesco\x99\x83\x8e\x8b') is garbage
    ch = bytes([b]).decode("cp1252", errors="ignore")
    p_letter, n_letter = prev.isalpha(), nxt.isalpha()
    if p_letter and n_letter:
        # Inside a word only an apostrophe, a compound dash and the Slavic/French letters fit, and never in front of
        # a ligature tail: PDF font slots put fi/fl/ff there ('Modi\x91cation', 'Brown\x9aeld', 'In\x99uences').
        if b not in _IN_WORD or _LIGATURE_TAIL.match(rest):
            return ""
    if b == 0x95:  # bullet: a list separator, never between two letters (PDF 'ff' slot: 'E\x95ects')
        return ch if not (p_letter and n_letter) else ""
    if b in _PUNCT:
        return ch
    if b in _LOW_QUOTES:  # an opening quote, and only when a closing quote follows ('\x82e Human' is a 'Th' slot)
        return ch if _is_opening(prev) and nxt.isalnum() and closes_later else ""
    if b in _ANGLE_QUOTES:
        return ch if not (p_letter and n_letter) else ""
    if b in _DAGGERS:
        return ch if not n_letter else ""
    if b == 0x80:  # euro sign: only beside a number, or after 'M'/'k' that follows a number or space
        if nxt.isdigit() or (nxt == " " and nxt2.isdigit()) or prev.isdigit():
            return ch
        return ""
    if b == 0x89:  # per mille: only after a digit
        return ch if prev.isdigit() else ""
    if b in _UPPER:  # word-initial capital before a lowercase letter, or inside an all-caps word
        if not p_letter and (nxt.islower() or (nxt.isupper() and nxt2.isupper())):
            return ch
        return ""
    if b == 0x9C:  # oe ligature: French words (oeuvre, coeur, soeur, noeud, Ricoeur) put it word-initially or after
        # c n s v f m h, before a plain letter. Not 'Quanti\x9ccation' (PDF 'fi' slot) or 'Innowacyjno\x9c\u015b\u0107'
        # (a cp1250 s-acute next to the real one).
        if (not p_letter or prev in "cnsvfmhCNSVFMH") and nxt.isascii() and nxt.islower():
            return ch
        return ""
    if b in _LOWER:  # s-caron, z-caron: inside or at the end of a word, or word-initial before a letter
        if n_letter or (p_letter and not n_letter):
            return ch
        return ""
    return ""  # 0x83 f-hook, 0x88 circumflex, 0x98 small tilde, and the five undefined bytes


def _map_c1(t):
    out = []
    n = len(t)
    for i, c in enumerate(t):
        b = ord(c)
        if 0x80 <= b <= 0x9F:
            prev = t[i - 1] if i > 0 else ""
            nxt = t[i + 1] if i + 1 < n else ""
            nxt2 = t[i + 2] if i + 2 < n else ""
            if b in _QUOTES and (nxt in _ASCII_QUOTES or prev in _ASCII_QUOTES):
                continue  # a C1 quote right beside an ASCII quote duplicates it
            out.append(_cp1252_char(b, prev, nxt, nxt2, t[i + 1:i + 9], bool(_CLOSING.search(t, i + 1))))
        else:
            out.append(c)
    return "".join(out)


def clean_title_controls(s):
    if not s or not isinstance(s, str) or not CTRL.search(s):
        return s
    t = repair_mojibake(s)              # 1 exact mojibake repair
    t = _MARC_PAIR.sub(r"\1", t)       # 2 MARC non-sort marks
    t = _map_c1(t)                      # 3 lone C1 -> its cp1252 character where it fits, else dropped
    t = _SEPARATORS.sub(" ", t)         # 4 separators -> space, every other control dropped
    t = _OTHER_CTRL.sub("", t)
    t = " ".join(t.split())
    return t if t else s


return clean_title_controls(s)
$$

In [ ]:
-- oxjob #1910: the SQL-only half (fast, Photon). C0 separators -> space, every other control dropped (so a MARC
-- non-sort pair U+0098 ... U+009C loses both marks and keeps its text), whitespace collapsed as Python's str.split()
-- does. Equal to clean_title_controls() for every title outside openalex.works.work_title_control_fixes (by
-- construction: that table holds exactly the disagreements).
-- No dollar sign in this notebook's SQL: the job's notebook task reads a dollar sign followed by a name or digit as a
-- parameter and substitutes ''. Night 2026-10-30, a regex group reference here became '', deleting the article between
-- the MARC marks; every marked title then disagreed with the Python clean and the fix table grew from 28K to 1.25M rows.
CREATE OR REPLACE FUNCTION openalex.common.drop_title_controls(s STRING)
RETURNS STRING
COMMENT 'oxjob #1910: drops control characters from a title (MARC non-sort marks keep their text); see clean_title_controls'
RETURN NULLIF(TRIM(REGEXP_REPLACE(
         REGEXP_REPLACE(
           REGEXP_REPLACE(s, '[\\t\\n\\x{000B}\\f\\r\\x{001C}-\\x{001F}]+', ' '),
           '[\\x{0000}-\\x{0008}\\x{000E}-\\x{001B}\\x{007F}-\\x{009F}]', ''),
         '[\\s\\p{Z}]+', ' ')), '');

-- oxjob #1910: the titles where dropping is not enough (a Windows-1252 byte that stands for a real character, exact
-- mojibake): one row per distinct location title holding a control character whose full clean differs from the drop.
-- ~28K rows, so CreateWorksBase broadcasts it; the Python function runs here on ~1M filtered strings, never inside the
-- main statement (a Python UDF there costs every one of the 463M rows).
CREATE OR REPLACE TABLE openalex.works.work_title_control_fixes
COMMENT 'oxjob #1910: raw location title -> clean title, where clean_title_controls differs from drop_title_controls. Rebuilt by CreateWorksBase each run.'
AS
SELECT raw_title, clean_title
FROM (
  SELECT raw_title, openalex.common.clean_title_controls(raw_title) AS clean_title
  FROM (SELECT DISTINCT title AS raw_title FROM openalex.works.locations_mapped_curated
        WHERE title RLIKE '[\\x{0000}-\\x{001F}\\x{007F}-\\x{009F}]')
)
WHERE clean_title IS DISTINCT FROM openalex.common.drop_title_controls(raw_title);

In [ ]:
-- oxjob #1920: built here as its own table (one row per PMID with MeSH, ~34M rows, ~30 s on PROD-WH), and the main
-- statement below only joins it. Computed inside the main statement (walden 4b1e1dfd, then the narrow-key form 46256a72),
-- the same SQL froze Works_Base's late stage on PROD-WH five times on 30 Oct 2026 (#1931), though alone it takes 30 s.
-- oxjob #1920: MeSH from ONE PubMed record per PMID, its latest version, as NCBI serves it. pubmed_items keeps every
-- record version and each yearly baseline's copy (136M rows for 41M PMIDs); reading them all listed each heading 2-3
-- times on 33.4M works, with copies that disagreed on is_major_topic on 284K. Latest = newest DateRevised, then newest
-- ingest (a baseline re-issues a record under the same DateRevised with that year's MeSH), then, for the ~2.7K PMIDs
-- whose newest version is still stored with different MeSH, the copy whose MeSH JSON sorts last (NCBI agreed 58/59 on
-- 30 Oct 2026; a hash pick 30/59). Checked on 29 Oct against NCBI efetch: 100/100 random works, 60/60 whose heading
-- set changes, 39/39 whose latest record has no MeSH, 40/40 same-date ties.
-- Plan shape (30 Oct, #1931): picking the record with MAX_BY(mesh, key incl. TO_JSON(mesh)) shuffled every record's
-- whole MeSH list twice and froze Works_Base on PROD-WH; here the headings shuffle as before, each tagged with its
-- record's narrow version key (dates, ingested_at, xxhash64 of the MeSH), and only the newest version's are kept.
CREATE OR REPLACE TABLE openalex.works.pmid_mesh_latest_record
COMMENT 'oxjob #1920: MeSH of the latest PubMed record of each PMID (pmid, mesh_formatted), as works serve it. Rebuilt by CreateWorksBase each run.'
AS
SELECT
  pmid,
  mesh_formatted
FROM
  (
    WITH pubmed_mesh_versions AS (
      SELECT
        FILTER(
          PubmedData.ArticleIdList.ArticleId,
          x -> x._IdType = 'pubmed'
        )._VALUE [0] AS pmid,
        MedlineCitation.MeshHeadingList AS mesh,
        STRUCT(
          TRY_CAST(MedlineCitation.DateRevised.Year AS INT) AS y,
          TRY_CAST(MedlineCitation.DateRevised.Month AS INT) AS m,
          TRY_CAST(MedlineCitation.DateRevised.Day AS INT) AS d,
          ingested_at
        ) AS version,
        XXHASH64(TO_JSON(MedlineCitation.MeshHeadingList)) AS mesh_hash
      FROM
        openalex.pubmed.pubmed_items
    ),
    -- PMIDs whose newest version is stored 2+ times with different MeSH (2,658 on 30 Oct 2026), and the copy to keep.
    pubmed_mesh_ties AS (
      SELECT
        pmid,
        latest_version
      FROM
        (
          SELECT
            pmid,
            MAX(version) AS latest_version,
            MAX_BY(mesh_hash, STRUCT(version, mesh_hash)) AS hash_hi,
            MAX_BY(mesh_hash, STRUCT(version, ~mesh_hash)) AS hash_lo
          FROM
            pubmed_mesh_versions
          GROUP BY
            pmid
        )
      WHERE
        hash_hi <> hash_lo
    ),
    -- The MeSH JSON is built only for those PMIDs' newest copies (broadcast join), never for all 136M rows.
    pubmed_mesh_tie_pick AS (
      SELECT
        pmid,
        MAX_BY(XXHASH64(TO_JSON(mesh)), TO_JSON(mesh)) AS mesh_hash
      FROM
        (
          SELECT /*+ BROADCAST(t) */
            v.pmid,
            v.mesh
          FROM
            (
              SELECT
                FILTER(
                  PubmedData.ArticleIdList.ArticleId,
                  x -> x._IdType = 'pubmed'
                )._VALUE [0] AS pmid,
                MedlineCitation.MeshHeadingList AS mesh,
                STRUCT(
                  TRY_CAST(MedlineCitation.DateRevised.Year AS INT) AS y,
                  TRY_CAST(MedlineCitation.DateRevised.Month AS INT) AS m,
                  TRY_CAST(MedlineCitation.DateRevised.Day AS INT) AS d,
                  ingested_at
                ) AS version
              FROM
                openalex.pubmed.pubmed_items
            ) v
            JOIN pubmed_mesh_ties t ON v.pmid = t.pmid
            AND v.version = t.latest_version
        )
      GROUP BY
        pmid
    )
    SELECT /*+ BROADCAST(tp) */
      h.pmid,
      CASE
        WHEN SIZE(
          FILTER(
            h.headings,
            e -> e.version = h.latest_version
            AND e.mesh_hash = COALESCE(tp.mesh_hash, e.mesh_hash)
            AND e.heading.descriptor_ui IS NOT NULL
          )
        ) > 0 THEN ARRAY_SORT(
          -- oxjob #1920: ARRAY_DISTINCT drops an entry one record lists twice (751 PMIDs on 29 Oct 2026).
          ARRAY_DISTINCT(
            TRANSFORM(
              FILTER(
                h.headings,
                e -> e.version = h.latest_version
                AND e.mesh_hash = COALESCE(tp.mesh_hash, e.mesh_hash)
                AND e.heading.descriptor_ui IS NOT NULL
              ),
              e -> e.heading
            )
          ),
          (l, r) -> CASE
            WHEN l.descriptor_ui < r.descriptor_ui THEN -1
            WHEN l.descriptor_ui > r.descriptor_ui THEN 1
            WHEN COALESCE(l.qualifier_ui, '') < COALESCE(r.qualifier_ui, '') THEN -1
            WHEN COALESCE(l.qualifier_ui, '') > COALESCE(r.qualifier_ui, '') THEN 1
            ELSE 0
          END
        )
      END AS mesh_formatted
    FROM
      (
        SELECT
          pmid,
          MAX(version) AS latest_version,
          COLLECT_LIST(
            STRUCT(
              version,
              mesh_hash,
              STRUCT(
                descriptor_ui,
                descriptor_name,
                qualifiers._UI AS qualifier_ui,
                qualifiers._VALUE AS qualifier_name,
                -- oxjob #1920: NLM stars a descriptor (every pair is major) or one qualifier (that pair is major):
                -- 'Neoplasms/drug therapy*'. Reading only the descriptor's star served false on 72.6M starred pairs.
                CASE
                  WHEN is_major_topic = 'Y' OR qualifiers._MajorTopicYN = 'Y' THEN TRUE
                  ELSE FALSE
                END AS is_major_topic
              ) AS heading
            )
          ) AS headings
        FROM
          (
            -- EXPLODE_OUTER keeps one row for a version without MeSH, so a latest record with none wins (and serves none).
            SELECT
              pmid,
              version,
              mesh_hash,
              EXPLODE_OUTER(mesh.MeshHeading) AS mesh_exploded,
              mesh_exploded.DescriptorName._UI AS descriptor_ui,
              mesh_exploded.DescriptorName._VALUE AS descriptor_name,
              EXPLODE_OUTER(
                ARRAYS_ZIP(
                  mesh_exploded.QualifierName._UI,
                  mesh_exploded.QualifierName._VALUE,
                  mesh_exploded.QualifierName._MajorTopicYN
                )
              ) AS qualifiers,
              mesh_exploded.DescriptorName._MajorTopicYN AS is_major_topic
            FROM
              pubmed_mesh_versions
          )
        GROUP BY
          pmid
      ) h
      LEFT JOIN pubmed_mesh_tie_pick tp ON tp.pmid = h.pmid
  )
WHERE
  mesh_formatted IS NOT NULL;

In [0]:
-- oxjob #2106: the one normaliser for served entity names (author profiles, sources). Elsevier's dotless i + accent
-- ('Valentı́n Fuster', 8.5K profiles), decomposed accents (18K), Turkish i + dot above, invisible format characters
-- (soft hyphen, ZWSP, LRM/RLM, BOM, word joiner: 11.6K profiles; 'IODP Bre\xadmen Core Re\xadpos\xadit\xadory'), and for
-- sources HTML entities ('&amp;', '&#35745;...'). Unicode NFC is the canonical form of the same text. ZWNJ/ZWJ stay
-- (Persian, Indic). Mojibake is left alone (its soft hyphen is a byte of a broken letter). Name search (authors index:
-- asciifolding) and autocomplete missed these names: display_name.search found 0/100 (9 Nov 2026).
-- Source of truth and tests: oxjobs #2106 work/name_text.py, work/test_name_text.py (this body is generated from it by
-- work/patch_walden.py). The same body is created in authors/CreateAuthors, sources/CreateSourcesApi and
-- end2end/CreateWorksBase. Revert: git revert the #2106 commit (the next runs serve the stored names again).
CREATE OR REPLACE FUNCTION openalex.common.normalize_name_text(s STRING, decode_entities BOOLEAN)
RETURNS STRING
LANGUAGE PYTHON
DETERMINISTIC
COMMENT 'oxjob #2106: served entity name text normalised: dotless i/j + mark, Turkish i + dot above, invisible format characters removed (ZWNJ/ZWJ kept), NFC, HTML entities decoded when decode_entities. Returns its input when there is nothing to fix. Source + tests: oxjobs #2106 work/name_text.py'
AS $$
import html
import re
import unicodedata
from html.entities import html5 as _HTML5

_DOTLESS = re.compile('([\u0131\u0237])(?=[\u0300-\u036f])')
_TURKISH_I = re.compile('(?<=[i\u0130])\u0307+')  # 'i\u0307' -> 'i', '\u0130\u0307' -> '\u0130': the letter already has its dot
_ZWSP = re.compile('\u200b+')
_INVISIBLE = re.compile('[\u00ad\u200b\u200e\u200f\ufeff\u2060]')
_ENTITY = re.compile(r'&(#[0-9]{2,7}|#[xX][0-9a-fA-F]{2,6}|[A-Za-z][A-Za-z0-9]{1,30})\x3b')  # \x3b = semicolon: a literal one would split the SQL cell
_MOJIBAKE = re.compile('[\u00c3\u00c2\u00c4\u00c5\u00d0\u00d1][\u0080-\u00bf]|\u00e2\u20ac')
_SPACES = re.compile(r'\s+')
_CONTROL = re.compile('[\x00-\x1f\x7f-\x9f]')


_CODECS = ('latin-1', 'cp1252', 'cp1250', 'cp1251')


def _zwsp(m):
    s, i, j = m.string, m.start(), m.end()
    return ' ' if 0 < i and j < len(s) and s[i - 1].islower() and s[j].isupper() else ''


def _soft_hyphen_is_a_byte(s):
    """True when some U+00AD is byte 0xAD of a UTF-8 letter read as a one-byte codepage: 'GarcÃ\\xada' (latin-1),
    'Р\\xad.' (cp1251 'Э'), 'MedellĂ\\xadN' (cp1250), Thai 'à¸\\xad'. 'Cí\\xadcero' is not: 0xED needs two more bytes."""
    for i, ch in enumerate(s):
        if ch != '\u00ad':
            continue
        for codec in _CODECS:
            for k in (1, 2, 3):  # lead byte k chars back, then k continuation bytes ending at or after the 0xAD
                if i - k < 0:
                    continue
                try:
                    b = s[i - k:i + 1].encode(codec) + s[i + 1:i + 1 + (3 - k)].encode(codec)
                except UnicodeEncodeError:
                    continue
                if len(b) <= k:
                    continue
                n = 2 if 0xC2 <= b[0] <= 0xDF else 3 if 0xE0 <= b[0] <= 0xEF else 4 if 0xF0 <= b[0] <= 0xF4 else 0
                if n == k + 1 or (n > k + 1 and len(b) >= n):
                    try:
                        b[:n].decode('utf-8')
                        return True
                    except UnicodeDecodeError:
                        pass
    return False


def _decode_entity(m):
    """One well-formed '&name' + semicolon or '&#...' + semicolon token. html.unescape on a whole string also decodes
    references with no semicolon ('Planning &regional' -> 'Planning ®ional', #2107), so: numeric tokens through
    html.unescape (its cp1252 and invalid-codepoint rules), named ones only when the whole name is an HTML5 entity."""
    g = m.group(1)
    if g[0] == '#':
        return html.unescape(m.group(0))
    return _HTML5.get(g + '\x3b', m.group(0))


def normalize_name_text(s, decode_entities=False):
    if s is None:
        return None
    if _MOJIBAKE.search(s) or ('\u00ad' in s and _soft_hyphen_is_a_byte(s)):
        return s
    t = s
    if decode_entities and _ENTITY.search(t):
        for _ in range(3):  # double-encoded entities, bounded
            u = _ENTITY.sub(_decode_entity, t)
            if u == t:
                break
            t = u
    t = _DOTLESS.sub(lambda m: 'i' if m.group(1) == '\u0131' else 'j', t)
    t = _TURKISH_I.sub('', t)
    t = _ZWSP.sub(_zwsp, t)
    t = _INVISIBLE.sub('', t)
    t = unicodedata.normalize('NFC', t)
    if t == s:
        return s
    if not _CONTROL.search(s):  # a catalogue string's line breaks are #1912's to read: leave its whitespace
        t = _SPACES.sub(' ', t).strip()
    return t if t.strip() else s


return normalize_name_text(s, decode_entities)
$$


In [0]:
-- oxjob #1163: the PMC copies (PMC OAI records) of works that get no PMCID from their PubMed record, each with NCBI's own
-- row for its PMCID (PMC-ids.csv.gz, kept as openalex.pubmed.ncbi_pmc_ids by the weekly PMC harvest gap check). The main
-- statement's ids build reads this to give such a work the PMCID of its own PMC copy, only where NCBI ties that PMCID to
-- the work (rules there). Kept small (~0.9M works, a title hash, no free text) so the main statement broadcasts it: a
-- shuffled join of the full 12.4M copies spilled 1.04 TB on 10 Nov 2026 (bar 0.62).
-- Left out here: copies whose PMCID is served elsewhere, by the PubMed route (the work holding the PMID whose PubMed
-- record states it) or by another work's PMC copy (a PMCID names one article), and works that hold a PMID whose PubMed
-- record states a PMCID (the PubMed route serves theirs).
CREATE OR REPLACE TABLE openalex.works.pmc_copy_ncbi_ids
COMMENT 'oxjob #1163: per work with no PubMed-record PMCID, its PMC copies whose PMCID sits on no other work, with NCBI PMC-ids row (doi, pmid, year, volume, page) and a title hash. Rebuilt by CreateWorksBase each run.'
AS
WITH copies AS (
  SELECT work_id,
    REGEXP_EXTRACT(native_id, '^oai:pubmedcentral[.]nih[.]gov:([0-9]+)', 1) AS pmc_digits,
    XXHASH64(LOWER(REGEXP_REPLACE(title, '[^\\p{L}\\p{N}]', ''))) AS title_hash,
    REGEXP_REPLACE(title, '[^\\p{L}\\p{N}]', '') <> '' AS has_title
  FROM openalex.works.locations_mapped_curated
  WHERE provenance = 'repo' AND endpoint_id = 'ac9de7698155b820de7'
    AND native_id RLIKE '^oai:pubmedcentral[.]nih[.]gov:[0-9]+$'
),
-- the same PubMed-record PMCIDs the main statement's `pmc` join serves, on every work holding that PMID
pubmed_route AS (
  SELECT l.work_id, p.pmcid_digits AS pmc_digits
  FROM (
    SELECT native_id AS pmid,
      MIN(NULLIF(REGEXP_EXTRACT(UPPER(TRIM(GET(FILTER(ids, x -> x.namespace = 'pmcid'), 0).id)),
                                '^PMC([0-9]+)([.][0-9]+)?$', 1), '')) AS pmcid_digits
    FROM openalex.pubmed.pubmed_works
    WHERE EXISTS(ids, x -> x.namespace = 'pmcid')
    GROUP BY native_id
  ) p
  JOIN openalex.works.locations_mapped_curated l ON l.provenance = 'pubmed' AND l.native_id = p.pmid
  WHERE p.pmcid_digits IS NOT NULL
),
holders AS (
  SELECT pmc_digits, COUNT(DISTINCT work_id) AS n_works
  FROM (SELECT pmc_digits, work_id FROM pubmed_route UNION ALL SELECT pmc_digits, work_id FROM copies)
  GROUP BY pmc_digits
),
-- one row per PMCID (NCBI's list repeats one)
ncbi AS (
  SELECT pmc_digits, MIN_BY(NAMED_STRUCT('doi', doi, 'pmid', pmid, 'year', year, 'volume', volume, 'page', page), release_date) AS r
  FROM openalex.pubmed.ncbi_pmc_ids
  GROUP BY pmc_digits
)
SELECT c.work_id,
  COLLECT_LIST(NAMED_STRUCT(
    'pmc_digits', c.pmc_digits, 'ncbi_doi', n.r.doi, 'ncbi_pmid', n.r.pmid,
    'ncbi_year', n.r.year, 'ncbi_volume', n.r.volume, 'ncbi_page', n.r.page,
    -- the title only matters where NCBI lists no DOI and no PMID (rule D)
    'title_hash', IF(n.r.doi IS NULL AND n.r.pmid IS NULL AND c.has_title, c.title_hash, NULL))) AS copies
FROM copies c
JOIN ncbi n ON n.pmc_digits = c.pmc_digits
JOIN holders h ON h.pmc_digits = c.pmc_digits AND h.n_works = 1
LEFT ANTI JOIN (SELECT DISTINCT work_id FROM pubmed_route) pr ON pr.work_id = c.work_id
GROUP BY c.work_id;

In [0]:
-- oxjob #1158: works whose Crossref deposit lists only the first author(s) while the work's own PubMed record lists everyone
-- (JAMA 2002 W2124013021: Crossref 'Paul K. Whelton', PubMed 12 authors). Crossref outranks PubMed for the byline, so the
-- co-authors were missing from the work, their profiles and their institutions (~270K works, 13 support tickets).
-- One row per such work: the PubMed authors that follow Crossref's. The main statement appends them to Crossref's list
-- (authors CTE), so existing seats keep their name, ORCID, affiliations and author id; only new seats are added, and
-- MatchAuthors' size_grew rematch (oxjob #649) gives them author ids.
-- A PubMed record qualifies only if it (a) names the Crossref record's DOI, or names no DOI and has the same normalized
-- title (never another paper's PMID: oxjob #2155 / #2167), (b) lists more named authors, at most 100 (blank names =
-- collective authors are dropped first), (c) starts with exactly Crossref's authors, seat by seat (author_key), and (d) repeats nobody under
-- a near-identical name (some stored PubMed records keep a corrected spelling as an extra author; NCBI no longer lists it).
-- Bar: NCBI esummary on fresh draws, oxjob #1158 README. Small (~270K rows), so the main statement broadcasts it.
CREATE OR REPLACE TABLE openalex.works.pubmed_fuller_bylines
COMMENT 'oxjob #1158: per work whose Crossref byline is a truncated prefix of its own PubMed record, the PubMed authors after Crossref''s last (authors_tail), with the Crossref record and list size they extend. Rebuilt by CreateWorksBase each run.'
AS
WITH pm AS (
  SELECT work_id, native_id AS pmid, LOWER(best_doi) AS pdoi, normalized_title AS p_title, updated_date AS p_updated,
         FILTER(authors, x -> TRIM(COALESCE(x.name, '')) <> '') AS pa
  FROM openalex.works.locations_mapped_curated
  WHERE provenance = 'pubmed' AND native_id_namespace = 'pmid' AND SIZE(authors) > 1
),
-- the Crossref record the authors CTE picks (same order as there; Crossref is priority 1)
cr AS (
  SELECT l.work_id, l.native_id AS cr_native_id, l.normalized_title AS c_title, l.authors AS ca
  FROM openalex.works.locations_mapped_curated l
  LEFT SEMI JOIN (SELECT DISTINCT work_id FROM pm) k ON k.work_id = l.work_id
  WHERE l.provenance = 'crossref' AND l.authors IS NOT NULL AND SIZE(l.authors) > 0
  QUALIFY ROW_NUMBER() OVER (PARTITION BY l.work_id
    ORDER BY l.authors_exist DESC NULLS LAST, SIZE(l.authors) DESC, l.native_id ASC, l.updated_date DESC NULLS LAST,
             xxhash64(to_json(l.authors)) ASC) = 1
)
SELECT c.work_id, c.cr_native_id, SIZE(c.ca) AS cr_n, p.pmid, SLICE(p.pa, SIZE(c.ca) + 1, SIZE(p.pa) - SIZE(c.ca)) AS authors_tail
FROM cr c JOIN pm p ON p.work_id = c.work_id AND SIZE(p.pa) > SIZE(c.ca)
-- CASE, not AND: the pairwise name check (d) must run only on the few rows that pass (a)-(c) and list at most 100 authors
-- (it is quadratic in the list; as a plain AND it ran on every pair, 3,000-author physics records included).
WHERE CASE WHEN (COALESCE(p.pdoi = LOWER(c.cr_native_id), FALSE)
                 OR (p.pdoi IS NULL AND COALESCE(LENGTH(c.c_title) >= 15 AND c.c_title = p.p_title, FALSE)))
            AND SIZE(p.pa) <= 100
            AND FORALL(SEQUENCE(0, SIZE(c.ca) - 1), i -> GET(c.ca, i).author_key IS NOT NULL AND GET(c.ca, i).author_key = GET(p.pa, i).author_key)
      THEN NOT EXISTS(SEQUENCE(1, SIZE(p.pa) - 1), i -> EXISTS(SEQUENCE(0, i - 1), k ->
        COALESCE(GET(SPLIT(GET(p.pa, i).author_key, '\\x3B'), 1) = GET(SPLIT(GET(p.pa, k).author_key, '\\x3B'), 1)
                 AND LEVENSHTEIN(GET(SPLIT(GET(p.pa, i).author_key, '\\x3B'), 0), GET(SPLIT(GET(p.pa, k).author_key, '\\x3B'), 0)) <= 2, TRUE)))
      ELSE FALSE END
QUALIFY ROW_NUMBER() OVER (PARTITION BY c.work_id
  ORDER BY COALESCE(p.pdoi = LOWER(c.cr_native_id), FALSE) DESC, SIZE(p.pa) DESC, p.pmid ASC, p.p_updated DESC NULLS LAST,
           xxhash64(to_json(p.pa)) ASC) = 1

In [ ]:
-- oxjob #2075 D11 round one (Jason, 22 Nov 2026): the editors of edited books, as seats carrying the contributor role
-- `editor`. Scope: a book (OpenAlex type book on the last published night) whose Crossref record (types book,
-- edited-book, monograph, reference-book, book-set) lists editors and no authors. One row per work and role (role-generic
-- so other Crossref person lists, e.g. translators (#2512), can add rows; only editors fill a byline):
--   * fill: when no record of the work names anyone, raw_authorships_curated in the main statement takes these editors as
--     the byline, in deposit order, positions like any byline (fill-only, as #1815's curated byline, which wins).
--     MatchAuthors' size_grew rematch gives the seats author ids, as for any byline that grew.
--   * relabel: CreateWorksEnriched's roles cell (work_contributor_roles) adds `editor` to each seat whose name is an
--     editor's, filled or not (most of these books already show their editors, from a repository or landing-page record).
-- Editors stay on the book itself, never copied onto its chapters (a chapter's own record lists the volume's editors).
-- Sized 22 Nov 2026: ~493K books, ~227K of them filled (~511K seats). Name as Crossref's parse builds it ('name', else
-- 'given family'). Revert: delete this cell, the oxjob #2075 lines in raw_authorships_curated and
-- raw_authorships_extended below, and the editor part of CreateWorksEnriched's roles cell.
CREATE OR REPLACE TABLE openalex.works.crossref_book_contributors
COMMENT 'oxjob #2075: per book whose Crossref record lists editors and no authors, the editors (role editor) in deposit order; fill = no record of the work names anyone. Rebuilt by CreateWorksBase each run.'
AS
WITH named AS (
  SELECT DISTINCT work_id FROM openalex.works.locations_mapped_curated WHERE authors IS NOT NULL AND SIZE(authors) > 0
)
SELECT l.work_id, l.native_id AS doi, 'editor' AS role,
       FILTER(TRANSFORM(c.editor, e -> NAMED_STRUCT(
           'name', SUBSTRING(TRIM(REGEXP_REPLACE(COALESCE(NULLIF(TRIM(e.name), ''), TRIM(CONCAT_WS(' ', e.given, e.family))), '\\s+', ' ')), 1, 500),
           'orcid', NULLIF(REGEXP_EXTRACT(e.ORCID, '([0-9]{4}-[0-9]{4}-[0-9]{4}-[0-9]{3}[0-9Xx])', 1), ''),
           'family', NULLIF(TRIM(e.family), ''))),
         x -> x.name IS NOT NULL AND x.name <> '') AS people,
       n.work_id IS NULL AS fill
FROM openalex.crossref.crossref_deduplicated c
JOIN (SELECT DISTINCT work_id, native_id FROM openalex.works.locations_mapped_curated WHERE provenance = 'crossref') l
  ON l.native_id = c.native_id
JOIN openalex.works.openalex_works w ON w.id = l.work_id AND w.type = 'book'
LEFT JOIN named n ON n.work_id = l.work_id
WHERE COALESCE(SIZE(c.editor), 0) > 0 AND COALESCE(SIZE(c.author), 0) = 0
  AND c.type IN ('book', 'edited-book', 'monograph', 'reference-book', 'book-set')
QUALIFY ROW_NUMBER() OVER (PARTITION BY l.work_id ORDER BY c.native_id) = 1

In [ ]:
-- oxjob #2512 (Jason, 22 Nov 2026): named translators as their own authorships, with the contributor role `translator`.
-- His words: "if a translator is listed, that translator should be added to the contributors list with the appropriate
-- translator role selected ... One Charles Darwin that looks the same, and another one called Pierre Fromage that looks
-- different because its type is translator." Per work whose Crossref record lists translators (any type: articles,
-- chapters, books, datasets, reference entries; 51,670 works, 61,410 translators on 22 Nov 2026), its translators in
-- deposit order, each once.
--   * the main statement (set_fields, oxjob #2512 lines) appends each translator the byline doesn't already show after
--     the byline's last seat (59,065 seats on 49,916 works): author_position NULL (first and last stay with the authors,
--     so last-author fields keep their author), never corresponding, no affiliations. MatchAuthors' size_grew rematch
--     gives the new seats author ids, as for any byline that grew; the authors' seats don't move.
--   * CreateWorksEnriched's roles cell gives `translator` to every seat that is one of these translators, appended or
--     already on the byline (~2,350: an author who also translated keeps one seat, with the role).
-- A seat is a translator when its name has the same words in any order ('Family, Given' read as 'Given Family'), or the
-- same last word and the same first word ('Marcos Steagall' / 'Marcos Mortensen Steagall'), or the same last word and
-- first initial where one side gives only an initial ('P. Fromage' / 'Pierre Fromage'). Name words: lower case, common
-- accents and combining marks dropped. k = the sorted words; first, last = the first and last word; ini = the first
-- initial; short = the first word is an initial. One Crossref record per work (the one listing the most translators).
-- DataCite's Translator contributors (~1K works): not yet.
-- Revert: delete this cell, the oxjob #2512 lines in set_fields below and the translator part of CreateWorksEnriched's
-- roles cell (the side table can stay).
CREATE OR REPLACE TABLE openalex.works.crossref_translators
COMMENT 'oxjob #2512: per work whose Crossref record lists translators, the translators in deposit order with their name keys. Rebuilt by CreateWorksBase each run.'
AS
WITH tr AS (
  SELECT l.work_id, c.native_id AS doi,
         FILTER(TRANSFORM(c.translator, e -> NAMED_STRUCT(
             'name', SUBSTRING(TRIM(REGEXP_REPLACE(COALESCE(NULLIF(TRIM(e.name), ''), TRIM(CONCAT_WS(' ', e.given, e.family))), '\\s+', ' ')), 1, 500),
             'orcid', NULLIF(REGEXP_EXTRACT(e.ORCID, '([0-9]{4}-[0-9]{4}-[0-9]{4}-[0-9]{3}[0-9Xx])', 1), ''))),
           x -> x.name IS NOT NULL AND x.name <> '') AS people
  FROM openalex.crossref.crossref_deduplicated c
  JOIN (SELECT DISTINCT work_id, native_id FROM openalex.works.locations_mapped_curated WHERE provenance = 'crossref') l
    ON l.native_id = c.native_id
  WHERE COALESCE(SIZE(c.translator), 0) > 0
  QUALIFY ROW_NUMBER() OVER (PARTITION BY l.work_id ORDER BY SIZE(c.translator) DESC, c.native_id) = 1
),
tokened AS (
  SELECT work_id, doi,
         TRANSFORM(people, p -> NAMED_STRUCT('name', p.name, 'orcid', p.orcid,
           't', FILTER(SPLIT(TRIM(REGEXP_REPLACE(REGEXP_REPLACE(TRANSLATE(LOWER(
                  CASE WHEN p.name RLIKE '^[^,]+,[^,]+$' THEN CONCAT(TRIM(SUBSTRING_INDEX(p.name, ',', -1)), ' ', TRIM(SUBSTRING_INDEX(p.name, ',', 1)))
                       ELSE p.name END),
                  'áàäâãåéèëêíìïîóòöôõøúùüûñçýćčšžłőűăşţșțđığ', 'aaaaaaeeeeiiiioooooouuuuncyccszlouaststdig'), '\\p{M}', ''), '[^\\p{L}]+', ' ')), ' '), w -> w <> ''))) AS people
  FROM tr
),
keyed AS (
  SELECT work_id, doi,
         FILTER(TRANSFORM(people, p -> NAMED_STRUCT('name', p.name, 'orcid', p.orcid,
                  'k', ARRAY_JOIN(ARRAY_SORT(p.t), ' '),
                  'first', TRY_ELEMENT_AT(p.t, 1),
                  'last', IF(SIZE(p.t) >= 2, TRY_ELEMENT_AT(p.t, -1), CAST(NULL AS STRING)),
                  'ini', LEFT(TRY_ELEMENT_AT(p.t, 1), 1),
                  'short', COALESCE(LENGTH(TRY_ELEMENT_AT(p.t, 1)) = 1, FALSE))),
                p -> p.k <> '') AS people
  FROM tokened
)
SELECT work_id, doi,
       FILTER(people, (p, i) -> ARRAY_POSITION(TRANSFORM(people, q -> q.k), p.k) = i + 1) AS people
FROM keyed
WHERE SIZE(people) > 0

In [ ]:
-- oxjob #1006: hosts that no longer serve the items OpenAlex links to. A location whose URL (the landing page, else the
-- PDF) is on one of these hosts is never an open-access copy and never leads the work's locations (sort_score 12 in the
-- main cell): the docs define oa_url / best_oa_location as a copy a reader can open (help.openalex.org, Open access), and
-- 'What right means' 2026-10-26 (data-bugs charter): right = what the record-owning host serves now. A PDF link on one of
-- these hosts beside a live landing page is dropped. kind: dead (no DNS, no connection, 404/410, a not-found page or a
-- redirect to a front page), retired (the operator closed the service), squatted (the domain now belongs to someone else).
-- rewrite 'cabi_an': CAB Direct closed in Jan 2024 and its domain was re-registered by a stranger in May 2026; each record
-- URL carries the CAB accession number, and CABI Digital Library serves the same record at /doi/10.5555/<number> (20/20 same
-- item, oxjob #1006 EXPLORE.md), behind a paywall, so it is rewritten and never an OA copy. url_rlike limits a host to the
-- dead paths (loc.gov /catdir/, INIS's old /search/ pages, journals.le.ac.uk /ojs1/, journals.viamedica.pl
-- /advances_in_respiratory_medicine/: the journal moved to MDPI in 2022 and its DOIs resolve there; oxjob #2019).
-- Each host entered on a fresh random draw of 20 of its locations, 19-20 dead and none serving the item (seed
-- oxjob1006-s1, 14 Nov 2026; wave 2: seed oxjob1006-s2, 17 Nov, a host failing only with 5xx needs a second draw 3+ days
-- later and dated proof it lasted; oxjob #1006 README). Re-probe before adding or removing a host:
-- a host that comes back leaves this list and its works return the next night.
-- Left the list on re-probe: repository.hkust.edu.hk (21 Nov 2026, oxjob #2336): s1's 20 'dead' URLs were a redirect
-- loop to its /ir/captcha page (curl rc 47), not a dead host; 20/20 serve the item again, its bitstream PDFs too.
-- Revert: git revert the commit (the main cell then ignores the table), then DROP TABLE openalex.works.dead_link_hosts.
CREATE OR REPLACE TABLE openalex.works.dead_link_hosts
COMMENT 'oxjob #1006: hosts that no longer serve the items OpenAlex links to (dead, retired, squatted). CreateWorksBase never serves a location on them as an OA copy or as the primary location while the work has another; cabdirect.org record URLs are rewritten to CABI Digital Library. Rebuilt from this notebook each run.'
AS SELECT * FROM VALUES
  ('91.222.128.133', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_conn 20'),
  ('actachemscand.org', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('alrjournal.com', NULL, 'squatted', NULL, '2026-11-14', 'parked-domain page (ticket 26689)'),
  ('app.trdizin.gov.tr', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('autopapers.ssrn.com', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('bonoi.org', NULL, 'dead', NULL, '2026-11-14', '526 invalid SSL since >= 22 Oct (ticket 26412)'),
  ('brief.land', NULL, 'squatted', NULL, '2026-11-14', 'casino domain, old Brieflands paths 404 (ticket 26594)'),
  ('cabdirect.org', NULL, 'squatted', 'cabi_an', '2026-11-14', 'CAB Direct closed 31 Jan 2024, domain re-registered 2026-05-06, 410 on every record URL'),
  ('campus.unibo.it', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('cheric.org', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: front_page 19, http_dead 1'),
  ('citeseerx.ist.psu.edu', NULL, 'retired', NULL, '2026-11-14', 'every URL 301s to web.archive.org (site retired)'),
  ('collectionscanada.ca', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: http_dead 20'),
  ('color.bougeret.fr', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_conn 20'),
  ('content.taylorfrancis.com', NULL, 'dead', NULL, '2026-11-17', '40/40 HTTP 502 on seeds oxjob1006-s1 (14 Nov) and oxjob1006-s2 (17 Nov); Wayback: 502 every month since Jan 2026'),
  ('dergipark.gov.tr', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: http_dead 20'),
  ('digilib.its.ac.id', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('digilib.uinsgd.ac.id', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: http_dead 20'),
  ('digilib.unila.ac.id', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_conn 20'),
  ('digitalcommons.olivet.edu', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: http_dead 20'),
  ('dissem.in', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_conn 20'),
  ('doc.rero.ch', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: http_dead 20'),
  ('dspace.nplg.gov.ge', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: http_dead 20'),
  ('dspace.uce.edu.ec', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_conn 9, http_dead 11'),
  ('dspace.unive.it', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('e-journal.uajy.ac.id', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('ecsocman.hse.ru', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: front_page 19, http_dead 1'),
  ('elar.rsvpular.rsvpu.ru', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('eprints.kingston.ac.uk', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('eprints.soas.ac.uk', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('eprints.uns.ac.id', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('eprints.usq.edu.au', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_conn 20'),
  ('eprints.utm.my', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: http_dead 20'),
  ('eprints.uwe.ac.uk', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('epsilon.insee.fr', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('etd.unsyiah.ac.id', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('exeley.com', NULL, 'squatted', NULL, '2026-11-17', 'parked: every URL serves one title-less page, desk Chrome lands on an ad funnel (17 Nov); ticket 25783'),
  ('exhibit.xavier.edu', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: http_dead 20'),
  ('forskningsdatabasen.dk', NULL, 'dead', NULL, '2026-11-15', '20/20 dead on seed oxjob881-s1: dead_conn 20; Danish research database closed Jan 2021 (ZD 3798, oxjob #881)'),
  ('fundamental-research.ru', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: http_dead 20'),
  ('i-scholar.in', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('iasj.net', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('idsa.confex.com', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('infona.pl', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: http_dead 20'),
  ('inis.iaea.org', '^https?://(www[.])?inis[.]iaea[.]org/search/', 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: http_dead 20'),
  ('ir.nctu.edu.tw', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('journal.kci.go.kr', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: front_page 20'),
  ('journals.le.ac.uk', '^https?://(www[.])?journals[.]le[.]ac[.]uk/ojs1/', 'dead', NULL, '2026-11-14', 'old /ojs1/ paths 404 (journal moved to /index.php/ and nmes.pubpub.org)'),
  ('journals.viamedica.pl', '^https?://(www[.])?journals[.]viamedica[.]pl/advances_in_respiratory_medicine/', 'dead', NULL, '2026-11-23', '20/20 front_page on seed oxjob2019-arm-s1: Advances in Respiratory Medicine left Via Medica for MDPI in 2022 (ticket 24760); other Via Medica journal paths answer its bot check (bc.php), not a front-page redirect'),
  ('jppa-unmul.com', NULL, 'squatted', NULL, '2026-11-14', 'slot/gambling redirect (ticket 24794)'),
  ('jsr.org', NULL, 'dead', NULL, '2026-11-14', '522 origin down since >= 27 Oct (ticket 26620)'),
  ('karya-ilmiah.um.ac.id', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('katalog.pustaka.unand.ac.id', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('knigozal.com', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_conn 20'),
  ('lat1.lis.soas.ac.uk', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('leyes.org.es', NULL, 'squatted', NULL, '2026-11-14', 'casino site (BetPlay Casino ES)'),
  ('lib.pdii.lipi.go.id', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('loc.gov', '^https?://(www[.])?loc[.]gov/catdir/', 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: http_dead 20'),
  ('moh-it.pure.elsevier.com', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: http_dead 20'),
  ('n-p-r.org', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('nouveautes-editeurs.bnf.fr', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: http_dead 20'),
  ('nrl.northumbria.ac.uk', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('nyuscholars.nyu.edu', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_conn 20'),
  ('oai.dtic.mil', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('oaji.net', NULL, 'dead', NULL, '2026-11-14', '502 since Sep 2026 (Wayback, ticket 25816 on 11 Oct)'),
  ('opus.ub.hsu-hh.de', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('pqdtopen.proquest.com', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: http_dead 20'),
  ('questia.com', NULL, 'retired', NULL, '2026-11-14', 'Questia closed Dec 2020 (Cengage), 20/20 time out'),
  ('r4d.dfid.gov.uk', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('rack.sbhealthsource.org', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('rcaap.pt', NULL, 'dead', NULL, '2026-11-17', '40/40 HTTP 500 on seeds oxjob1006-s1 and oxjob1006-s2; desk Chrome shows the RCAAP error page; Wayback: /v1/detail.jsp 500 since Oct 2026'),
  ('rcin.org.pl', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_conn 20'),
  ('rechercheisidore.fr', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_conn 20'),
  ('reference.sabinet.co.za', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('remeri.org.mx', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: http_dead 20'),
  ('repository.urosario.edu.co', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: http_dead 9, dead_conn 11'),
  ('repository.usu.ac.id', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('repository.wima.ac.id', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_conn 19, dead_dns 1'),
  ('scholar.dkyobobook.co.kr', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('scholar.ndsl.kr', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: front_page 20'),
  ('scholar.uwindsor.ca', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: front_page 20'),
  ('scholars.northwestern.edu', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: http_dead 20'),
  ('scholarworks.montana.edu', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: http_dead 20'),
  ('t2r2.star.titech.ac.jp', NULL, 'retired', NULL, '2026-11-14', 'operator notice 2026-07-22: T2R2/STARSearch stopped, 403 on every item'),
  ('tampub.uta.fi', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('territorystories.nt.gov.au', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: http_dead 20'),
  ('tesi.eprints.luiss.it', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_conn 20'),
  ('textjournal.com.au', NULL, 'squatted', NULL, '2026-11-17', 'casino site (ticket 24026, 9 Sep 2026); parking redirect ww84.textjournal.com.au in Jan 2026'),
  ('tojdac.org', NULL, 'squatted', NULL, '2026-11-14', 'gambling redirect buysellhodlapp.com (ticket 23999)'),
  ('transactions-journal.aesop-planning.eu', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('vestnik.astu.org', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: dead_dns 20'),
  ('works.bepress.com', NULL, 'retired', NULL, '2026-11-14', 'SelectedWorks retired: every URL -> Digital Commons retirement page'),
  ('wprim.whocc.org.cn', '^https?://(www[.])?wprim[.]whocc[.]org[.]cn/local/detail[.]jsp', 'dead', NULL, '2026-11-17', '39/39 /local/detail.jsp 404 on seeds oxjob1006-s1 and oxjob1006-s2; Wayback: 404 since May 2026'),
  ('yadda.icm.edu.pl', NULL, 'dead', NULL, '2026-11-14', '20/20 dead on seed oxjob1006-s1: http_dead 20')
AS t(host, url_rlike, kind, rewrite, checked, why);

In [ ]:
-- oxjob #1628: Zenodo records whose own latest OAI record says restricted, closed or embargoed access (dc:rights), by
-- record number. #2266's Repo.py veto stores such a record is_oa false, but two other copies of the same deposit kept
-- reading open: its DataCite DOI 10.5281/zenodo.<record> (DataCite.py marks every DataCite record open, and Zenodo's
-- DataCite rightsList is empty on most restricted deposits: 3,218 Zenodo DOIs say restricted there against 337,768 here)
-- and ~35K repo records whose re-parse was dropped (#2266 Handoff). The main cell treats both as closed Zenodo copies.
-- Policy: a record its own repository marks restricted, with no file, is not an OA copy (#1628 Log, follows RULINGS § 3
-- #1168 + #1165). Bar: Zenodo API on a random 40 of the copies this adds (30 DataCite, 10 repo), 40/40 restricted or
-- embargoed with no public file (17 Nov 2026, #1628 work/). Latest copy per record, so an embargo that lifts drops out
-- the next night. ~340K rows; the main cell broadcasts it.
-- Revert: git revert the commit (the main cell then ignores the table), then DROP TABLE openalex.works.zenodo_closed_records.
CREATE OR REPLACE TABLE openalex.works.zenodo_closed_records
COMMENT 'oxjob #1628: Zenodo record numbers whose latest OAI record (openalex.repo.repo_items dc:rights) says restricted, closed or embargoed access. CreateWorksBase never serves a copy of them (repo oai:zenodo.org:N or DataCite 10.5281/zenodo.N) as open. Rebuilt from this notebook each run.'
AS
SELECT rec
FROM (
  SELECT
    TRY_CAST(REGEXP_EXTRACT(`ns0:header`.`ns0:identifier`, '^oai:zenodo[.]org:([0-9]+)$', 1) AS BIGINT) AS rec,
    MAX_BY(`ns0:metadata`.`ns1:dc`.`dc:rights`, ingested_at) AS rights
  FROM openalex.repo.repo_items
  WHERE `ns0:header`.`ns0:identifier` LIKE 'oai:zenodo.org:%'
  GROUP BY 1
)
WHERE rec IS NOT NULL
  AND EXISTS(rights, x -> LOWER(x) RLIKE 'info:eu-repo/semantics/(restricted|closed|embargoed)access')

In [ ]:
-- oxjob #2209: works whose journal's own copy should be the primary location. 4.59M works served a repository or
-- aggregator record (Dialnet, IRDB, DOAJ, LA Referencia, PubMed ...) as primary_location although they hold a location on
-- their journal: the location sort ranks any record with a version label above a legacy (MAG) journal record, which has
-- none. The docs say a journal article's primary copy is the journal's (help.openalex.org, data/locations).
-- A legacy journal assignment alone is not trusted (bar 13 Nov 2026, seed 2209: 87% right), so a work is listed here only
-- when ANOTHER record on it names that journal itself: its journal string equals, or starts with, the journal's name or
-- an alternate title, or is one of the journal's ISSNs (fresh draw, seed 22091: 95 of 97 right; exact 43/44, starts-with
-- 47/48, ISSN 40/40). Works with a Crossref location are never listed. The main statement below reads this table
-- (broadcast: ~2M rows, two columns) and only moves primary_location and that location's place in `locations`: type,
-- best_oa_location, open_access, apc_list and topics_key keep reading the sort's first location.
-- pick_keys = XXHASH64(provenance, native_id) of each evidence-backed journal copy on the work.
-- Revert: git revert this commit, then DROP TABLE openalex.works.journal_copy_primary_picks.
-- oxjob #2250: three more evidence tiers, read only from repository records off the hosts other jobs own (IRDB, DOAJ,
-- PubMed / PMC / Europe PMC, RePEc, CyberLeninka, Persee): (1) an ISSN anywhere in the string is one of the journal's;
-- (2) the names match after repairing mojibake (latin-1 read as UTF-8), unescaping HTML entities, folding accents,
-- dropping 'and' / '&' and a leading 'the', and cutting a trailing citation (volume, issue, pages, year); (3) a HAL
-- record's dalloz.fr Document?id=<CODE>/ names the journal by Dalloz's code. Bars in oxjob #2250 (fresh draws).
-- The repaired key is one expression written out three times (journal names, record string, record string cut):
-- edit it in oxjob #2250's work/gen_cell.py and regenerate, never by hand in one place.
-- Revert #2250 alone: git revert its commit; the next run rebuilds the table without the three tiers.
-- oxjob #2250 tier 4: openalex.works.journal_string_judged_pairs lists (record string, journal) pairs a judge read as
-- the same serial (renames, abbreviations, translations): Opus 5.5 once per pair on the full strings (rubric 2250-t4-v1,
-- high confidence) and again on the cut text alone when the cut removed text (2250-t4-cut-v1), so an author-first
-- citation cut to the author never matches. string_key is the record string's norm_cut (below). Built by #2250 work/.
-- Revert tier 4 alone: git revert its commit (the pairs table can stay; nothing else reads it).
-- oxjob #2249: RePEc's own series names the journal. A RePEc record (oai:RePEc:<archive>:<series>:<item>) carries no
-- journal string, so the evidence above never fires for RePEc's ~120K journal articles. RePEc's series record
-- (IDEAS series page: name and ISSN) does name the journal; openalex.works.repec_series_journals lists each RePEc
-- article series confirmed to be one journal source (ISSN, or the same title; built by oxjob #2249 work/, rebuildable).
-- A journal candidate copy on that source is a pick when the work holds a RePEc 'article' record of that series
-- (working-paper series are raw_type 'preprint' and never listed). Bar 15 Nov 2026: see oxjob #2249 README.
-- Revert #2249 alone: delete the repec CTE and the UNION ALL branch, restore the final SELECT (one SELECT ... GROUP BY
-- c.work_id) and candidates' select list (drop l.source_id); then DROP TABLE openalex.works.repec_series_journals.
CREATE OR REPLACE TABLE openalex.works.journal_copy_primary_picks
COMMENT 'oxjob #2209: per work with no Crossref location, the journal-source locations (XXHASH64(provenance, native_id)) that another record on the work names by journal title or ISSN, or (oxjob #2249) that the series of a RePEc article record on the work is confirmed to be (repec_series_journals). CreateWorksBase serves the first as primary_location when the sort put a repository copy first. Rebuilt by CreateWorksBase each run.'
AS
WITH journals AS (
  SELECT id AS source_id,
    ARRAY_DISTINCT(FILTER(TRANSFORM(CONCAT(ARRAY(display_name), COALESCE(alternate_titles, ARRAY())),
                                    x -> REGEXP_REPLACE(LOWER(x), '[^\\p{L}\\p{N}]', '')), x -> LENGTH(x) >= 4)) AS name_keys,
    TRANSFORM(COALESCE(issns, ARRAY()), x -> REGEXP_REPLACE(LOWER(x), '[^0-9x]', '')) AS issn_keys,
    -- oxjob #2250 tier 2: the same names under the repaired key (min 4 characters, as above)
    ARRAY_DISTINCT(FILTER(TRANSFORM(CONCAT(ARRAY(display_name), COALESCE(alternate_titles, ARRAY())),
                                    x -> REGEXP_REPLACE(REGEXP_REPLACE(TRANSLATE(LOWER(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(CASE WHEN x RLIKE '[ÃÂ]' AND x NOT RLIKE '[^\\x{00}-\\x{FF}]' AND NOT CONTAINS(DECODE(ENCODE(x, 'ISO-8859-1'), 'UTF-8'), '\uFFFD') THEN DECODE(ENCODE(x, 'ISO-8859-1'), 'UTF-8') ELSE x END, '&amp;', '&'), '&#038;', '&'), '&#38;', '&'), '&quot;', '"'), '&#39;', ''''), '&#039;', ''''), '&apos;', ''''), '&#8217;', ''''), '&lt;', '<'), '&gt;', '>'), '&nbsp;', ' ')), 'áàâäãåāăąéèêëēėęěíìîïīįóòôöõøōőúùûüūůűçćčñńňšśşžźżýÿğłđ', 'aaaaaaaaaeeeeeeeeiiiiiioooooooouuuuuuucccnnnssszzzyygld'), '&|\\b(and|et|und|y|e)\\b|^\\s*the\\b', ''), '[^\\p{L}\\p{N}]', '')), x -> LENGTH(x) >= 4)) AS norm_keys
  FROM openalex.sources.sources
  -- oxjob #2494: a conference proceedings series is a venue in its own right, like a journal (help.openalex.org Source
  -- types: 'Proceedings and abstracts from academic conferences'). 1,289 proceedings series typed journal until 21 Nov 2026
  -- (Proc. SPIE, AIP Conf. Proc., J. Phys. Conf. Ser.) keep their picks; conference series get the same rule.
  WHERE type IN ('journal', 'conference')
),
loc AS (
  SELECT work_id, IF(provenance = 'repo_backfill', 'repo', provenance) AS provenance, native_id, source_id, source_name, endpoint_id,
    LOWER(COALESCE(landing_page_url, pdf_url)) AS link_url -- oxjob #1006
  FROM openalex.works.locations_mapped_curated
  WHERE work_id IS NOT NULL
),
-- the journal's own copy: a location on a journal source that is not a repository-harvest record, or is one from an
-- endpoint the registry flags as the journal's host (the same gate as host_type 'publisher', oxjob #1409)
candidates AS (
  SELECT /*+ BROADCAST(j), BROADCAST(jh), BROADCAST(dl) */ l.work_id, l.provenance, l.native_id, l.source_id, j.name_keys, j.issn_keys,
    j.norm_keys
  FROM loc l
  JOIN journals j ON j.source_id = l.source_id
  LEFT JOIN (SELECT DISTINCT endpoint_id FROM openalex.sources.endpoint_to_source WHERE is_journal_host) jh
    ON jh.endpoint_id = l.endpoint_id
  LEFT ANTI JOIN (SELECT DISTINCT work_id FROM openalex.works.locations_mapped_curated WHERE provenance = 'crossref') cr
    ON cr.work_id = l.work_id
  -- oxjob #1006: a copy on a dead or squatted host (dead_link_hosts, cell above) is never served as primary
  LEFT ANTI JOIN openalex.works.dead_link_hosts dl
    ON dl.host = REGEXP_EXTRACT(l.link_url, '^[a-z]+://(?:www[.])?([^/:?#]+)', 1)
    AND (dl.url_rlike IS NULL OR l.link_url RLIKE dl.url_rlike)
    AND dl.rewrite IS NULL
  WHERE l.provenance <> 'crossref' AND (l.provenance <> 'repo' OR jh.endpoint_id IS NOT NULL)
),
-- what the work's other records call their journal (never a MAG record: MAG's string is where the assignment came from)
evidence AS (
  SELECT work_id, provenance, native_id, source_id,
    REGEXP_REPLACE(LOWER(source_name), '[^\\p{L}\\p{N}]', '') AS raw_name,
    REGEXP_REPLACE(LOWER(source_name), '[^0-9x]', '') AS raw_issn,
    -- oxjob #2250: tiers 1-3 read only repository records off the hosts other jobs own, and never Cairn's book-collection
    -- records ('Repères | 5e éd. | 2004-02-01 | p. 5-6 | 0993-7625', 'BàO La Boîte à Outils | - | 2020-03-18 | ...'): a
    -- chapter in a publisher's book series, which the registry types journal (journal items carry month and issue there)
    provenance = 'repo' AND COALESCE(source_id, 0) NOT IN (7407056385, 4306401280, 4306525036, 2764455111, 4306400806, 4306401271, 4306401404, 4406922388, 4406923040)
      AND NOT source_name RLIKE '^[^|]+ [|] (-|[^|]*[ée]d[.]?[^|]*) [|] [0-9]{4}-[0-9]{2}-[0-9]{2} [|]' AS tiered,
    TRANSFORM(REGEXP_EXTRACT_ALL(IF(provenance = 'repo', source_name, ''), '(?<![0-9])([0-9]{4}-[0-9]{3}[0-9Xx])(?![0-9Xx])', 1),
              x -> REGEXP_REPLACE(LOWER(x), '[^0-9x]', '')) AS issns_in_name,
    IF(provenance = 'repo', REGEXP_REPLACE(REGEXP_REPLACE(TRANSLATE(LOWER(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(CASE WHEN source_name RLIKE '[ÃÂ]' AND source_name NOT RLIKE '[^\\x{00}-\\x{FF}]' AND NOT CONTAINS(DECODE(ENCODE(source_name, 'ISO-8859-1'), 'UTF-8'), '\uFFFD') THEN DECODE(ENCODE(source_name, 'ISO-8859-1'), 'UTF-8') ELSE source_name END, '&amp;', '&'), '&#038;', '&'), '&#38;', '&'), '&quot;', '"'), '&#39;', ''''), '&#039;', ''''), '&apos;', ''''), '&#8217;', ''''), '&lt;', '<'), '&gt;', '>'), '&nbsp;', ' ')), 'áàâäãåāăąéèêëēėęěíìîïīįóòôöõøōőúùûüūůűçćčñńňšśşžźżýÿğłđ', 'aaaaaaaaaeeeeeeeeiiiiiioooooooouuuuuuucccnnnssszzzyygld'), '&|\\b(and|et|und|y|e)\\b|^\\s*the\\b', ''), '[^\\p{L}\\p{N}]', ''), NULL) AS norm_name,
    IF(provenance = 'repo', REGEXP_REPLACE(REGEXP_REPLACE(TRANSLATE(LOWER(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(CASE WHEN REGEXP_EXTRACT(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(CASE WHEN source_name RLIKE '[ÃÂ]' AND source_name NOT RLIKE '[^\\x{00}-\\x{FF}]' AND NOT CONTAINS(DECODE(ENCODE(source_name, 'ISO-8859-1'), 'UTF-8'), '\uFFFD') THEN DECODE(ENCODE(source_name, 'ISO-8859-1'), 'UTF-8') ELSE source_name END, '&amp;', '&'), '&#038;', '&'), '&#38;', '&'), '&quot;', '"'), '&#39;', ''''), '&#039;', ''''), '&apos;', ''''), '&#8217;', ''''), '&lt;', '<'), '&gt;', '>'), '&nbsp;', ' '), '^(.*?)(\\s*[,.;:(|\\[]\\s*(vol|v\\.|no\\.|nr|n[ºo°]|núm|num|tome|t\\.|bd|band|jg|jahrg|issue|issn|[0-9])|\\s+[0-9]{1,4}\\s*[(,.]|\\s+[0-9]{4}\\b|$)', 1) RLIKE '[ÃÂ]' AND REGEXP_EXTRACT(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(CASE WHEN source_name RLIKE '[ÃÂ]' AND source_name NOT RLIKE '[^\\x{00}-\\x{FF}]' AND NOT CONTAINS(DECODE(ENCODE(source_name, 'ISO-8859-1'), 'UTF-8'), '\uFFFD') THEN DECODE(ENCODE(source_name, 'ISO-8859-1'), 'UTF-8') ELSE source_name END, '&amp;', '&'), '&#038;', '&'), '&#38;', '&'), '&quot;', '"'), '&#39;', ''''), '&#039;', ''''), '&apos;', ''''), '&#8217;', ''''), '&lt;', '<'), '&gt;', '>'), '&nbsp;', ' '), '^(.*?)(\\s*[,.;:(|\\[]\\s*(vol|v\\.|no\\.|nr|n[ºo°]|núm|num|tome|t\\.|bd|band|jg|jahrg|issue|issn|[0-9])|\\s+[0-9]{1,4}\\s*[(,.]|\\s+[0-9]{4}\\b|$)', 1) NOT RLIKE '[^\\x{00}-\\x{FF}]' AND NOT CONTAINS(DECODE(ENCODE(REGEXP_EXTRACT(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(CASE WHEN source_name RLIKE '[ÃÂ]' AND source_name NOT RLIKE '[^\\x{00}-\\x{FF}]' AND NOT CONTAINS(DECODE(ENCODE(source_name, 'ISO-8859-1'), 'UTF-8'), '\uFFFD') THEN DECODE(ENCODE(source_name, 'ISO-8859-1'), 'UTF-8') ELSE source_name END, '&amp;', '&'), '&#038;', '&'), '&#38;', '&'), '&quot;', '"'), '&#39;', ''''), '&#039;', ''''), '&apos;', ''''), '&#8217;', ''''), '&lt;', '<'), '&gt;', '>'), '&nbsp;', ' '), '^(.*?)(\\s*[,.;:(|\\[]\\s*(vol|v\\.|no\\.|nr|n[ºo°]|núm|num|tome|t\\.|bd|band|jg|jahrg|issue|issn|[0-9])|\\s+[0-9]{1,4}\\s*[(,.]|\\s+[0-9]{4}\\b|$)', 1), 'ISO-8859-1'), 'UTF-8'), '\uFFFD') THEN DECODE(ENCODE(REGEXP_EXTRACT(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(CASE WHEN source_name RLIKE '[ÃÂ]' AND source_name NOT RLIKE '[^\\x{00}-\\x{FF}]' AND NOT CONTAINS(DECODE(ENCODE(source_name, 'ISO-8859-1'), 'UTF-8'), '\uFFFD') THEN DECODE(ENCODE(source_name, 'ISO-8859-1'), 'UTF-8') ELSE source_name END, '&amp;', '&'), '&#038;', '&'), '&#38;', '&'), '&quot;', '"'), '&#39;', ''''), '&#039;', ''''), '&apos;', ''''), '&#8217;', ''''), '&lt;', '<'), '&gt;', '>'), '&nbsp;', ' '), '^(.*?)(\\s*[,.;:(|\\[]\\s*(vol|v\\.|no\\.|nr|n[ºo°]|núm|num|tome|t\\.|bd|band|jg|jahrg|issue|issn|[0-9])|\\s+[0-9]{1,4}\\s*[(,.]|\\s+[0-9]{4}\\b|$)', 1), 'ISO-8859-1'), 'UTF-8') ELSE REGEXP_EXTRACT(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(CASE WHEN source_name RLIKE '[ÃÂ]' AND source_name NOT RLIKE '[^\\x{00}-\\x{FF}]' AND NOT CONTAINS(DECODE(ENCODE(source_name, 'ISO-8859-1'), 'UTF-8'), '\uFFFD') THEN DECODE(ENCODE(source_name, 'ISO-8859-1'), 'UTF-8') ELSE source_name END, '&amp;', '&'), '&#038;', '&'), '&#38;', '&'), '&quot;', '"'), '&#39;', ''''), '&#039;', ''''), '&apos;', ''''), '&#8217;', ''''), '&lt;', '<'), '&gt;', '>'), '&nbsp;', ' '), '^(.*?)(\\s*[,.;:(|\\[]\\s*(vol|v\\.|no\\.|nr|n[ºo°]|núm|num|tome|t\\.|bd|band|jg|jahrg|issue|issn|[0-9])|\\s+[0-9]{1,4}\\s*[(,.]|\\s+[0-9]{4}\\b|$)', 1) END, '&amp;', '&'), '&#038;', '&'), '&#38;', '&'), '&quot;', '"'), '&#39;', ''''), '&#039;', ''''), '&apos;', ''''), '&#8217;', ''''), '&lt;', '<'), '&gt;', '>'), '&nbsp;', ' ')), 'áàâäãåāăąéèêëēėęěíìîïīįóòôöõøōőúùûüūůűçćčñńňšśşžźżýÿğłđ', 'aaaaaaaaaeeeeeeeeiiiiiioooooooouuuuuuucccnnnssszzzyygld'), '&|\\b(and|et|und|y|e)\\b|^\\s*the\\b', ''), '[^\\p{L}\\p{N}]', ''), NULL) AS norm_cut,
    IF(provenance = 'repo', UPPER(REGEXP_EXTRACT(source_name, 'dalloz[.]fr/.*[?&]id=([A-Za-z0-9_-]+)/', 1)), NULL) AS dalloz_code
  FROM loc
  WHERE provenance NOT IN ('mag', 'crossref') AND source_name IS NOT NULL
    AND REGEXP_REPLACE(LOWER(source_name), '[^\\p{L}\\p{N}]', '') <> ''
),
-- oxjob #2249: works holding a RePEc article record whose series is confirmed to be a journal source
repec AS (
  SELECT /*+ BROADCAST(rs) */ DISTINCT r.work_id, rs.source_id
  FROM openalex.works.locations_mapped_curated r
  JOIN openalex.works.repec_series_journals rs
    ON rs.series_key = LOWER(CONCAT(GET(SPLIT(r.native_id, ':'), 2), ':', GET(SPLIT(r.native_id, ':'), 3)))
  WHERE r.source_id = 4306401271 AND r.native_id LIKE 'oai:RePEc:%' AND r.raw_type = 'article' AND r.work_id IS NOT NULL
)
SELECT work_id, ARRAY_SORT(COLLECT_SET(pick_key)) AS pick_keys
FROM (
SELECT /*+ BROADCAST(jp) */ c.work_id, XXHASH64(c.provenance, c.native_id) AS pick_key
FROM candidates c
JOIN evidence e
  ON e.work_id = c.work_id
  AND NOT (e.provenance = c.provenance AND e.native_id = c.native_id)
LEFT JOIN (VALUES ('RECUEIL', 4306525509), ('AJDA', 4306517994), ('AJDA', 4306500668), ('RTDCIV', 4306528868), ('RTDCOM', 4306528870), ('RTDCOM', 4306528490), ('DS', 4306508428), ('RDI', 4306528167), ('REVSOC', 4306528265), ('RTDEUR', 4306528871), ('RTDEUR', 4306528491), ('RSC', 4210170923), ('RDSS', 4306528174), ('AJPEN', 4306500670), ('REVTRAV', 4306528164), ('RFDA', 4306528339), ('AJDI', 4306500667), ('REVDIP', 4210205640), ('AJCA', 4306501234), ('RMC', 4306528318), ('RMC', 4306528206), ('AJFAM', 4306500669), ('AJCT', 4306500666), ('DIPIT', 4306507537), ('JT', 4306516542), ('AJFP', 4306501235), ('JS', 4306516527), ('CONSTIT', 4306506946), ('JA', 4306516523), ('JAC', 4306516522), ('CAHJU', 4210168999)) AS dz(code, source_id)
  ON e.tiered AND dz.code = e.dalloz_code AND dz.source_id = c.source_id
-- oxjob #2250 tier 4: a judged pair (the record's cut string, this journal)
LEFT JOIN openalex.works.journal_string_judged_pairs jp
  ON e.tiered AND jp.string_key = e.norm_cut AND jp.source_id = c.source_id
WHERE ARRAY_CONTAINS(c.name_keys, e.raw_name)
   OR EXISTS(c.name_keys, n -> LENGTH(n) >= 8 AND STARTSWITH(e.raw_name, n))
   OR (LENGTH(e.raw_name) <= 9 AND LENGTH(e.raw_issn) = 8 AND ARRAY_CONTAINS(c.issn_keys, e.raw_issn))
   -- oxjob #2250 tier 1: an ISSN anywhere in the string is one of the journal's. Not from Lund University Publications
   -- (S4306400536): its 'ISSN: ...' field names the wrong journal too often (2 of 6 sampled, e.g. Botaniska Notiser
   -- filed under Nordic Journal of Botany's ISSN)
   OR (e.tiered AND e.source_id <> 4306400536 AND EXISTS(e.issns_in_name, x -> ARRAY_CONTAINS(c.issn_keys, x)))
   -- oxjob #2250 tier 2: the repaired names are equal, or the string starts with the journal's name (8+ characters),
   -- or the string cut before its citation tail is the journal's name
   OR (e.tiered AND (ARRAY_CONTAINS(c.norm_keys, e.norm_name) OR ARRAY_CONTAINS(c.norm_keys, e.norm_cut)
                     OR EXISTS(c.norm_keys, n -> LENGTH(n) >= 8 AND STARTSWITH(e.norm_name, n))))
   -- oxjob #2250 tier 3: Dalloz's code for the journal
   OR dz.code IS NOT NULL
   -- oxjob #2250 tier 4: a judge read the record's string as this journal's name
   OR jp.source_id IS NOT NULL
UNION ALL
-- oxjob #2249: the RePEc series names this journal
SELECT c.work_id, XXHASH64(c.provenance, c.native_id) AS pick_key
FROM candidates c
JOIN repec rp ON rp.work_id = c.work_id AND rp.source_id = c.source_id
)
GROUP BY work_id;

In [ ]:
-- oxjob #1225 (ZD 2991, 5779): items on an institution's Figshare portal (Deakin's dro.deakin.edu.au, Monash's
-- bridges.monash.edu, ICES's library, Swinburne, La Trobe ...) reach us twice: figshare.com's own OAI record
-- (oai:figshare.com:article/<N>, filed under the generic Figshare source S4377196282, version submittedVersion) and the
-- institution's DataCite DOI, filed under the institution's repository source, whose URL is the portal's page of the same
-- article <N>. The sort ranks the figshare.com record first (it states a version, the DOI copy none), so 335,379 works
-- (34,566 visible, night 2026-11-22) served generic Figshare as primary_location although the institution's own copy is on
-- the work, e.g. W6889719546 (10.26187/deakin.28693058). This table lists, per such work, its figshare.com records and the
-- portal's DataCite copies: same Figshare article id at the end of the DataCite record's URL (bar 22 Nov 2026: a random
-- 50 visible works against Figshare's own API, 50 of 50 the same item; oxjob #1225 EXPLORE.md). The main statement below
-- serves the first of those DataCite copies in sort order as primary_location when the sort put one of the work's
-- figshare.com records of that article first. Only primary_location and that location's place in `locations` move:
-- type, best_oa_location, open_access and topics_key keep reading the sort (6,325 visible works' type would change if it
-- followed the DataCite copy, mostly to 'other'). OA: 0 works' primary copy goes from open to closed.
-- Revert: delete this cell, the fpp join and BROADCAST(fpp) in the main statement, and the fpp branches in
-- primary_location and the locations reorder; then DROP TABLE openalex.works.figshare_portal_primary_picks.
CREATE OR REPLACE TABLE openalex.works.figshare_portal_primary_picks
COMMENT 'oxjob #1225: per work holding a figshare.com OAI record (S4377196282) and the DataCite copy of the same Figshare article on an institution portal, the figshare.com native_ids and the DataCite native_ids. CreateWorksBase serves the DataCite copy as primary_location when the sort put the figshare.com record first. Rebuilt by CreateWorksBase each run.'
AS
WITH fs AS (
  SELECT work_id, native_id,
    REGEXP_EXTRACT(native_id, '^oai:figshare[.]com:article/([0-9]+)$', 1) AS article_id
  FROM openalex.works.locations_mapped_curated
  WHERE work_id IS NOT NULL AND provenance IN ('repo', 'repo_backfill') AND source_id = 4377196282
    AND native_id LIKE 'oai:figshare.com:article/%'
),
-- the institution's DataCite copy: its record's URL is the portal's page of the article, ending /<N> or /<N>/<version>
portal AS (
  SELECT l.work_id, l.native_id,
    REGEXP_EXTRACT(u.url, '/([0-9]+)(/[0-9]+)?/?$', 1) AS article_id
  FROM openalex.works.locations_mapped_curated l
  LATERAL VIEW EXPLODE(l.urls) t AS u
  WHERE l.work_id IS NOT NULL AND l.provenance = 'datacite' AND l.source_id IS NOT NULL AND l.source_id <> 4377196282
)
SELECT fs.work_id,
  COLLECT_SET(fs.native_id) AS figshare_native_ids,
  COLLECT_SET(portal.native_id) AS pick_native_ids
FROM fs
JOIN portal ON portal.work_id = fs.work_id AND portal.article_id = fs.article_id AND fs.article_id <> ''
GROUP BY fs.work_id;

In [ ]:
-- oxjob #2241 part 2: works dated 1 January only because their Crossref record states a bare year, while the work's own
-- PubMed record dates it later that same year (W4288470994, 10.1039/d2dt01359b: Crossref issued [2022], served 2022-01-01,
-- PubMed 2022-08-23). Crossref outranks PubMed for the date, and a year-only date is padded to 1 January, so ~1.03M works
-- served a day no source states. One row per such work: the padded date and the PubMed date. The main statement takes the
-- PubMed date only where the picked date is still that padded 1 January. No year changes (same year by construction) and
-- no stated date loses: a Crossref record stating a month or day that year (issued [2022, 1], online 2022-01-01) or a DOI
-- created or deposited on that 1 January is not padded and keeps its date. Part 1 (walden 499775a4, Crossref.py) already
-- lets a finer date inside the same Crossref record win.
-- A PubMed record counts only if it names the Crossref record's DOI, or names no DOI and has the same normalized title
-- (never another paper's PMID, as #1158 / #2155). Several qualifying PubMed records: the earliest, and only if it is later
-- than 1 January (a PubMed record dated 1 January that year leaves the work alone).
-- Built here as its own table (memory: a new pick inside the main statement froze Works_Base, #1931). Bar and sizes:
-- oxjob #2241 README. Revert: delete this cell and the join and CASE in the main statement (grep 2241), drop the table.
CREATE OR REPLACE TABLE openalex.works.crossref_year_only_pubmed_dates
COMMENT 'oxjob #2241: per work whose date is a Crossref bare year padded to 1 January while its own PubMed record dates it later that year: work_id, padded_date, pubmed_date. Rebuilt by CreateWorksBase each run.'
AS
WITH cr_dates AS (
  SELECT
    LOWER(native_id) AS doi,
    ARRAY(
      NAMED_STRUCT('d', TRY_TO_DATE(FORMAT_STRING('%04d-%02d-%02d', GET(GET(issued.`date-parts`, 0), 0), COALESCE(GET(GET(issued.`date-parts`, 0), 1), 1), COALESCE(GET(GET(issued.`date-parts`, 0), 2), 1))), 'p', SIZE(GET(issued.`date-parts`, 0))),
      NAMED_STRUCT('d', TRY_TO_DATE(FORMAT_STRING('%04d-%02d-%02d', GET(GET(published.`date-parts`, 0), 0), COALESCE(GET(GET(published.`date-parts`, 0), 1), 1), COALESCE(GET(GET(published.`date-parts`, 0), 2), 1))), 'p', SIZE(GET(published.`date-parts`, 0))),
      NAMED_STRUCT('d', TRY_TO_DATE(FORMAT_STRING('%04d-%02d-%02d', GET(GET(`published-online`.`date-parts`, 0), 0), COALESCE(GET(GET(`published-online`.`date-parts`, 0), 1), 1), COALESCE(GET(GET(`published-online`.`date-parts`, 0), 2), 1))), 'p', SIZE(GET(`published-online`.`date-parts`, 0))),
      NAMED_STRUCT('d', TRY_TO_DATE(FORMAT_STRING('%04d-%02d-%02d', GET(GET(`published-print`.`date-parts`, 0), 0), COALESCE(GET(GET(`published-print`.`date-parts`, 0), 1), 1), COALESCE(GET(GET(`published-print`.`date-parts`, 0), 2), 1))), 'p', SIZE(GET(`published-print`.`date-parts`, 0))),
      NAMED_STRUCT('d', TRY_TO_DATE(FORMAT_STRING('%04d-%02d-%02d', GET(GET(approved.`date-parts`, 0), 0), COALESCE(GET(GET(approved.`date-parts`, 0), 1), 1), COALESCE(GET(GET(approved.`date-parts`, 0), 2), 1))), 'p', SIZE(GET(approved.`date-parts`, 0)))
    ) AS pd,
    TO_DATE(SUBSTR(created.`date-time`, 1, 10)) AS cr_created,
    TO_DATE(SUBSTR(deposited.`date-time`, 1, 10)) AS cr_deposited
  FROM openalex.crossref.crossref_deduplicated
),
-- Crossref records whose earliest bare year is padded to 1 January and stated by nothing finer that year
cr_year_only AS (
  SELECT doi, y1 AS padded_date
  FROM (SELECT doi, pd, cr_created, cr_deposited,
               ARRAY_MIN(TRANSFORM(FILTER(pd, x -> x.d IS NOT NULL AND x.p = 1), x -> x.d)) AS y1
        FROM cr_dates)
  WHERE y1 IS NOT NULL
    AND NOT EXISTS(pd, x -> x.d IS NOT NULL AND x.p >= 2 AND YEAR(x.d) = YEAR(y1))
    AND COALESCE(cr_created <> y1, TRUE) AND COALESCE(cr_deposited <> y1, TRUE)
),
-- the work's Crossref locations still dated that padded 1 January
cr AS (
  SELECT l.work_id, LOWER(l.native_id) AS cr_doi, l.normalized_title AS c_title, l.published_date AS padded_date
  FROM openalex.works.locations_mapped_curated l
  JOIN cr_year_only y ON y.doi = LOWER(l.native_id) AND y.padded_date = l.published_date
  WHERE l.provenance = 'crossref' AND MONTH(l.published_date) = 1 AND DAY(l.published_date) = 1
),
pm AS (
  SELECT work_id, LOWER(best_doi) AS pdoi, normalized_title AS p_title, published_date AS pm_date
  FROM openalex.works.locations_mapped_curated
  WHERE provenance = 'pubmed' AND native_id_namespace = 'pmid' AND published_date IS NOT NULL
    AND work_id IN (SELECT work_id FROM cr)
)
SELECT c.work_id, MIN(c.padded_date) AS padded_date, MIN(p.pm_date) AS pubmed_date
FROM cr c
JOIN pm p ON p.work_id = c.work_id
  AND (COALESCE(p.pdoi = c.cr_doi, FALSE)
       OR (p.pdoi IS NULL AND COALESCE(LENGTH(c.c_title) >= 15 AND c.c_title = p.p_title, FALSE)))
  AND YEAR(p.pm_date) = YEAR(c.padded_date)
GROUP BY c.work_id
HAVING MIN(p.pm_date) > MIN(c.padded_date) AND COUNT(DISTINCT c.padded_date) = 1

In [ ]:
-- oxjob #1154: works whose picked date is a placeholder a record carries instead of a real date. OJS deposits with no
-- date set reach Crossref as 1970-01-01 / 1969-12-31 (the Unix epoch), MAG carries 1899-12-30 (Excel's day zero), and
-- some publishers deposit 1400 / 1500 / 1900 for "unknown" (W2007178635 Journal of Networks 2014: 1969-12-31;
-- W2985925327 AFIPS 1978: 1899-12-30; W4236025242 Endocrine Journal 2010: 1500-01-01; Frontiers in Neuroscience
-- conference abstracts: 1900-01-01). A record date is a placeholder when it is
--   1969-12-31 or 1899-12-30 (nothing pads a date to them), or, on a record that is not MAG's (MAG pads a bare year to
--   1 January, so its 1970-01-01 can be a real 1970: W1578004390, an Apollo 11 lunar-sample paper):
--   1970-01-01 on a journal whose records start (1st percentile of its other dates, >= 20 records) after 1972;
--   1900 / 1800 / 1000-01-01 on a journal whose records start in 1950 or later (journals from 1903-1949 hold real 1900
--   papers dated by year alone, e.g. Trans. AMS and ApJ; e-book platforms hold reissues of old books);
--   1500 / 1400-01-01 on a source that is not a repository and starts more than 2 years later (Bloomsbury deposits its
--   Drama Online plays as 1400).
-- One row per work whose picked date (lowest priority, then earliest, as get_highest_priority_value) is such a
-- placeholder and no other record states that same date: the placeholder and its replacement, NULL unless the work's
-- PubMed record dates it, or its DataCite / MAG record does and a record of another kind states the same year. NULL
-- means no date ("No year beats a doubtful year", Jason 26 Oct 2026, plans/xpac-integration.md). Bar (oxjob #1154,
-- fresh random draws checked against each work's own page): the placeholder is wrong on 145/145; a repository or
-- Crossref date would be right on 71/94 (they carry the day a back issue was uploaded) and a lone DataCite / MAG date
-- on 13/15; the dates this keeps are right on 25/25. The main statement swaps only where its own pick still equals
-- placeholder_date.
-- oxjob #1154 (21 Nov 2026): an AFIPS conference paper (DOI 10.1109/afips.YYYY.N, the IEEE Computer Society's digitised
-- AFIPS proceedings) that no record dates but MAG's 1899-12-30 takes 1 January of its DOI's year (2,031 works, e.g.
-- W2575251107 1965, W2985925327 1978). The IEEE CS Digital Library record states that year on 58 of 58 random such
-- works that have one, and MAG's own real dates on AFIPS DOIs agree with it on 235 of 238 (3 off by one year).
-- Revert this part alone: drop afips_doi_year, afips_years and their WHEN below.
-- oxjob #1154 step 3 (21 Nov 2026): a work still left with no date takes 1 January of the year its DOI's landing page
-- states in citation_date (openalex.landing_page.meta_tags, the page's latest fetch), when the page names one such year,
-- its citation_title is the work's title, and the year is not 1969 / 1970 (OJS repeats the epoch placeholder there) nor
-- the fetch year or later (some pages print the day they are served). On OJS pages citation_date is the issue's year;
-- their DC.Date.issued is often the day a back issue was uploaded (Porta Linguarum 32, W2999898476: citation_date 2019,
-- DC.Date.issued 2021-04-18), so it is not read. About 33,500 works (dry run 21 Nov). Bar: the year printed on the
-- article's own PDF matches on 31 of 31 random such works whose PDF prints one, 0 wrong; within a journal, year minus
-- volume is one constant for 88% of them (upload days would scatter it). Revert this part alone: the final SELECT
-- becomes placeholder_rows' (drop undated_dois, pages, page_years, doi_titles, doi and title_key).
-- oxjob #1154 step 4 (21 Nov 2026): a work step 3 still leaves with no date takes 1 January of a year two independent
-- records state: every dated record of the other works in its issue (same source, volume and issue as its Crossref
-- record) is MAG's and states that one year, and the earliest year its own repository records state (DOAJ, Redalyc,
-- LA Referencia, an institutional repository) is the same. Either alone is not enough: Crossref / repository dates
-- often carry the year a back issue was uploaded (issue siblings with a non-MAG record matched the work's landing-page
-- year on 63%, its DOAJ record on 77%), and one MAG sibling alone matched 3 of 10. Together they matched the landing-page
-- year (step 3) on 468 of 471 works that have both (3 one year off, 0 more). About 209 works more (dry run 21 Nov):
-- Nova Tellus 132 (each DOI states the same year), Canadian Journal of Higher Education 8 (W2167973157 2007, its page
-- on the journal's new host), Journal of Networks 3 (W2007178635 2014). Revert this part alone: drop issue_rule from the
-- final COALESCE (and undated_issue_keys, issue_years, issue_rule, issue_keys, repo_min_year, source_id/volume/issue).
-- Built here as its own table (a new pick inside the main statement froze Works_Base, #1931). Revert: delete this cell
-- and the join and WHEN in the main statement (grep 1154), drop the table.
CREATE OR REPLACE TABLE openalex.works.placeholder_publication_dates
COMMENT 'oxjob #1154: per work whose picked publication date is a placeholder (1970-01-01 epoch, 1899-12-30, a 1400/1500/1900 year before its journal existed): work_id, placeholder_date, new_date (PubMed/DataCite/MAG date, the year of an AFIPS DOI, of the DOI landing page citation_date, or that the MAG-dated works of its issue and its own repository records both state, or NULL). Rebuilt by CreateWorksBase each run.'
AS
WITH placeholder_values AS (
  SELECT EXPLODE(ARRAY(DATE'1970-01-01', DATE'1969-12-31', DATE'1899-12-30', DATE'1900-01-01', DATE'1800-01-01',
                       DATE'1500-01-01', DATE'1400-01-01', DATE'1000-01-01')) AS d
),
source_first_years AS (
  SELECT l.source_id, PERCENTILE_APPROX(YEAR(l.published_date), 0.01) AS first_year
  FROM openalex.works.locations_mapped_curated l
  WHERE l.source_id IS NOT NULL AND l.published_date IS NOT NULL AND YEAR(l.published_date) BETWEEN 1000 AND 9999
    AND l.published_date NOT IN (SELECT d FROM placeholder_values)
  GROUP BY l.source_id
  HAVING COUNT(*) >= 20
),
candidate_works AS (
  SELECT DISTINCT l.work_id
  FROM openalex.works.locations_mapped_curated l
  WHERE l.published_date IN (SELECT d FROM placeholder_values)
),
records AS (
  SELECT l.work_id, l.provenance, p.priority, l.published_date,
    NULLIF(REGEXP_EXTRACT(LOWER(COALESCE(l.best_doi, l.merge_key.doi)), '^10\\.1109/afips\\.([0-9]{4})\\.', 1), '') AS afips_doi_year,
    LOWER(COALESCE(l.best_doi, l.merge_key.doi)) AS doi,
    l.source_id, LOWER(TRIM(l.volume)) AS volume, LOWER(TRIM(l.issue)) AS issue,
    REGEXP_REPLACE(REGEXP_REPLACE(LOWER(l.title), '&[a-z0-9#]+;', ''), '[^\\p{L}\\p{N}]', '') AS title_key,
    (l.published_date IN (DATE'1969-12-31', DATE'1899-12-30')
      OR (l.provenance <> 'mag' AND (
        -- oxjob #2494: conference series too (a 1970-01-01 date on a series whose first year is later is a placeholder).
           (l.published_date = DATE'1970-01-01' AND s.type IN ('journal', 'conference') AND fy.first_year > 1972)
        OR (l.published_date IN (DATE'1900-01-01', DATE'1800-01-01', DATE'1000-01-01') AND s.type IN ('journal', 'conference') AND fy.first_year >= 1950)
        OR (l.published_date IN (DATE'1500-01-01', DATE'1400-01-01') AND COALESCE(s.type, '') <> 'repository'
            AND fy.first_year > YEAR(l.published_date) + 2)))) IS TRUE AS is_placeholder
  FROM openalex.works.locations_mapped_curated l
  JOIN candidate_works c ON c.work_id = l.work_id
  LEFT JOIN openalex.system.priority_table p ON p.provenance = l.provenance
  LEFT JOIN source_first_years fy ON fy.source_id = l.source_id
  LEFT JOIN openalex.sources.sources s ON s.id = l.source_id
  WHERE l.published_date IS NOT NULL AND YEAR(l.published_date) < 10000
),
per_work AS (
  SELECT work_id,
    MIN_BY(published_date, NAMED_STRUCT('p', priority, 'd', CAST(published_date AS STRING))) AS picked_date,
    COLLECT_SET(afips_doi_year) AS afips_years,
    COLLECT_SET(CASE WHEN doi IS NOT NULL THEN NAMED_STRUCT('doi', doi, 'title_key', title_key) END) AS doi_titles,
    -- step 4 (header above): the issue its Crossref record names, and the earliest year its repository records state
    COLLECT_SET(CASE WHEN provenance = 'crossref' AND source_id IS NOT NULL AND COALESCE(volume, '') <> '' AND COALESCE(issue, '') <> ''
                     THEN NAMED_STRUCT('source_id', source_id, 'volume', volume, 'issue', issue) END) AS issue_keys,
    MIN(CASE WHEN provenance NOT IN ('crossref', 'datacite', 'pubmed', 'mag')
              AND NOT ARRAY_CONTAINS(ARRAY(DATE'1970-01-01', DATE'1969-12-31', DATE'1899-12-30', DATE'1900-01-01', DATE'1800-01-01', DATE'1500-01-01', DATE'1400-01-01', DATE'1000-01-01'), published_date)
              AND YEAR(published_date) BETWEEN 1901 AND YEAR(CURRENT_DATE())
             THEN YEAR(published_date) END) AS repo_min_year,
    COLLECT_SET(CASE WHEN is_placeholder THEN published_date END) AS placeholder_dates,
    COLLECT_SET(CASE WHEN NOT is_placeholder THEN published_date END) AS stated_dates,
    MIN_BY(CASE WHEN NOT is_placeholder AND provenance IN ('pubmed', 'datacite', 'mag')
                THEN NAMED_STRUCT('provenance', provenance, 'published_date', published_date) END,
           CASE WHEN NOT is_placeholder AND provenance IN ('pubmed', 'datacite', 'mag')
                THEN NAMED_STRUCT('p', priority, 'd', CAST(published_date AS STRING)) END) AS candidate,
    COLLECT_SET(CASE WHEN NOT is_placeholder THEN NAMED_STRUCT('provenance', provenance, 'y', YEAR(published_date)) END) AS stated_years
  FROM records
  GROUP BY work_id
),
placeholder_rows AS (
  SELECT work_id, picked_date AS placeholder_date,
    CASE WHEN candidate.provenance = 'pubmed'
           OR EXISTS(stated_years, x -> x.provenance <> candidate.provenance AND x.y = YEAR(candidate.published_date))
         THEN candidate.published_date
         -- an AFIPS paper with no other date takes its DOI's year (header above)
         WHEN SIZE(afips_years) = 1 AND INT(GET(afips_years, 0)) BETWEEN 1951 AND 1990
         THEN MAKE_DATE(INT(GET(afips_years, 0)), 1, 1) END AS new_date,
    doi_titles, issue_keys, repo_min_year
  FROM per_work
  WHERE ARRAY_CONTAINS(placeholder_dates, picked_date) AND NOT ARRAY_CONTAINS(stated_dates, picked_date)
),
-- step 3 (header above): the DOI landing pages of the works still left with no date
undated_dois AS (
  SELECT DISTINCT pr.work_id, x.doi, x.title_key
  FROM placeholder_rows pr LATERAL VIEW EXPLODE(pr.doi_titles) e AS x
  WHERE pr.new_date IS NULL
),
pages AS (
  SELECT LOWER(m.native_id) AS doi,
    -- the latest fetch; a fetch can hold two rows with the same fetched_at, one without the tags (step 4's run, 21 Nov:
    -- 21 of 67,714 DOIs), so the tie goes to the row with more citation tags, then file_key, the same every night
    MAX_BY(NAMED_STRUCT('tags', FILTER(m.meta_tags, x -> x RLIKE 'name="citation_(date|title)"'), 'fetched_at', m.fetched_at),
           NAMED_STRUCT('f', m.fetched_at, 'n', SIZE(FILTER(m.meta_tags, x -> x RLIKE 'name="citation_(date|title)"')),
                        'k', m.file_key)) AS page
  FROM openalex.landing_page.meta_tags m
  WHERE m.status = 'ok' AND LOWER(m.native_id) IN (SELECT doi FROM undated_dois)
  GROUP BY 1
),
page_years AS (
  SELECT u.work_id,
    ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(FILTER(
      TRANSFORM(FILTER(p.page.tags, x -> x RLIKE 'name="citation_date"'),
                x -> TRY_CAST(REGEXP_EXTRACT(x, 'content="[^"]*?([12][0-9]{3})', 1) AS INT)),
      y -> y IS NOT NULL)))) AS years,
    MIN(YEAR(p.page.fetched_at)) AS fetch_year,
    MAX(CASE WHEN EXISTS(
          TRANSFORM(FILTER(p.page.tags, x -> x RLIKE 'name="citation_title"'),
                    x -> REGEXP_REPLACE(REGEXP_REPLACE(LOWER(REGEXP_EXTRACT(x, 'content="([^"]*)"', 1)), '&[a-z0-9#]+;', ''),
                                        '[^\\p{L}\\p{N}]', '')),
          pt -> LENGTH(pt) >= 10 AND LENGTH(u.title_key) >= 10 AND (pt = u.title_key OR LEFT(pt, 30) = LEFT(u.title_key, 30)))
        THEN 1 ELSE 0 END) AS title_match
  FROM undated_dois u
  JOIN pages p ON p.doi = u.doi
  GROUP BY u.work_id
),
-- step 4 (header above): the issues of the works left with no date, and the years the issue's other works state
undated_issue_keys AS (
  SELECT DISTINCT pr.work_id, k.source_id, k.volume, k.issue
  FROM placeholder_rows pr LATERAL VIEW EXPLODE(pr.issue_keys) e AS k
  WHERE pr.new_date IS NULL
),
issue_years AS (
  SELECT k.source_id, k.volume, k.issue,
    COUNT(DISTINCT CASE WHEN l.provenance <> 'mag' THEN l.work_id END) AS n_not_mag,
    ARRAY_DISTINCT(COLLECT_LIST(YEAR(l.published_date))) AS years
  FROM openalex.works.locations_mapped_curated l
  JOIN (SELECT DISTINCT source_id, volume, issue FROM undated_issue_keys) k
    ON k.source_id = l.source_id AND k.volume = LOWER(TRIM(l.volume)) AND k.issue = LOWER(TRIM(l.issue))
  LEFT ANTI JOIN (SELECT work_id FROM placeholder_rows WHERE new_date IS NULL) u ON u.work_id = l.work_id
  WHERE l.published_date IS NOT NULL
    AND NOT ARRAY_CONTAINS(ARRAY(DATE'1970-01-01', DATE'1969-12-31', DATE'1899-12-30', DATE'1900-01-01', DATE'1800-01-01', DATE'1500-01-01', DATE'1400-01-01', DATE'1000-01-01'), l.published_date)
    AND YEAR(l.published_date) BETWEEN 1901 AND YEAR(CURRENT_DATE())
  GROUP BY k.source_id, k.volume, k.issue
),
issue_rule AS (
  SELECT u.work_id, MIN(GET(iy.years, 0)) AS y, COUNT(DISTINCT GET(iy.years, 0)) AS n_years
  FROM undated_issue_keys u
  JOIN issue_years iy ON iy.source_id = u.source_id AND iy.volume = u.volume AND iy.issue = u.issue
  WHERE SIZE(iy.years) = 1 AND iy.n_not_mag = 0
  GROUP BY u.work_id
)
SELECT pr.work_id, pr.placeholder_date,
  COALESCE(pr.new_date,
    CASE WHEN py.title_match = 1 AND SIZE(py.years) = 1
          AND GET(py.years, 0) BETWEEN 1901 AND py.fetch_year - 1 AND GET(py.years, 0) NOT IN (1969, 1970)
         THEN MAKE_DATE(GET(py.years, 0), 1, 1) END,
    CASE WHEN ir.n_years = 1 AND ir.y = pr.repo_min_year THEN MAKE_DATE(ir.y, 1, 1) END) AS new_date
FROM placeholder_rows pr
LEFT JOIN page_years py ON py.work_id = pr.work_id
LEFT JOIN issue_rule ir ON ir.work_id = pr.work_id

In [ ]:
-- oxjob #2383 (policy D16, RULINGS § 3 Work dates: MAG ranks last): a work whose date comes from its frozen Microsoft
-- Academic (MAG) record takes its repository records' year instead when every repository record of the work states that
-- same year and it is at least 6 years earlier than MAG's (W3162698795, 53 Denv. L.J. 553: its repository says 1976,
-- MAG 2021, the day it was put online). Only these: a blind check of each work's own pages (oxjob #2383 EXPLORE.md,
-- 300 + 150 works) found the repository year right on 89% here and MAG's on 3%, but MAG right where a repository year
-- is LATER (deposit, upload and scan days), where the repository records disagree, and too often where they are 1-5
-- years earlier (MAG right on 15-39%), so all of those keep MAG's year. A repository
-- record here is a repo location that is not a ResearchGate copy (#1874); a work with any Crossref, DataCite or PubMed
-- record is left out (those outrank both). The date taken is the earliest repository date (get_highest_priority_value's
-- order inside a provenance), never one more than 12 months ahead (#1865).
-- Built here as its own table (a new pick inside the main statement froze Works_Base, #1931); its columns are not named
-- publication_date (the main statement reads YEAR(publication_date) through a lateral alias, #1644). The main statement
-- swaps only where its own pick still equals mag_date, after #2241's and #1154's swaps.
-- Revert: delete this cell and the join and WHEN in the main statement (grep 2383), drop the table.
CREATE OR REPLACE TABLE openalex.works.mag_outranked_publication_dates
COMMENT 'oxjob #2383: per work dated by its MAG record whose repository records all state one year at least 6 years earlier: work_id, mag_date, repo_date. Rebuilt by CreateWorksBase each run.'
AS
WITH mag_works AS (
  SELECT DISTINCT work_id FROM openalex.works.locations_mapped_curated
  WHERE provenance = 'mag' AND published_date IS NOT NULL
),
records AS (
  SELECT l.work_id, l.provenance, l.published_date,
    COALESCE((l.native_id_namespace = 'doi' AND LOWER(l.native_id) LIKE '10.13140/%')
      OR LOWER(COALESCE(l.pdf_url, l.landing_page_url, '')) RLIKE '^https?://([^/]*[.])?researchgate[.]net([/:?#]|$)', FALSE) AS is_rg
  FROM openalex.works.locations_mapped_curated l
  JOIN mag_works m ON m.work_id = l.work_id
  WHERE l.published_date IS NOT NULL AND YEAR(l.published_date) < 10000
),
per_work AS (
  SELECT work_id,
    MIN(CASE WHEN provenance = 'mag' THEN published_date END) AS mag_date,
    MIN(CASE WHEN provenance = 'repo' AND NOT is_rg THEN published_date END) AS repo_date,
    COUNT(DISTINCT CASE WHEN provenance = 'repo' AND NOT is_rg THEN YEAR(published_date) END) AS repo_years,
    COUNT_IF(provenance IN ('crossref', 'datacite', 'pubmed')) AS n_outranking
  FROM records
  GROUP BY work_id
)
SELECT work_id, mag_date, repo_date
FROM per_work
WHERE n_outranking = 0 AND repo_years = 1
  AND YEAR(mag_date) - YEAR(repo_date) >= 6
  AND repo_date <= ADD_MONTHS(CURRENT_DATE(), 12)

In [ ]:
-- oxjob #2097 (follows the dedup rule of 2026-09-28, oxjob #1256: "a preprint and its published article are ONE work;
-- the published side survives", as cr_vor_priority already does for dates inside Crossref, #589): a work dated
-- by its preprint's DataCite record (typed preprint: arXiv 10.48550, some Zenodo) takes the date of the journal's own
-- published copy instead, when it holds one: a repository record from an endpoint the registry flags as the journal's
-- host (is_journal_host, the same gate as host_type 'publisher', #1409) on a journal source. TMLR (#2097's feed) and OJS
-- journals: W4403444212, arXiv 2024-10-10, TMLR March 2025. A work with any Crossref record is left out (Crossref
-- outranks both). The date taken is the journal copy's earliest, never one more than 12 months ahead (#1865).
-- Bar (19 Nov 2026, #2097 work/date_bar_sample60_2026-11-19*.tsv, random 60 non-TMLR works whose year changes): of the
-- 46 whose journal page states the issue year, the journal copy's year is right on 40 and the preprint's on 3 (the 6
-- misses are back issues whose OJS record carries the upload day); TMLR dates come from TMLR's own index.
-- Sized 19 Nov (inputs and openalex_works of night 11-19): 5,745 rows; 5,577 works change date, 3,759 change year
-- (3,522 visible, 2,433 TMLR-primary; 3,604 to a later year, 155 earlier).
-- Built here as its own table, as #2383's (a new pick inside the main statement froze Works_Base, #1931); its columns
-- are not named publication_date (#1644). The main statement swaps only where its own pick still equals preprint_date,
-- after #2241's, #1154's and #2383's swaps.
-- Revert: delete this cell and the join and WHEN in the main statement (grep 2097), drop the table.
CREATE OR REPLACE TABLE openalex.works.preprint_outranked_publication_dates
COMMENT 'oxjob #2097: per work dated by its DataCite preprint record that holds the journal''s own published copy (a journal-host repository record, no Crossref record): work_id, preprint_date, journal_date. Rebuilt by CreateWorksBase each run.'
AS
WITH journal_host_endpoints AS (
  SELECT DISTINCT endpoint_id FROM openalex.sources.endpoint_to_source WHERE is_journal_host
),
journal_copies AS (
  SELECT l.work_id, MIN(l.published_date) AS journal_date
  FROM openalex.works.locations_mapped_curated l
  JOIN journal_host_endpoints jh ON jh.endpoint_id = l.endpoint_id
  -- oxjob #2494: a conference series' own host endpoint counts as the journal's does (same gate as host_type below).
  JOIN openalex.sources.sources s ON s.id = l.source_id AND s.type IN ('journal', 'conference')
  WHERE l.provenance IN ('repo', 'repo_backfill') AND l.work_id IS NOT NULL
    AND l.published_date IS NOT NULL AND YEAR(l.published_date) BETWEEN 1000 AND 9999
    -- never a placeholder an OJS record carries for "no date" (#1154's list: one 1970-01-01 on 19 Nov)
    AND l.published_date NOT IN (DATE'1970-01-01', DATE'1969-12-31', DATE'1899-12-30', DATE'1900-01-01', DATE'1800-01-01',
                                 DATE'1500-01-01', DATE'1400-01-01', DATE'1000-01-01')
  GROUP BY l.work_id
),
preprints AS (
  SELECT l.work_id, MIN(l.published_date) AS preprint_date
  FROM openalex.works.locations_mapped_curated l
  WHERE l.provenance = 'datacite' AND l.type = 'preprint' AND l.work_id IS NOT NULL
    AND l.published_date IS NOT NULL AND YEAR(l.published_date) < 10000
  GROUP BY l.work_id
)
SELECT p.work_id, p.preprint_date, j.journal_date
FROM preprints p
JOIN journal_copies j ON j.work_id = p.work_id
LEFT ANTI JOIN (SELECT DISTINCT work_id FROM openalex.works.locations_mapped_curated
                WHERE provenance = 'crossref' AND work_id IS NOT NULL) cr
  ON cr.work_id = p.work_id
WHERE j.journal_date <> p.preprint_date
  AND j.journal_date <= ADD_MONTHS(CURRENT_DATE(), 12)

In [ ]:
-- oxjob #1154 (ZD 2171; policy D16, RULINGS § 3 Work dates: MAG ranks last): a work dated only by its frozen Microsoft
-- Academic (MAG) record takes 1 January of the year its RePEc records state, when that year is at least 6 years earlier.
-- RePEc's OAI records carry no date, only the year inside the handle (oai:RePEc:aea:aecrev:v:73:y:1983:i:4:...), so
-- #2383's repository-year rule never sees them: MAG dated ~640 classic American Economic Review papers 2016-01-01
-- (W10961258, AER vol. 73 no. 4, 1983). Only works with no dated record but MAG's whose RePEc records all state one
-- year. Guard: a work whose MAG volume differs from its handle's is left out (handles carry placeholders too: Cato
-- Journal files 1998 papers as v:1:y:1981:i:1, MAG volume 18). Bar (oxjob #1154 work/62_*, 63_*, 19 Nov 2026): on the
-- 1,115 candidates whose journal's Crossref records fix a volume-to-year offset, the handle's year is right (within 1)
-- on 1,105 and MAG's on 2; of 24 random works checked against their own pages, the handle is right on 19, 4 can't be
-- told, 1 wrong (the Cato case, which the guard drops). 1,384 works on 21 Nov 2026 (639 AER).
-- Built here as its own table, as #2383's (a new pick inside the main statement froze Works_Base, #1931); its columns
-- are not named publication_date (#1644). The main statement swaps only where its own pick still equals mag_date,
-- after #2241's, #1154's placeholder, #2383's and #2097's swaps.
-- Revert: delete this cell and the join and WHEN in the main statement (grep repec_year), drop the table.
CREATE OR REPLACE TABLE openalex.works.repec_year_publication_dates
COMMENT 'oxjob #1154: per work dated only by its MAG record whose RePEc records all state one year at least 6 years earlier (the year in the RePEc handle): work_id, mag_date, repec_date (1 January of that year). Rebuilt by CreateWorksBase each run.'
AS
WITH repec_records AS (
  SELECT work_id,
    TRY_CAST(NULLIF(REGEXP_EXTRACT(native_id, ':y:([0-9]{4})', 1), '') AS INT) AS handle_year,
    NULLIF(REGEXP_EXTRACT(native_id, ':v:([^:]*)', 1), '') AS handle_volume
  FROM openalex.works.locations_mapped_curated
  WHERE provenance = 'repo' AND native_id LIKE 'oai:RePEc:%' AND work_id IS NOT NULL
),
repec_works AS (
  SELECT work_id, MIN(handle_year) AS handle_year, COLLECT_SET(handle_volume) AS handle_volumes
  FROM repec_records
  GROUP BY work_id
  HAVING COUNT_IF(handle_year IS NULL) = 0 AND COUNT(DISTINCT handle_year) = 1
),
per_work AS (
  SELECT l.work_id,
    MIN(CASE WHEN l.provenance = 'mag' THEN l.published_date END) AS mag_date,
    MIN(CASE WHEN l.provenance = 'mag' THEN NULLIF(TRIM(l.volume), '') END) AS mag_volume,
    COUNT_IF(l.provenance <> 'mag' AND l.published_date IS NOT NULL) AS n_other_dated
  FROM openalex.works.locations_mapped_curated l
  JOIN repec_works r ON r.work_id = l.work_id
  GROUP BY l.work_id
)
SELECT p.work_id, p.mag_date, MAKE_DATE(r.handle_year, 1, 1) AS repec_date
FROM per_work p
JOIN repec_works r ON r.work_id = p.work_id
WHERE p.mag_date IS NOT NULL AND p.n_other_dated = 0
  AND r.handle_year >= 1000 AND YEAR(p.mag_date) - r.handle_year >= 6
  AND (p.mag_volume IS NULL OR FORALL(r.handle_volumes, v -> v = p.mag_volume))

In [ ]:
-- oxjob #1154 step 6 (follows oxjob #2241, 16 Nov 2026: a 1 January padded from a bare year yields to a date a record of
-- the work states that year; and RULINGS § 3 Work dates, D6: a Crossref work's date is the earliest date the publisher
-- states): works dated 1 January only because one of their Crossref records states a bare year, while another Crossref
-- record of the same work states a later date that year. Most are works held in two Crossref records: an alias or
-- old-prefix DOI (BioOne SICI 10.1658/..., APA 10.1037//..., Project MUSE 10.1353/..., an SSRN preprint 10.2139/...)
-- that deposits only the year, beside the publisher's DOI that states the month or day. W4293775574: its SSRN
-- preprint 10.2139/ssrn.4117533 deposits [2022], its article DOI 10.1016/j.apcata.2022.118840 is dated 2022-08-27,
-- served 2022-01-01: same-rank records tie on the earliest date, so the padded day won. One row per such work: the padded
-- date and the earliest later date that year one of the work's other Crossref records states. The main statement takes
-- it only where its pick is still that padded 1 January. No year changes (same year by construction). Padded = as
-- #2241: the record's earliest date is a bare year and nothing in it (issued, published, online, print, approved,
-- posted) states a finer date that year, nor is that day the DOI's created or deposited day. Left alone: a work with a
-- DataCite record (its padding is not read here), two padded dates, another Crossref record stating that 1 January or
-- an earlier day, or a later date of 31 December (some deposits pad a year to its last day).
-- Bar (21 Nov 2026, dry run 341,397 works, 341,256 visible): of the works whose own DOI landing page (meta_tags) states
-- a full date that year, the new date is the page's day on 73% and its month on 82%; the padded 1 January matched 7%.
-- Built here as its own table (a new pick inside the main statement froze Works_Base, #1931). Revert: delete this cell
-- and the join and WHEN in the main statement (grep padded_year_doi_dates), drop the table.
CREATE OR REPLACE TABLE openalex.works.padded_year_doi_dates
COMMENT 'oxjob #1154 step 6: per work dated 1 January only because one of its Crossref records states a bare year while another of its Crossref records states a later date that year: work_id, padded_date, doi_date. Rebuilt by CreateWorksBase each run.'
AS
WITH cr_dates AS (
  SELECT
    LOWER(native_id) AS doi,
    ARRAY(
      NAMED_STRUCT('d', TRY_TO_DATE(FORMAT_STRING('%04d-%02d-%02d', GET(GET(issued.`date-parts`, 0), 0), COALESCE(GET(GET(issued.`date-parts`, 0), 1), 1), COALESCE(GET(GET(issued.`date-parts`, 0), 2), 1))), 'p', SIZE(GET(issued.`date-parts`, 0))),
      NAMED_STRUCT('d', TRY_TO_DATE(FORMAT_STRING('%04d-%02d-%02d', GET(GET(published.`date-parts`, 0), 0), COALESCE(GET(GET(published.`date-parts`, 0), 1), 1), COALESCE(GET(GET(published.`date-parts`, 0), 2), 1))), 'p', SIZE(GET(published.`date-parts`, 0))),
      NAMED_STRUCT('d', TRY_TO_DATE(FORMAT_STRING('%04d-%02d-%02d', GET(GET(`published-online`.`date-parts`, 0), 0), COALESCE(GET(GET(`published-online`.`date-parts`, 0), 1), 1), COALESCE(GET(GET(`published-online`.`date-parts`, 0), 2), 1))), 'p', SIZE(GET(`published-online`.`date-parts`, 0))),
      NAMED_STRUCT('d', TRY_TO_DATE(FORMAT_STRING('%04d-%02d-%02d', GET(GET(`published-print`.`date-parts`, 0), 0), COALESCE(GET(GET(`published-print`.`date-parts`, 0), 1), 1), COALESCE(GET(GET(`published-print`.`date-parts`, 0), 2), 1))), 'p', SIZE(GET(`published-print`.`date-parts`, 0))),
      NAMED_STRUCT('d', TRY_TO_DATE(FORMAT_STRING('%04d-%02d-%02d', GET(GET(approved.`date-parts`, 0), 0), COALESCE(GET(GET(approved.`date-parts`, 0), 1), 1), COALESCE(GET(GET(approved.`date-parts`, 0), 2), 1))), 'p', SIZE(GET(approved.`date-parts`, 0)))
      ,NAMED_STRUCT('d', TRY_TO_DATE(FORMAT_STRING('%04d-%02d-%02d', GET(GET(posted.`date-parts`, 0), 0), COALESCE(GET(GET(posted.`date-parts`, 0), 1), 1), COALESCE(GET(GET(posted.`date-parts`, 0), 2), 1))), 'p', SIZE(GET(posted.`date-parts`, 0)))
    ) AS pd,
    TO_DATE(SUBSTR(created.`date-time`, 1, 10)) AS cr_created,
    TO_DATE(SUBSTR(deposited.`date-time`, 1, 10)) AS cr_deposited
  FROM openalex.crossref.crossref_deduplicated
),

-- Crossref records whose earliest bare year is padded to 1 January and stated by nothing finer that year (as #2241)
cr_year_only AS (
  SELECT doi, y1 AS padded_date
  FROM (SELECT doi, pd, cr_created, cr_deposited,
               ARRAY_MIN(TRANSFORM(FILTER(pd, x -> x.d IS NOT NULL AND x.p = 1), x -> x.d)) AS y1
        FROM cr_dates)
  WHERE y1 IS NOT NULL
    AND NOT EXISTS(pd, x -> x.d IS NOT NULL AND x.p >= 2 AND YEAR(x.d) = YEAR(y1))
    AND COALESCE(cr_created <> y1, TRUE) AND COALESCE(cr_deposited <> y1, TRUE)
),
-- the work's Crossref locations still dated that padded 1 January
padded AS (
  SELECT l.work_id, LOWER(l.native_id) AS doi, l.published_date AS padded_date
  FROM openalex.works.locations_mapped_curated l
  JOIN cr_year_only y ON y.doi = LOWER(l.native_id) AND y.padded_date = l.published_date
  WHERE l.provenance = 'crossref' AND MONTH(l.published_date) = 1 AND DAY(l.published_date) = 1
),
recs AS (
  SELECT l.work_id, l.provenance, LOWER(l.native_id) AS doi, l.published_date
  FROM openalex.works.locations_mapped_curated l
  WHERE l.published_date IS NOT NULL AND l.work_id IN (SELECT work_id FROM padded)
),
per_work AS (
  SELECT r.work_id, MIN(p.padded_date) AS padded_date, COUNT(DISTINCT p.padded_date) AS n_padded,
    -- the dates the work's other (not padded) Crossref records state
    COLLECT_SET(CASE WHEN r.provenance = 'crossref' AND p2.doi IS NULL THEN r.published_date END) AS cr_dates,
    MAX(CASE WHEN r.provenance = 'datacite' THEN 1 ELSE 0 END) AS has_datacite
  FROM recs r
  JOIN padded p ON p.work_id = r.work_id
  LEFT JOIN padded p2 ON p2.work_id = r.work_id AND p2.doi = r.doi
  GROUP BY r.work_id
)
SELECT work_id, padded_date, doi_date
FROM (SELECT work_id, padded_date, n_padded, has_datacite, cr_dates,
             ARRAY_MIN(FILTER(cr_dates, d -> YEAR(d) = YEAR(padded_date) AND d > padded_date)) AS doi_date
      FROM per_work)
WHERE n_padded = 1 AND has_datacite = 0 AND doi_date IS NOT NULL
  AND NOT EXISTS(cr_dates, d -> d <= padded_date)
  AND NOT (MONTH(doi_date) = 12 AND DAY(doi_date) = 31)

In [ ]:
-- oxjob #2264 (Jason, 15 Nov 2026, plans/retractions.md § 5, decisions 1-3): is_pulled is true when the publisher, host
-- or authors retracted, withdrew or removed the work. This table holds the Crossref half that is_retracted does not carry:
-- (1) DOIs whose own Crossref record declares a withdrawal or removal (updated-by, or an update-to naming the record
-- itself, as Elsevier stamps its "WITHDRAWN:" and "REMOVED:" articles), and (2) DOIs of notices about ANOTHER record
-- (withdrawal, removal or retraction), so a notice titled "WITHDRAWN: <paper>" is never pulled itself (Kyle 2026-10-02,
-- the #1500 rule for retraction notices). Cochrane's withdrawals (10.1002/14651858) stay out: a review "withdrawn as out
-- of date" is not pulled (decision 1). ~59K rows on 16 Nov 2026 (3.1K pulled, 55.8K notices), so the main statement
-- broadcasts it. Revert: delete this cell, and the is_pulled column, the cps join and its hint in the main statement
-- (grep 2264); drop the table.
CREATE OR REPLACE TABLE openalex.works.crossref_pull_signals
COMMENT 'oxjob #2264: cleaned lowercase DOI -> is_pulled_record (its own Crossref record declares a withdrawal or removal; Cochrane left out), is_pull_notice (it is a withdrawal, removal or retraction notice about another record). Rebuilt by CreateWorksBase each run.'
AS
SELECT doi, BOOL_OR(pulled) AS is_pulled_record, BOOL_OR(notice) AS is_pull_notice
FROM (
  SELECT REGEXP_REPLACE(LOWER(native_id), '[^a-zA-Z0-9./-]', '') AS doi,
    (COALESCE(EXISTS(`updated-by`, x -> LOWER(x.type) IN ('withdrawal', 'removal')), FALSE)
      OR COALESCE(EXISTS(`update-to`, x -> LOWER(x.type) IN ('withdrawal', 'removal') AND LOWER(x.DOI) = LOWER(native_id)), FALSE))
      AND LOWER(native_id) NOT LIKE '10.1002/14651858%' AS pulled,
    COALESCE(EXISTS(`update-to`, x -> LOWER(x.type) IN ('withdrawal', 'removal', 'retraction')
                                      AND LOWER(x.DOI) <> LOWER(native_id)), FALSE) AS notice
  FROM openalex.crossref.crossref_exploded
  WHERE EXISTS(`updated-by`, x -> LOWER(x.type) IN ('withdrawal', 'removal'))
     OR EXISTS(`update-to`, x -> LOWER(x.type) IN ('withdrawal', 'removal', 'retraction'))
)
GROUP BY doi
HAVING BOOL_OR(pulled) OR BOOL_OR(notice);

In [ ]:
-- oxjob #2264 (17 Nov 2026): retractions that do NOT make a work pulled (plans/retractions.md § 3, and § 5 decision 1).
-- Retraction Watch labels every retract-and-replace and every Cochrane withdrawal a "Retraction", so is_retracted carries
-- them, but the pulled rule leaves out: a retract-and-replace (the corrected version stands, often under the same DOI, as
-- at JAMA and The Lancet); a work withdrawn only as out of date (Cochrane reviews, practice guidelines); and a Cochrane
-- withdrawal whose reasons don't question the review (Kyle 2026-10-02: Cochrane withdrawals are not retractions). A
-- Cochrane withdrawal Retraction Watch gives a problem reason for (unreliable data, misconduct, an investigation) stays
-- pulled. One row per DOI; leaves_out only when EVERY Retraction Watch retraction of it is one of those, so a later full
-- retraction still pulls it. The main statement reads a Cochrane DOI with no row here as left out too. Sizes on
-- 17 Nov 2026: 502 Cochrane reviews and 257 other works leave pulled (85 Cochrane reviews stay); a work whose own title
-- says "Retracted:" stays pulled. Revert: delete this cell, and the rwx join, its hint and its term in is_pulled in the
-- main statement (grep rwx); drop the table.
CREATE OR REPLACE TABLE openalex.works.retraction_pull_exemptions
COMMENT 'oxjob #2264: cleaned lowercase DOI -> leaves_out (is_retracted does not make this work pulled: a retract-and-replace, a withdrawal as out of date, or a Cochrane withdrawal with no problem reason, per Retraction Watch) and why. Rebuilt by CreateWorksBase each run.'
AS
WITH records AS (
  SELECT REGEXP_REPLACE(LOWER(original_paper_doi), '[^a-zA-Z0-9./-]', '') AS doi,
    TRANSFORM(SPLIT(COALESCE(reason, ''), ';'), x -> REGEXP_REPLACE(TRIM(x), '^[+]', '')) AS reasons
  FROM openalex.works.retraction_watch
  WHERE original_paper_doi IS NOT NULL AND retraction_nature = 'Retraction'
),
flagged AS (
  SELECT doi,
    ARRAY_CONTAINS(reasons, 'Retract and Replace') AS replaced,
    ARRAY_CONTAINS(reasons, 'Withdrawn as Out of Date') AS out_of_date,
    -- a reason that questions the work; the others are Retraction Watch's bookkeeping
    EXISTS(reasons, x -> x <> '' AND x NOT LIKE 'Notice - %' AND x NOT IN ('Retract and Replace', 'Withdrawn as Out of Date',
      'Removed', 'Temporary Removal', 'No Further Action', 'Date of Article and/or Notice Unknown',
      'Upgrade/Update of Prior Notice(s)')) AS has_problem_reason,
    doi LIKE '10.1002/14651858%' AS is_cochrane
  FROM records
)
SELECT doi,
  BOOL_AND(replaced OR (out_of_date AND NOT has_problem_reason) OR (is_cochrane AND NOT has_problem_reason)) AS leaves_out,
  CASE
    WHEN BOOL_AND(replaced) THEN 'retract and replace'
    WHEN BOOL_AND(replaced OR (out_of_date AND NOT has_problem_reason)) THEN 'out of date'
    WHEN BOOL_AND(replaced OR (out_of_date AND NOT has_problem_reason) OR (is_cochrane AND NOT has_problem_reason))
      THEN 'Cochrane withdrawal, no problem reason'
    ELSE 'pulled: a retraction with a problem reason'
  END AS why
FROM flagged
GROUP BY doi
HAVING BOOL_OR(replaced OR out_of_date OR is_cochrane);

In [ ]:
-- oxjob #1594: the old system's copy of an initials-keyed affiliation block. CreateCrossrefSuperAuthorships pairs a block
-- such as "From the Department of Neurology (S.M., D.R.), All India Institute of Medical Sciences ...; Brain Health Unit
-- (K.K., T.D.), World Health Organization ..." with the authors its initials name (openalex.works.crossref_initials_keyed_split,
-- ~104K DOIs). But works_legacy.raw_affiliation_strings holds the same block, or its pieces, on every seat of ~94K of
-- those works, and the main statement below adds legacy strings to every seat, so every author would still show every
-- institution of the paper (ZD 3464: W3214050275, W4385854855, W2914576615).
-- For each work whose Crossref block was paired, this table holds the legacy rows to read instead of the stored ones:
--   1. the same function pairs the legacy strings, with the paired Crossref record's names (seat i = legacy
--      author_sequence i, as the main statement joins them), so a legacy copy whose initials resolve is paired the same way
--      (an untruncated legacy copy also pairs the authors named after Crossref's '...' cut)
--   2. a legacy string still on 2+ seats whose text is part of the paired Crossref block, or starts with it (a longer
--      copy), is dropped: it is the deposited block, and each seat it names already holds its own segments
--   3. every other legacy string (a seat's own) stays as stored.
-- An author the block does not name gets no institution from it, as in CreateCrossrefSuperAuthorships: none rather than
-- every institution of the paper. Shadow 16 Nov: 101,426 works, their legacy rows 2.4M -> 1,018,602, 33 s on a 4X-Large.
-- Output strings are decoded and repaired here, with the main statement's own gates, because the main statement reads
-- this table after its legacy_affiliations_repaired step (a SQL warehouse allows 5 Python functions per statement).
-- Revert: delete this cell and the subquery in the main statement's legacy_affiliations (git revert the commit).
CREATE OR REPLACE TABLE openalex.works.legacy_initials_keyed_split AS
WITH split AS (
  SELECT doi, authorships,
         TRANSFORM(blocks, x -> LOWER(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(x, '&(amp\\x3B)+', '&'), '(\\.\\.\\.|…)\\s*$', ''), '[^\\p{L}\\p{N}]', ''))) AS nblocks
  FROM openalex.works.crossref_initials_keyed_split
  WHERE authorships IS NOT NULL
),
-- the works holding a paired Crossref record (a work with two: the first DOI)
split_works AS (
  SELECT /*+ BROADCAST(s) */ l.work_id, MIN(s.doi) AS doi
  FROM openalex.works.locations_mapped_curated l
  JOIN split s ON l.provenance = 'crossref' AND l.native_id = s.doi
  GROUP BY l.work_id
),
legacy_seats AS (
  SELECT /*+ BROADCAST(w) */ r.work_id, r.author_sequence, ARRAY_SORT(COLLECT_SET(r.raw_affiliation_string)) AS strings
  FROM openalex.works_legacy.raw_affiliation_strings r
  JOIN split_works w ON r.work_id = w.work_id
  WHERE r.raw_affiliation_string IS NOT NULL
  GROUP BY r.work_id, r.author_sequence
),
byline AS (
  SELECT w.work_id, x.pos, x.a
  FROM split_works w
  JOIN split s ON s.doi = w.doi
  LATERAL VIEW POSEXPLODE(s.authorships) x AS pos, a
),
-- 1. the legacy strings in the function's input shape: the Crossref names, each seat's legacy strings as its affiliations
paired AS (
  SELECT b.work_id,
    openalex.works.pair_initials_keyed_authorships(TRANSFORM(ARRAY_SORT(COLLECT_LIST(STRUCT(b.pos AS pos, NAMED_STRUCT(
        'given', b.a.given, 'family', b.a.family, 'name', b.a.name, 'orcid', b.a.orcid,
        'affiliations', TRANSFORM(COALESCE(g.strings, ARRAY()), x -> NAMED_STRUCT('name', x, 'department', CAST(NULL AS STRING), 'ror_id', CAST(NULL AS STRING))),
        'is_corresponding', b.a.is_corresponding, 'author_key', b.a.author_key) AS au))), y -> y.au)) AS authorships
  FROM byline b
  LEFT JOIN legacy_seats g ON g.work_id = b.work_id AND g.author_sequence = b.pos
  GROUP BY b.work_id
),
seat_strings AS (
  SELECT g.work_id, g.author_sequence,
         CASE WHEN p.authorships IS NOT NULL AND g.author_sequence < SIZE(p.authorships)
              THEN TRANSFORM(TRY_ELEMENT_AT(p.authorships, g.author_sequence + 1).affiliations, x -> x.name)
              ELSE g.strings END AS strings
  FROM legacy_seats g LEFT JOIN paired p USING (work_id)
),
counted AS (
  SELECT work_id, author_sequence, s,
         COUNT(*) OVER (PARTITION BY work_id, s) AS seats_with_s,
         LOWER(REGEXP_REPLACE(REGEXP_REPLACE(s, '&(amp\\x3B)+', '&'), '[^\\p{L}\\p{N}]', '')) AS ns
  FROM seat_strings LATERAL VIEW EXPLODE(strings) t AS s
),
-- 2. drop a copy of the paired block still on 2+ seats, 3. keep the rest
kept AS (
  SELECT c.work_id, c.author_sequence, c.s
  FROM counted c JOIN split_works w ON w.work_id = c.work_id JOIN split k ON k.doi = w.doi
  WHERE NOT (c.seats_with_s >= 2 AND EXISTS(k.nblocks, nb -> LENGTH(c.ns) > 0 AND (INSTR(nb, c.ns) > 0 OR STARTSWITH(c.ns, nb))))
),
-- decoded and repaired exactly as the main statement treats legacy_affiliations_decoded / _repaired (same gates)
decoded AS (
  SELECT work_id, author_sequence,
         CASE WHEN s RLIKE '&(#[0-9]{1,7}|#[xX][0-9a-fA-F]{1,6}|[A-Za-z][A-Za-z0-9]{1,31})\\x3B' THEN openalex.common.decode_affiliation_entities(s) ELSE s END AS s
  FROM kept
),
repaired AS (
  SELECT work_id, author_sequence,
         CASE WHEN s RLIKE '[\\u00C2-\\u00DF\\u00E0-\\u00EF][\\u0080-\\u00BF\\u20AC\\u201A\\u0192\\u201E\\u2026\\u2020\\u2021\\u02C6\\u2030\\u0160\\u2039\\u0152\\u017D\\u2018\\u2019\\u201C\\u201D\\u2022\\u2013\\u2014\\u02DC\\u2122\\u0161\\u203A\\u0153\\u017E\\u0178]|[\\u0102\\u0139]' THEN openalex.common.repair_mojibake(s) ELSE s END AS s
  FROM decoded WHERE s IS NOT NULL
)
SELECT w.work_id,
       COALESCE(SORT_ARRAY(COLLECT_LIST(CASE WHEN r.s IS NOT NULL
               THEN NAMED_STRUCT('author_sequence', r.author_sequence, 'raw_affiliation_string', r.s) END)), ARRAY()) AS legacy
FROM split_works w
LEFT JOIN repaired r ON r.work_id = w.work_id
GROUP BY w.work_id

In [ ]:
-- oxjob #1098: Crossref article numbers (article-number with no page: Wiley e70493, APS 094517, OUP giab008), one per work,
-- only from DOI prefixes whose numbers PubMed prints as the page on >= 80% of >= 50 works (some publishers deposit an
-- internal id or the article's place in the issue: Inderscience 130208 on page 141), never a number that repeats the DOI.
-- The main statement shows it as biblio.first_page when no record gives the work a page and no curation sets it.
-- Revert: git revert the commit, then DROP TABLE openalex.works.crossref_article_numbers.
CREATE OR REPLACE TABLE openalex.works.crossref_article_numbers
COMMENT 'oxjob #1098: per work, the article number its Crossref record deposits with no page (published version first, then the lowest), from DOI prefixes whose numbers PubMed prints as the page. CreateWorksBase serves it as biblio.first_page when no record gives a page. Rebuilt by CreateWorksBase each run.'
AS
WITH numbers AS (
  SELECT l.work_id, l.version, c.native_id, SUBSTRING_INDEX(c.native_id, '/', 1) AS prefix,
    TRIM(c.`article-number`) AS article_number, NULLIF(TRIM(c.page), '') AS page
  FROM openalex.crossref.crossref_deduplicated c
  JOIN openalex.works.locations_mapped_curated l ON l.provenance = 'crossref' AND l.native_id = c.native_id
  WHERE NULLIF(TRIM(c.`article-number`), '') IS NOT NULL AND l.work_id IS NOT NULL
),
pubmed_vouched AS (
  SELECT n.prefix
  FROM numbers n
  JOIN openalex.works.locations_mapped_curated p
    ON p.work_id = n.work_id AND p.provenance = 'pubmed' AND NULLIF(TRIM(p.first_page), '') IS NOT NULL
  GROUP BY n.prefix
  HAVING COUNT(*) >= 50
    AND AVG(IF(REGEXP_REPLACE(REGEXP_REPLACE(LOWER(p.first_page), '[^a-z0-9]', ''), '^[a-z]*0*', '')
             = REGEXP_REPLACE(REGEXP_REPLACE(LOWER(n.article_number), '[^a-z0-9]', ''), '^[a-z]*0*', ''), 1, 0)) >= 0.8
)
SELECT n.work_id,
  MIN_BY(n.article_number, STRUCT(n.version IS DISTINCT FROM 'publishedVersion', n.article_number)) AS article_number
FROM numbers n
JOIN pubmed_vouched v ON v.prefix = n.prefix
LEFT ANTI JOIN (
  SELECT work_id FROM openalex.works.work_curated_fields WHERE ARRAY_CONTAINS(curated_fields, 'biblio.first_page')
) wc ON wc.work_id = n.work_id
WHERE n.page IS NULL
  AND REGEXP_REPLACE(LOWER(n.article_number), '[^a-z0-9]', '') NOT IN (
    REGEXP_REPLACE(LOWER(n.native_id), '[^a-z0-9]', ''),
    REGEXP_REPLACE(LOWER(SUBSTR(n.native_id, INSTR(n.native_id, '/') + 1)), '[^a-z0-9]', ''))
GROUP BY n.work_id

In [ ]:
-- oxjob #1098 step 2c: the volume and issue a repository record's own landing page states in its citation_volume and
-- citation_issue meta tags (the Highwire tags OJS, DSpace, Pure, HAL, DOAJ and most journal sites print), one row per
-- work. Repository records (OAI dc) have no volume or issue field, so about 4M works known from them showed none.
-- Read from openalex.landing_page.meta_tags (every stored page's raw <meta> lines; Meta Tags Daily adds new fetches),
-- the newest page per record. A volume is 1-4 digits and not a year (19xx/20xx, a stand-in for journals without
-- volumes); an issue is 1-3 digits or a combined issue (3-4, 1/2); leading zeros dropped. A work takes the volume only
-- when all its pages agree on it, and the issue only when they all state the same one. Works with a curated volume or
-- issue are left out (a curation, even a removal, wins). Beside a volume a record already gives, the page's agreed on
-- 98.95% of 21.8M works, its issue on 98.6% (22 Nov 2026). The main statement uses it fill-only (biblio.volume and
-- biblio.issue), below every record, curation and the repository citation fill.
-- Shared record ids (23 Nov 2026): meta_tags is keyed by the OAI id alone, and OJS sites left on a default repository
-- id share theirs (oai:ojs.pkp.sfu.ca:article/1057, oai::article/1057), so pages of other journals' articles can sit
-- under a record's id. When an id is such a default one, or its pages carry more than one article title, a page is used
-- only if the URL it states for itself (citation_abstract_html_url, citation_pdf_url, og:url, ...) is on one of the
-- record's own hosts, or its citation_title is the record's title (same letters and digits, one a prefix of the other,
-- or within one edit in five).
-- Revert: git revert the commit, then DROP TABLE openalex.works.repo_page_volumes.
CREATE OR REPLACE TABLE openalex.works.repo_page_volumes
COMMENT 'oxjob #1098 step 2c: per work, the volume (and issue) the landing pages of its repository records state in citation_volume and citation_issue meta tags (newest page per record, all pages agreeing; under a shared record id only pages on a host of the record or with its title). CreateWorksBase serves it fill-only as biblio.volume and biblio.issue. Rebuilt by CreateWorksBase each run.'
AS
WITH pages AS (
  SELECT native_id AS pmh_id, fetched_at,
    TRANSFORM(FILTER(meta_tags, x -> LOWER(x) RLIKE 'citation_(volume|issue|title|abstract_html_url|fulltext_html_url|pdf_url)|og:url'),
      x -> NAMED_STRUCT(
        'n', LOWER(TRIM(REGEXP_EXTRACT(x, '(?i)\\b(?:name|property)\\s*=\\s*[\\x22\\x27]([^\\x22\\x27]*)[\\x22\\x27]', 1))),
        'c', TRIM(REGEXP_REPLACE(REGEXP_EXTRACT(x, '(?i)\\bcontent\\s*=\\s*[\\x22\\x27]([^\\x22\\x27]*)[\\x22\\x27]', 1), '\\s+', ' ')))) AS tags
  FROM openalex.landing_page.meta_tags
  WHERE native_id_namespace = 'pmh' AND status = 'ok' AND EXISTS(meta_tags, x -> LOWER(x) LIKE '%citation_volume%')
),
parsed AS (
  SELECT pmh_id, fetched_at,
    GET(FILTER(tags, t -> t.n = 'citation_volume' AND t.c <> ''), 0).c AS vol,
    GET(FILTER(tags, t -> t.n = 'citation_issue' AND t.c <> ''), 0).c AS iss,
    REGEXP_REPLACE(LOWER(REGEXP_REPLACE(GET(FILTER(tags, t -> t.n = 'citation_title' AND t.c <> ''), 0).c,
      '&#?[a-zA-Z0-9]+;', '')), '[^\\p{L}\\p{N}]', '') AS title_key,
    TRANSFORM(FILTER(tags, t -> t.n IN ('citation_abstract_html_url', 'citation_fulltext_html_url', 'citation_pdf_url', 'og:url')),
      t -> LOWER(REGEXP_EXTRACT(t.c, '^https?://(?:www\\.)?([^/:?#]+)', 1))) AS page_hosts
  FROM pages
),
ids AS (
  SELECT pmh_id,
    pmh_id RLIKE '^oai:(ojs\\.pkp\\.sfu\\.ca|ojs\\.localhost|localhost)?:' OR COUNT(DISTINCT NULLIF(LEFT(title_key, 40), '')) > 1 AS shared
  FROM parsed
  GROUP BY pmh_id
),
cand AS (
  SELECT p.pmh_id, p.fetched_at, p.vol, p.iss, p.title_key, p.page_hosts, i.shared,
    ROW_NUMBER() OVER (PARTITION BY p.pmh_id ORDER BY p.fetched_at DESC) AS rn
  FROM parsed p JOIN ids i ON i.pmh_id = p.pmh_id
),
joined AS (
  SELECT l.work_id, c.pmh_id, c.fetched_at, c.vol, c.iss, c.shared, c.title_key, c.page_hosts,
    IF(c.shared, REGEXP_REPLACE(LOWER(REGEXP_REPLACE(l.title, '&#?[a-zA-Z0-9]+;', '')), '[^\\p{L}\\p{N}]', ''), NULL) AS rec_title_key,
    IF(c.shared, TRANSFORM(CONCAT(COALESCE(TRANSFORM(l.urls, u -> u.url), ARRAY()), ARRAY(COALESCE(l.landing_page_url, ''))),
      u -> LOWER(REGEXP_EXTRACT(u, '^https?://(?:www\\.)?([^/:?#]+)', 1))), NULL) AS rec_hosts
  FROM cand c
  JOIN openalex.works.locations_mapped_curated l ON l.provenance IN ('repo', 'repo_backfill') AND l.native_id = c.pmh_id
  WHERE l.work_id IS NOT NULL AND (c.rn = 1 OR c.shared)
),
own AS (
  SELECT work_id, vol, iss, ROW_NUMBER() OVER (PARTITION BY work_id, pmh_id ORDER BY fetched_at DESC) AS rn
  FROM joined
  WHERE NOT shared
    OR ARRAYS_OVERLAP(FILTER(page_hosts, h -> h <> ''), FILTER(rec_hosts, h -> h <> ''))
    OR (title_key <> '' AND rec_title_key <> '' AND (
      title_key = rec_title_key
      OR (LEAST(LENGTH(title_key), LENGTH(rec_title_key)) >= 15
        AND (STARTSWITH(title_key, rec_title_key) OR STARTSWITH(rec_title_key, title_key)))
      OR LEVENSHTEIN(LEFT(title_key, 60), LEFT(rec_title_key, 60))
        <= GREATEST(2, LEAST(LENGTH(title_key), LENGTH(rec_title_key), 60) / 5)))
),
records AS (
  SELECT work_id,
    IF(REGEXP_EXTRACT(vol, '^(0*[1-9][0-9]{0,3})', 1) = vol, REGEXP_REPLACE(vol, '^0+', ''), NULL) AS vol,
    IF(REGEXP_EXTRACT(iss, '^(0*[1-9][0-9]{0,2}(?:[-/]0*[1-9][0-9]{0,2})?)', 1) = iss,
      REGEXP_REPLACE(iss, '(?<![0-9])0+(?=[1-9])', ''), NULL) AS iss
  FROM own
  WHERE rn = 1
)
SELECT r.work_id, GET(COLLECT_SET(r.vol), 0) AS volume,
  IF(SIZE(COLLECT_SET(r.iss)) = 1 AND COUNT(r.iss) = COUNT(*), GET(COLLECT_SET(r.iss), 0), NULL) AS issue
FROM records r
LEFT ANTI JOIN (
  SELECT work_id FROM openalex.works.work_curated_fields
  WHERE ARRAY_CONTAINS(curated_fields, 'biblio.volume') OR ARRAY_CONTAINS(curated_fields, 'biblio.issue')
) wc ON wc.work_id = r.work_id
WHERE r.vol IS NOT NULL AND NOT (LENGTH(r.vol) = 4 AND r.vol RLIKE '^(19|20)')
GROUP BY r.work_id
HAVING SIZE(COLLECT_SET(r.vol)) = 1

In [ ]:
-- Register helper function
CREATE OR REPLACE FUNCTION get_highest_priority_value(
    all_structs ARRAY<STRUCT<field_value: STRING, priority: INT>>, field_name STRING
  )
  RETURNS STRING
  RETURN
    (
      SELECT
        AGGREGATE(
          FILTER(all_structs, y -> y.field_value IS NOT NULL),
          STRUCT(CAST(NULL AS STRING) AS field_value, 999 AS priority),
          (acc, x) -> CASE
            WHEN x.priority < acc.priority THEN x
            WHEN x.priority = acc.priority AND x.field_value < COALESCE(acc.field_value, '') THEN x
            ELSE acc
          END
        ).field_value
    );

-- Final pipeline to create openalex_works
CREATE
OR REPLACE TABLE openalex.works.openalex_works_base CLUSTER BY (id, updated_date)
TBLPROPERTIES (
  'delta.dataSkippingNumIndexedCols' = 36,
  'delta.deletedFileRetentionDuration' = '60 days',
  -- default is 7
  'delta.logRetentionDuration' = '60 days' -- default is 30
) AS (
  WITH mat_sources AS (
    SELECT /*+ BROADCAST(pl) */
      s.id AS source_id,
      -- oxjob #2106: the source name works embed, normalised as sources_api serves it (registry name untouched)
      CASE WHEN s.display_name RLIKE '&[a-zA-Z][a-zA-Z0-9]{1,30}\\x3b|&#[xX]?[0-9a-fA-F]{2,7}\\x3b|[\\x{0300}-\\x{036F}\\x{00AD}\\x{200B}\\x{200E}\\x{200F}\\x{FEFF}\\x{2060}]'
        THEN openalex.common.normalize_name_text(s.display_name, TRUE) ELSE s.display_name END AS display_name,
      s.issn_l,
      s.issns,
      s.is_in_doaj,
      s.is_core,
      s.listed_in,
      s.publisher AS source_publisher,
      s.publisher_id,
      s.institution_id,
      s.apc_prices AS apc_prices,
      s.apc_usd,
      s.type AS source_type,
      i.display_name AS institution_name,
      p.display_name AS publisher_name,
      p.parent_publisher,
      pl.ancestor_urls AS publisher_ancestor_urls,
      pl.ancestor_names AS publisher_ancestor_names,
      s.is_in_doaj_start_year,
      s.high_oa_rate_start_year,
      s.is_oa_high_oa_rate,
      s.is_fully_open_in_jstage,
      s.doaj_license
    FROM
      openalex.sources.sources s
      LEFT JOIN openalex.institutions.institutions i ON s.institution_id = i.id
      LEFT JOIN openalex.publishers.publishers p ON s.publisher_id = p.id
      -- oxjob #1813: every ancestor of the source's publisher, nearest first, from the table /publishers and /sources
      -- build their lineage from (publisher_ancestors, rebuilt by Sync Publishers an hour before this run). Works used
      -- to carry the publisher and its parent only, so 3.9M works on a publisher two or more levels down (Routledge ->
      -- Taylor & Francis -> Informa) missed the top group and lineage filters undercounted. A few thousand rows.
      LEFT JOIN (
        SELECT
          x.publisher_id,
          FIRST(x.ancestor_urls) AS ancestor_urls,
          TRANSFORM(ARRAY_SORT(COLLECT_LIST(STRUCT(x.pos AS pos, ap.display_name AS name))), y -> y.name) AS ancestor_names
        FROM (
          SELECT publisher_id, ancestor_urls, pos, url
          FROM openalex.publishers.publisher_ancestors LATERAL VIEW POSEXPLODE(ancestor_urls) e AS pos, url
        ) x
        LEFT JOIN openalex.publishers.publishers ap ON ap.id = CAST(REGEXP_EXTRACT(x.url, 'P([0-9]+)$', 1) AS BIGINT)
        GROUP BY x.publisher_id
      ) pl ON pl.publisher_id = s.publisher_id
  ),
  -- oxjob #1926 (Jason, 29 Oct 2026): the Springer/Elsevier takedown rule (set_fields) hid Research Square preprints'
  -- abstracts, because Crossref names Research Square's publisher Springer, though each preprint is CC BY in its own
  -- Crossref deposit. Jason's condition: show only the preprint's OWN abstract under its own licence, never the published
  -- paper's. One row per Research Square DOI whose deposit itself carries an abstract under CC BY / CC0, keyed by that
  -- text's hash and length, so a location qualifies only when the abstract it holds IS its own deposit's (not a PDF
  -- fill, not another record's).
  -- Revert: delete this CTE, its join and column in base, preprint_own_abstract in collect_all_values, and put the
  -- takedown condition back inline as CASE WHEN <condition> THEN NULL in abstract_inverted_index (git revert).
  rs_deposit_abstracts AS (
    SELECT DISTINCT
      lower(native_id) AS rs_doi,
      xxhash64(abstract) AS abstract_hash,
      length(abstract) AS abstract_len
    FROM openalex.works.locations_parsed
    WHERE provenance = 'crossref'
      AND native_id LIKE '10.21203/%'
      AND abstract IS NOT NULL
      AND lower(license) IN ('cc-by', 'cc0')
  ),
  -- oxjob #737: the DOIs Retraction Watch lists as the original paper of a retraction and never as reinstated. Crossref
  -- carries a Retraction Watch retraction only when Retraction Watch names the notice's DOI (Crossref community thread
  -- 16586, 2026-10-15), so ~1.3K retractions never reach a Crossref record. Loaded daily by the Retraction Watch job.
  retraction_watch_dois AS (
    SELECT original_paper_doi AS doi
    FROM openalex.works.retraction_watch
    WHERE original_paper_doi IS NOT NULL
    GROUP BY original_paper_doi
    HAVING COUNT_IF(retraction_nature = 'Retraction') > 0 AND COUNT_IF(retraction_nature = 'Reinstatement') = 0
  ),
  priority_table AS (
    -- some comment
    SELECT
      *
    FROM
      openalex.system.priority_table
  ),
  openapc_paid AS (
    SELECT
      paper_id as work_id,
      /* if multiple rows per work, pick the latest year's amounts */
      MAX_BY(CAST(apc_in_euro AS DOUBLE), year) AS apc_in_euro,
      MAX_BY(CAST(apc_in_usd  AS DOUBLE), year) AS apc_in_usd
    FROM openalex.mid.work_openapc
    GROUP BY paper_id
  ),
  base AS (
    -- SHUFFLE_HASH(s), oxjob #1813: keep the hash build on mat_sources (280K rows). With the publisher-ancestor join inside
    -- mat_sources the optimizer lost its size estimate and built the hash on the 3B-location side (shadow build: 1.03 TB spill).
    SELECT /*+ BROADCAST(rsd, rw), SHUFFLE_HASH(s) */
      a.work_id,
      a.provenance,
      a.native_id,
      a.native_id_namespace,
      a.best_doi,
      -- oxjob #1902 F2 (Casey 2026-11-07): a tie between two Crossref DOIs of one work goes to the newest Crossref registration,
      -- for the work's DOI and its primary location: a publisher's older deposit of the same item was the dead DOI in 112 of 131
      -- measured live/dead ties (the alphabetically first, the old tie-break, in 81).
      CASE WHEN a.provenance = 'crossref' THEN a.created_date END AS cr_registered,
      -- oxjob #1581: the concept DOI this DataCite record is a version of (its single declared IsVersionOf DOI, else its own DOI
      -- minus a `.vN` suffix: Figshare and its portals declare nothing), so the concept DOI can win the work's DOI below
      CASE WHEN a.provenance = 'datacite' THEN COALESCE(
        CASE WHEN SIZE(ARRAY_DISTINCT(TRANSFORM(FILTER(a.ids, x -> x.relationship = 'IsVersionOf' AND LOWER(COALESCE(x.namespace, 'doi')) = 'doi'),
                    x -> regexp_replace(regexp_replace(LOWER(TRIM(x.id)), '^(https?://(dx[.])?doi[.]org/|doi:)', ''), '[^a-z0-9./-]', '')))) = 1
             THEN ARRAY_DISTINCT(TRANSFORM(FILTER(a.ids, x -> x.relationship = 'IsVersionOf' AND LOWER(COALESCE(x.namespace, 'doi')) = 'doi'),
                    x -> regexp_replace(regexp_replace(LOWER(TRIM(x.id)), '^(https?://(dx[.])?doi[.]org/|doi:)', ''), '[^a-z0-9./-]', '')))[0] END,
        CASE WHEN LOWER(a.native_id) RLIKE '[.]v[0-9]+$' THEN regexp_replace(LOWER(a.native_id), '[.]v[0-9]+$', '') END) END AS version_of_doi,
      -- oxjob #1057: the DOIs this DataCite dataset record declares it IsPartOf, minus its own (a Dataverse file names its
      -- dataset), so the parent's record wins the work's title, DOI and primary location when both sit on one work.
      -- Datasets only, as #1057's merge (Casey 2026-10-23): a chapter beside its book keeps its own title. A DOI the record
      -- also names as a version is left to #1581 (a Zenodo concept record declares IsPartOf its own version).
      CASE WHEN a.provenance = 'datacite' AND a.type = 'dataset' THEN ARRAY_EXCEPT(
        ARRAY_DISTINCT(TRANSFORM(FILTER(a.ids, x -> x.relationship = 'IsPartOf' AND LOWER(COALESCE(x.namespace, 'doi')) = 'doi' AND x.id IS NOT NULL),
          x -> regexp_replace(LOWER(TRIM(x.id)), '^(https?://(dx[.])?doi[.]org/|doi:)', ''))),
        ARRAY_UNION(ARRAY(LOWER(a.native_id)),
          TRANSFORM(FILTER(a.ids, x -> x.relationship IN ('IsVersionOf', 'HasVersion') AND LOWER(COALESCE(x.namespace, 'doi')) = 'doi' AND x.id IS NOT NULL),
            x -> regexp_replace(LOWER(TRIM(x.id)), '^(https?://(dx[.])?doi[.]org/|doi:)', '')))) END AS part_of_dois,
      a.title,
      a.type,
      a.abstract,
      a.referenced_works_count,
      a.referenced_works,
      a.abstract_inverted_index,
      -- oxjob #2142: a published version's own record (Crossref or PubMed, publishedVersion, not typed preprint). Its abstract
      -- outranks a preprint record's when a preprint and its article are one work: "the published side survives" (Casey,
      -- 2026-09-28), as cr_vor_priority does for titles. A PubMed record of a preprint (NIH preprint pilot) is typed preprint.
      COALESCE(a.provenance IN ('crossref', 'pubmed') AND a.type IS DISTINCT FROM 'preprint' AND a.version = 'publishedVersion', FALSE) AS is_published_record,
      -- oxjob #1926: a Research Square Crossref record holding its own deposit's CC abstract, and the licence this location
      -- shows is CC BY / CC0 too
      COALESCE(rsd.rs_doi IS NOT NULL AND lower(a.license) IN ('cc-by', 'cc0'), FALSE) AS is_preprint_own_cc_abstract,
      b.priority,
      a.openalex_created_dt,
      a.openalex_updated_dt,
      s.source_id,
      s.display_name,
      s.issn_l,
      s.issns,
      s.is_in_doaj,
      CASE
        WHEN GET(s.apc_prices, 0) IS NULL THEN NULL
        ELSE s.apc_prices
      END AS apc_prices,
      s.apc_usd,
      COALESCE(s.is_core, FALSE) AS is_core,
      COALESCE(s.listed_in, ARRAY()) AS listed_in,
      a.is_oa,
      COALESCE(a.is_oa, FALSE) AS is_oa_raw,
      COALESCE(s.is_in_doaj, FALSE) AS is_in_doaj_raw,
      COALESCE(
        is_in_doaj_raw
        AND (
          ISNULL(s.is_in_doaj_start_year)
          OR YEAR(a.published_date) >= s.is_in_doaj_start_year
        ),
        FALSE
      ) AS is_in_doaj_stg,
      COALESCE(
        s.is_oa_high_oa_rate
        AND (
          ISNULL(s.high_oa_rate_start_year)
          OR YEAR(a.published_date) >= s.high_oa_rate_start_year
        )
        OR s.is_fully_open_in_jstage,
        FALSE
      ) AS is_oa_high_rate,
      (
        is_oa_high_rate
        or is_in_doaj_stg
      ) AS source_is_oa,
      -- oxjob #1170: a curated is_oa wins over the journal rules (curated_fields, ApplyLocationCurations).
      CASE
        WHEN array_contains(a.curated_fields, 'is_oa') THEN is_oa_raw
        WHEN a.is_researchgate_copy THEN FALSE -- oxjob #1874: a repository's OA flag never opens a ResearchGate copy
        WHEN a.is_not_oa_copy THEN FALSE -- oxjob #1006: a copy on a dead or squatted host, or a CABI abstract record;
        -- oxjob #2266: a Zenodo record whose access right is restricted / closed / embargoed
        ELSE is_oa_raw OR source_is_oa
      END AS composite_is_oa,
      COALESCE(s.is_in_doaj_start_year, 0) AS is_in_doaj_start_year,
      s.source_type,
      a.source_name,
      a.publisher,
      a.published_date,
      a.volume,
      a.issue,
      a.first_page,
      a.last_page,
      -- oxjob #1871: a PubMed record's NLM Language (the language NLM's indexers record for the article itself) beats text
      -- classification: PubMed's title is NLM's English translation ("[...]") and its abstract the English one, so fastText
      -- read 'en' for ~3M Russian, Japanese, Chinese, French, German... articles. An English or 'und' (NULL) declaration
      -- adds nothing, so those keep the classification. Checked against the articles themselves (oxjob #1871 EXPLORE.md):
      -- NLM right on 96-100% of 100 per language where the article's own title or the publisher's record shows it.
      -- oxjob #1954 adds ja and zh (NLM right on 203 of 204 Japanese and 299 of 299 Chinese articles, every era, read on
      -- J-STAGE PDFs and the Chinese Medical Association's own pages) and hr for Croatian journals from 1991 (195 of 197 on
      -- a fresh draw; before 1991 and outside Croatia NLM's one Serbo-Croatian code also covers Serbian, Bosnian and
      -- Slovenian articles). he and sr wait: no article evidence reachable yet (oxjob #1954 README).
      CASE WHEN a.provenance = 'pubmed' AND a.language IS NOT NULL AND (a.language NOT IN ('en', 'he', 'hr', 'sr') OR (a.language = 'hr' AND YEAR(a.published_date) >= 1991 AND a.source_name IN ('Lijecnicki vjesnik', 'Acta medica Croatica : casopis Hravatske akademije medicinskih znanosti', 'Reumatizam', 'Arhiv za higijenu rada i toksikologiju', 'Acta medico-historica adriatica : AMHA', 'Jugoslavenska ginekologija i perinatologija', 'Acta stomatologica Croatica', 'Acta medica Iugoslavica', 'Neurologia croatica : glasilo Udruzenja neurologa Jugoslavije = official journal of Yugoslav Neurological Association', 'Geografski glasnik = Bulletin de geographie', 'Migracijske teme', 'Casopis za suvremenu povijest', 'Povijesni prilozi', 'Neuropsihijatrija', 'Zbornik Odsjeka za povijesne znanosti Zavoda za povijesne i drustvene znanosti Hrvatske akademije znanosti i umjetnosti', 'Acta geographica Croatica : glasilo Geografskog odjela PMF-a Sveucilista u Zagrebu', 'Geoadria', 'Radovi Medicinskog faculteta u Zagrebu'))) THEN a.language
           ELSE COALESCE(a.language_classification.language, a.language) END as language, --prefer fasttext classified language
      COALESCE(a.provenance = 'pubmed' AND a.language IS NOT NULL AND (a.language NOT IN ('en', 'he', 'hr', 'sr') OR (a.language = 'hr' AND YEAR(a.published_date) >= 1991 AND a.source_name IN ('Lijecnicki vjesnik', 'Acta medica Croatica : casopis Hravatske akademije medicinskih znanosti', 'Reumatizam', 'Arhiv za higijenu rada i toksikologiju', 'Acta medico-historica adriatica : AMHA', 'Jugoslavenska ginekologija i perinatologija', 'Acta stomatologica Croatica', 'Acta medica Iugoslavica', 'Neurologia croatica : glasilo Udruzenja neurologa Jugoslavije = official journal of Yugoslav Neurological Association', 'Geografski glasnik = Bulletin de geographie', 'Migracijske teme', 'Casopis za suvremenu povijest', 'Povijesni prilozi', 'Neuropsihijatrija', 'Zbornik Odsjeka za povijesne znanosti Zavoda za povijesne i drustvene znanosti Hrvatske akademije znanosti i umjetnosti', 'Acta geographica Croatica : glasilo Geografskog odjela PMF-a Sveucilista u Zagrebu', 'Geoadria', 'Radovi Medicinskog faculteta u Zagrebu'))), FALSE) AS is_nlm_declared_language,
      a.authors,
      TRANSFORM(
        a.urls,
        x -> STRUCT(
          REGEXP_REPLACE(x.url, 'dx.doi.org', 'doi.org') AS url,
          x.content_type
        )
      ) AS urls,
      -- oxjob #314: the article's own registered license wins; the DOAJ
      -- journal default only fills a gap.
      CASE
        WHEN array_contains(a.curated_fields, 'license') THEN a.license -- oxjob #1170: a curated licence wins
        WHEN is_in_doaj_stg THEN COALESCE(a.license, s.doaj_license)
        ELSE a.license
      END AS license,
      -- oxjob #2269: the licence came from a curation (served even on a closed location: served_license below)
      COALESCE(array_contains(a.curated_fields, 'license'), FALSE) AS is_license_curated,
      s.institution_name,
      s.publisher_name,
      s.parent_publisher,
      s.publisher_ancestor_urls,
      s.publisher_ancestor_names,
      s.institution_id,
      s.publisher_id,
      -- oxjob #805: content on the journal's own endpoint is the version of record.
      -- Same gate as the host_type companion clause below.
      CASE
        WHEN a.provenance IN ('repo', 'repo_backfill')
        -- oxjob #2494: and a conference series' own endpoint. The three OA gates below (DataCite publisher copy, journal-host
        -- endpoint, OA-venue copy from any non-repository record) read conference series as journals: a proceedings series
        -- is the publisher's venue, and its fully-OA copies are gold whichever record carried them. Revert: 'journal' only.
        AND s.source_type IN ('journal', 'conference')
        AND jh.endpoint_id IS NOT NULL
        THEN 'publishedVersion'
        ELSE a.version
      END AS version,
      a.landing_page_url,
      a.pdf_url,
      a.is_dead_link, -- oxjob #1006: read by the locations sort only (sort_score 12)
      a.is_retracted OR rw.doi IS NOT NULL AS is_retracted, -- oxjob #737
      a.raw_type,
      a.endpoint_id,
      a.pdf_s3_id,
      a.grobid_s3_id,
      CASE
        WHEN s.source_id IS NULL THEN NULL
        WHEN (
          a.provenance = 'crossref'
          AND s.source_type != 'repository'
        ) THEN 'publisher'
        -- DataCite container-ISSN tier companion (SPEC-v2 2026-08-12): a DataCite
        -- location resolved to a journal is a publisher location, not a repository
        -- copy (otherwise DataCite's hardcoded is_oa would keep these rows 'green'
        -- and set any_repository_has_fulltext).
        WHEN (
          a.provenance = 'datacite'
          AND s.source_type IN ('journal', 'conference')
        ) THEN 'publisher'
        -- Journal-endpoint PMH companion (oxjob #805): the PMH endpoint IS the journal,
        -- so its locations are publisher copies. Gated on the registry's opt-in
        -- is_journal_host flag (oxjob #1409), not on the binding alone: endpoint_to_source
        -- holds journal bindings that are really multi-journal aggregators (measured
        -- 08-19: two misbindings alone cover 12.7M locations). Keyed on the location's own
        -- endpoint, so other repositories' copies attached to the journal stay repository.
        WHEN (
          a.provenance IN ('repo', 'repo_backfill')
          AND s.source_type IN ('journal', 'conference')
          AND jh.endpoint_id IS NOT NULL
        ) THEN 'publisher'
        ELSE 'repository'
      END AS host_type,
      CASE
        WHEN composite_is_oa
        AND host_type = 'publisher' THEN CASE
          WHEN ZEROIFNULL(s.apc_usd) = 0
          AND source_is_oa THEN 1
          WHEN source_is_oa THEN 2
          WHEN a.license IS NOT NULL
          AND a.license != 'publisher-specific-oa' THEN 3
          ELSE 4
        END
        -- oxjob #1867: gold and diamond mean "published in a fully-OA journal" (help.openalex.org, Open access), so an
        -- open copy whose journal is fully OA that year is gold (diamond with no known APC, as above) whichever record
        -- carried it: MAG copies of OA-journal articles were green (2.3M works). Repository records keep the rule below:
        -- an endpoint bound to a journal is the publisher only when the registry says it hosts it (#1409, #1225).
        WHEN composite_is_oa
        AND s.source_type IN ('journal', 'conference')
        AND source_is_oa
        AND a.provenance NOT IN ('repo', 'repo_backfill') THEN CASE
          WHEN ZEROIFNULL(s.apc_usd) = 0 THEN 1
          ELSE 2
        END
        -- oxjob #1867: no source means no known journal, so a repository-type record (DataCite, repository, MAG) or a
        -- preprint (the docs count preprint servers as repositories) is green, not gold (6.4M works were gold).
        WHEN host_type IS NULL
        AND (
          a.is_oa
          OR composite_is_oa
        )
        AND (
          a.provenance != 'crossref'
          OR COALESCE(a.type, '') = 'preprint'
        ) THEN 5
        -- A Crossref record with no source stays gold for now: many are peer reviews, comments or supplements of
        -- fully-OA journals (Copernicus, eLife, F1000) that OpenAlex has not bound to their journal, so no other label is
        -- right for all of them yet (#1867 Handoff).
        WHEN host_type IS NULL
        AND (
          a.is_oa
          OR composite_is_oa
        ) THEN 2
        WHEN (
          a.is_oa
          OR composite_is_oa
        )
        AND host_type = 'repository' THEN 5
        ELSE 6
      END AS oa_status
    FROM
      -- oxjob #1874: a ResearchGate copy is not OA evidence (help.openalex.org, Open access: ResearchGate "requires login
      -- before reading. OpenAlex already excludes it"). A ResearchGate copy is a DataCite record under its DOI prefix
      -- 10.13140, or a location whose URL (the PDF, else the landing page) is on researchgate.net. DataCite.py marks every
      -- DataCite record open, and the ingest-time OA flags only ever turn on (#994), so the rule lives here, where every
      -- work is rebuilt each night. A curated is_oa still wins (#1170).
      -- oxjob #1006: a location on a host that no longer serves its item (openalex.works.dead_link_hosts, built in the cell
      -- above the #2209 picks) is no OA copy and sorts after every other location (is_dead_link -> sort_score 12), so it is
      -- never primary or best OA while the work has another location; a work whose only location is dead keeps it, closed.
      -- A cabdirect.org record URL becomes its CABI Digital Library record (same accession number), never an OA copy. A PDF
      -- link on a registry host beside a live landing page is dropped. A curated is_oa still wins (#1170).
      -- Revert: restore this subquery, the is_dead_link / is_not_oa_copy lines in base and sort_score.
      -- oxjob #2266: a Zenodo record whose dc:rights say restricted / closed / embargoed (Repo.py stores it is_oa false;
      -- Zenodo's API showed no public file on 50 of 50 sampled, 16 Nov 2026) is no OA copy, though Zenodo is an OA source
      -- (source_is_oa would open it: ~75K works carried such a false open copy, ~29K of them read open on it alone). It
      -- stays a location, closed. A curated is_oa still wins (#1170). Revert: drop each is_closed_zenodo_record mention.
      -- oxjob #1628: the same holds for every copy of a deposit Zenodo's latest OAI record restricts (zenodo_closed_records,
      -- built above): its DataCite DOI 10.5281/zenodo.<record> (DataCite.py opens every DataCite record) and a repo record
      -- whose stored is_oa is stale. Their own is_oa turns false too, since a repository copy stays open on is_oa_raw.
      -- Revert: drop the zc join, its OR and the #1628 WHEN line (the next night reopens them).
      -- oxjob #2244: an en.cnki.com.cn article URL (Article_en/<DB>TOTAL-<FILE>.htm, ~5.8M locations) becomes CNKI's own
      -- oversea.cnki.net page for the same article (kcms/detail/detail.aspx?dbcode=<DB>&filename=<FILE>): the old host's
      -- https fails TLS and its http answers 418 (15 Nov 2026), and the new link opened the same article on a fresh random
      -- draw (#2244 PLAN.md Bar). Only the URL changes: the location keeps its OA flag, source and order.
      -- Revert: drop the #2244 WHEN line (the next night restores the en.cnki.com.cn URLs).
      (
        SELECT
          * EXCEPT (is_oa, landing_page_url, pdf_url, dead_link_kind, dead_link_rewrite, dead_pdf_kind, cab_an, is_closed_zenodo_record, language),
          CASE
            WHEN is_researchgate_copy AND NOT COALESCE(array_contains(curated_fields, 'is_oa'), FALSE) THEN FALSE
            WHEN dead_link_kind IS NOT NULL AND NOT COALESCE(array_contains(curated_fields, 'is_oa'), FALSE) THEN FALSE -- oxjob #1006
            WHEN is_closed_zenodo_record AND NOT COALESCE(array_contains(curated_fields, 'is_oa'), FALSE) THEN FALSE -- oxjob #1628
            ELSE is_oa
          END AS is_oa,
          CASE
            WHEN dead_link_rewrite = 'cabi_an' AND cab_an <> '' THEN CONCAT('https://www.cabidigitallibrary.org/doi/10.5555/', cab_an)
            -- oxjob #2244: CNKI's English site (en.cnki.com.cn) now answers only through oversea.cnki.net's CDN front, which
            -- serves another certificate (https fails) and 418 to http; CNKI serves the same article at oversea.cnki.net
            WHEN landing_page_url RLIKE '(?i)^https?://(www[.])?en[.]cnki[.]com[.]cn/article(_|%5f)en/[a-z]+total-[a-z0-9.]+[.]htm$'
              THEN CONCAT('https://oversea.cnki.net/kcms/detail/detail.aspx?dbcode=',
                upper(REGEXP_EXTRACT(landing_page_url, '(?i)/article(?:_|%5f)en/([a-z]+)total-', 1)),
                '&filename=', upper(REGEXP_EXTRACT(landing_page_url, '(?i)total-([a-z0-9.]+)[.]htm$', 1)))
            ELSE landing_page_url
          END AS landing_page_url,
          CASE
            WHEN dead_link_kind IS NULL AND dead_pdf_kind IS NOT NULL THEN NULL
            ELSE pdf_url
          END AS pdf_url,
          COALESCE(dead_link_kind IS NOT NULL AND NOT (dead_link_rewrite = 'cabi_an' AND cab_an <> ''), FALSE) AS is_dead_link,
          dead_link_kind IS NOT NULL OR is_closed_zenodo_record AS is_not_oa_copy,
          -- oxjob #724: a record's declared language, normalised to what the API documents (ISO 639-1, else a three-letter
          -- ISO 639-2/3 code). Records declare 'EN', 'en-US', 'zh-CN', 'eng', '中文', 'enc' (Zenodo's picker, 97% English
          -- text by fastText) and served them as-is: 153 bad values on 227,807 works. Key: lowercase, trimmed, region suffix
          -- cut ('en-US' -> 'en'). The MAP holds every ISO 639-1 code (itself) plus aliases (639-2/B and /T codes, names,
          -- country codes used as languages): '' there means no language (und, mis, mul, zxx, 'other', ambiguous country
          -- codes). A key not in the MAP stays when it is a three-letter code outside the private-use qaa-qtz, else NULL.
          -- The fastText label still wins over this value (below), and PubMed's declared codes do not change. Built from
          -- oxjobs #724 work/language_normalizer_tables.json, SQL == Python on all 1,233 distinct values, 18 Nov 2026.
          -- Revert: drop language from the EXCEPT list above and this column (the next night serves the raw values).
          NULLIF(COALESCE(
          try_element_at(MAP(
            'aa', 'aa', 'aar', 'aa', 'ab', 'ab', 'abk', 'ab', 'ae', 'ae', 'af', 'af', 'afr', 'af', 'ak', 'ak', 'aka', 'ak', 'al', '',
            'alb', 'sq', 'am', 'am', 'amh', 'am', 'an', 'an', 'ang', 'en', 'anglais', 'en', 'ar', 'ar', 'ara', 'ar', 'arb', 'ar', 'arg', 'an',
            'arm', 'hy', 'as', 'as', 'asm', 'as', 'at', '', 'au', '', 'av', 'av', 'ava', 'av', 'ave', 'ae', 'ay', 'ay', 'aym', 'ay',
            'az', 'az', 'aze', 'az', 'azj', 'az', 'ba', 'ba', 'bak', 'ba', 'bam', 'bm', 'baq', 'eu', 'be', 'be', 'bel', 'be', 'ben', 'bn',
            'bg', 'bg', 'bh', 'bh', 'bi', 'bi', 'bis', 'bi', 'bm', 'bm', 'bn', 'bn', 'bo', 'bo', 'bod', 'bo', 'bos', 'bs', 'br', 'br',
            'bre', 'br', 'bs', 'bs', 'bul', 'bg', 'bur', 'my', 'by', 'be', 'ca', 'ca', 'cat', 'ca', 'ce', 'ce', 'ces', 'cs', 'ch', 'ch',
            'cha', 'ch', 'che', 'ce', 'chi', 'zh', 'chn', '', 'chu', 'cu', 'chv', 'cv', 'cmn', 'zh', 'cn', 'zh', 'co', 'co', 'cor', 'kw',
            'cos', 'co', 'cr', 'cr', 'cre', 'cr', 'cs', 'cs', 'cu', 'cu', 'cv', 'cv', 'cy', 'cy', 'cym', 'cy', 'cz', 'cs', 'cze', 'cs',
            'da', 'da', 'dan', 'da', 'danish', 'da', 'de', 'de', 'deu', 'de', 'deutsch', 'de', 'div', 'dv', 'dk', 'da', 'du', '', 'dut', 'nl',
            'dutch', 'nl', 'dv', 'dv', 'dz', 'dz', 'dzo', 'dz', 'ee', 'ee', 'ekk', 'et', 'el', 'el', 'ell', 'el', 'en', 'en', 'enc', 'en',
            'eng', 'en', 'engilsh', 'en', 'englisch', 'en', 'englisg', 'en', 'english', 'en', 'eo', 'eo', 'epo', 'eo', 'es', 'es', 'esp', 'es', 'español', 'es',
            'est', 'et', 'et', 'et', 'eu', 'eu', 'eus', 'eu', 'ewe', 'ee', 'fa', 'fa', 'falsch', '', 'fao', 'fo', 'fas', 'fa', 'ff', 'ff',
            'fi', 'fi', 'fij', 'fj', 'fin', 'fi', 'fj', 'fj', 'fo', 'fo', 'fr', 'fr', 'fra', 'fr', 'france', 'fr', 'français', 'fr', 'fre', 'fr',
            'free', '', 'french', 'fr', 'fry', 'fy', 'ful', 'ff', 'fy', 'fy', 'ga', 'ga', 'gb', 'en', 'gd', 'gd', 'ge', '', 'geman', 'de',
            'geo', 'ka', 'ger', 'de', 'german', 'de', 'gl', 'gl', 'gla', 'gd', 'gle', 'ga', 'glg', 'gl', 'glv', 'gv', 'gn', 'gn', 'gr', 'el',
            'gre', 'el', 'greek', 'el', 'grn', 'gn', 'gu', 'gu', 'gug', 'gn', 'guj', 'gu', 'gv', 'gv', 'ha', 'ha', 'hat', 'ht', 'hau', 'ha',
            'hbs', 'sh', 'he', 'he', 'heb', 'he', 'her', 'hz', 'hi', 'hi', 'hin', 'hi', 'hmo', 'ho', 'ho', 'ho', 'hr', 'hr', 'hrv', 'hr',
            'ht', 'ht', 'hu', 'hu', 'hun', 'hu', 'hy', 'hy', 'hye', 'hy', 'hz', 'hz', 'ia', 'ia', 'ibo', 'ig', 'ice', 'is', 'id', 'id',
            'ido', 'io', 'ie', 'ie', 'ig', 'ig', 'ii', 'ii', 'iii', 'ii', 'ik', 'ik', 'iku', 'iu', 'ile', 'ie', 'in', 'id', 'ina', 'ia',
            'ind', 'id', 'ingles', 'en', 'inglés', 'en', 'io', 'io', 'ipk', 'ik', 'is', 'is', 'isl', 'is', 'it', 'it', 'ita', 'it', 'ital', 'it',
            'italiano', 'it', 'iu', 'iu', 'iw', 'he', 'ja', 'ja', 'jap', 'ja', 'jav', 'jv', 'ji', 'yi', 'jp', 'ja', 'jpn', 'ja', 'jv', 'jv',
            'jw', 'jv', 'ka', 'ka', 'kal', 'kl', 'kan', 'kn', 'kas', 'ks', 'kat', 'ka', 'kau', 'kr', 'kaz', 'kk', 'kg', 'kg', 'khk', 'mn',
            'khm', 'km', 'ki', 'ki', 'kik', 'ki', 'kin', 'rw', 'kir', 'ky', 'kj', 'kj', 'kk', 'kk', 'kl', 'kl', 'km', 'km', 'kmr', 'ku',
            'kn', 'kn', 'ko', 'ko', 'kom', 'kv', 'kon', 'kg', 'kor', 'ko', 'kr', 'kr', 'ks', 'ks', 'ku', 'ku', 'kua', 'kj', 'kur', 'ku',
            'kv', 'kv', 'kw', 'kw', 'ky', 'ky', 'kz', '', 'la', 'la', 'lao', 'lo', 'lat', 'la', 'lav', 'lv', 'lb', 'lb', 'lg', 'lg',
            'li', 'li', 'lim', 'li', 'lin', 'ln', 'lit', 'lt', 'ln', 'ln', 'lo', 'lo', 'lt', 'lt', 'ltz', 'lb', 'lu', 'lu', 'lub', 'lu',
            'lug', 'lg', 'lv', 'lv', 'lvs', 'lv', 'mac', 'mk', 'mah', 'mh', 'mal', 'ml', 'mao', 'mi', 'mar', 'mr', 'may', 'ms', 'mg', 'mg',
            'mh', 'mh', 'mi', 'mi', 'mis', '', 'mk', 'mk', 'mkd', 'mk', 'ml', 'ml', 'mlg', 'mg', 'mlt', 'mt', 'mn', 'mn', 'mo', 'ro',
            'mon', 'mn', 'mr', 'mr', 'mri', 'mi', 'ms', 'ms', 'msa', 'ms', 'mt', 'mt', 'mul', '', 'multi', '', 'my', 'my', 'mya', 'my',
            'na', 'na', 'nan', '', 'nau', 'na', 'nav', 'nv', 'nb', 'nb', 'nbl', 'nr', 'nd', 'nd', 'nde', 'nd', 'ndo', 'ng', 'ne', 'ne',
            'nep', 'ne', 'ng', 'en', 'nl', 'nl', 'nld', 'nl', 'nn', 'nn', 'nno', 'nn', 'no', 'no', 'nob', 'nb', 'nolang', '', 'none', '',
            'nor', 'no', 'npi', 'ne', 'nr', 'nr', 'nv', 'nv', 'ny', 'ny', 'nya', 'ny', 'oc', 'oc', 'oci', 'oc', 'oj', 'oj', 'oji', 'oj',
            'om', 'om', 'or', 'or', 'ori', 'or', 'orm', 'om', 'ory', 'or', 'os', 'os', 'oss', 'os', 'other', '', 'otro', '', 'pa', 'pa',
            'pan', 'pa', 'per', 'fa', 'pes', 'fa', 'pi', 'pi', 'pidgin', '', 'pl', 'pl', 'pli', 'pi', 'plt', 'mg', 'po', '', 'pol', 'pl',
            'polish', 'pl', 'por', 'pt', 'ps', 'ps', 'pt', 'pt', 'pus', 'ps', 'qu', 'qu', 'que', 'qu', 're', '', 'rm', 'rm', 'rn', 'rn',
            'ro', 'ro', 'roh', 'rm', 'romanian', 'ro', 'ron', 'ro', 'rs', '', 'ru', 'ru', 'rum', 'ro', 'run', 'rn', 'rus', 'ru', 'rw', 'rw',
            'sa', 'sa', 'sag', 'sg', 'san', 'sa', 'sc', 'sc', 'sd', 'sd', 'se', 'se', 'sg', 'sg', 'sh', 'sh', 'si', 'si', 'silent', '',
            'sin', 'si', 'sk', 'sk', 'sl', 'sl', 'slk', 'sk', 'slo', 'sk', 'slv', 'sl', 'sm', 'sm', 'sme', 'se', 'smo', 'sm', 'sn', 'sn',
            'sna', 'sn', 'snd', 'sd', 'so', 'so', 'som', 'so', 'sot', 'st', 'sp', 'es', 'spa', 'es', 'spanish', 'es', 'sq', 'sq', 'sqi', 'sq',
            'sr', 'sr', 'srd', 'sc', 'srp', 'sr', 'ss', 'ss', 'ssw', 'ss', 'st', 'st', 'su', 'su', 'sun', 'su', 'sv', 'sv', 'sw', 'sw',
            'swa', 'sw', 'swe', 'sv', 'swh', 'sw', 'ta', 'ta', 'tah', 'ty', 'tam', 'ta', 'tat', 'tt', 'te', 'te', 'tel', 'te', 'test', '',
            'tg', 'tg', 'tgk', 'tg', 'tgl', 'tl', 'th', 'th', 'tha', 'th', 'thai', 'th', 'ti', 'ti', 'tib', 'bo', 'tir', 'ti', 'tj', '',
            'tk', 'tk', 'tl', 'tl', 'tn', 'tn', 'to', 'to', 'ton', 'to', 'tr', 'tr', 'ts', 'ts', 'tsn', 'tn', 'tso', 'ts', 'tt', 'tt',
            'tu', '', 'tuk', 'tk', 'tur', 'tr', 'tw', 'tw', 'twi', 'tw', 'ty', 'ty', 'ua', 'uk', 'ug', 'ug', 'uig', 'ug', 'uk', 'uk',
            'ukr', 'uk', 'und', '', 'unk', '', 'ur', 'ur', 'urd', 'ur', 'us', 'en', 'uz', 'uz', 'uzb', 'uz', 'uzn', 'uz', 've', 've',
            'ven', 've', 'vi', 'vi', 'vie', 'vi', 'vn', '', 'vo', 'vo', 'vol', 'vo', 'wa', 'wa', 'wel', 'cy', 'wln', 'wa', 'wo', 'wo',
            'wol', 'wo', 'xh', 'xh', 'xho', 'xh', 'xx', '', 'ydd', 'yi', 'yi', 'yi', 'yid', 'yi', 'yo', 'yo', 'yor', 'yo', 'yu', '',
            'za', 'za', 'zh', 'zh', 'zha', 'za', 'zho', 'zh', 'zsm', 'ms', 'zu', 'zu', 'zul', 'zu', 'zxx', '', 'zz', '', '中文', 'zh',
            '俄文', 'ru', '俄语', 'ru', '其他', '', '和文', 'ja', '德文', 'de', '德语', 'de', '日文', 'ja', '日语', 'ja', '汉语', 'zh', '法文', 'fr',
            '法语', 'fr', '英国', 'en', '英文', '', '英语', 'en', '韓文', 'ko', '韩文', 'ko', '韩语', 'ko', '영어', 'en', '한국어', 'ko', 'ｆｒ', 'fr'),
            regexp_replace(regexp_replace(lower(language), '^\\s+|\\s+\\z', ''), '(?s)(?<=^[a-z]{2,3})[-_].*', '')),
          regexp_extract(regexp_replace(regexp_replace(lower(language), '^\\s+|\\s+\\z', ''), '(?s)(?<=^[a-z]{2,3})[-_].*', ''), '^((?!q[a-t])[a-z]{3})\\z', 1)), '') AS language
        FROM (
          SELECT /*+ BROADCAST(dl), BROADCAST(dp), BROADCAST(zc) */
            lm.*,
            COALESCE(
              (lm.native_id_namespace = 'doi' AND lower(lm.native_id) LIKE '10.13140/%')
              OR lower(COALESCE(lm.pdf_url, lm.landing_page_url, '')) RLIKE '^https?://([^/]*[.])?researchgate[.]net([/:?#]|$)',
              FALSE
            ) AS is_researchgate_copy,
            COALESCE(lm.provenance = 'repo' AND lm.native_id LIKE 'oai:zenodo.org:%' AND NOT lm.is_oa, FALSE)
              OR zc.rec IS NOT NULL AS is_closed_zenodo_record, -- oxjob #2266, #1628
            dl.kind AS dead_link_kind,
            dl.rewrite AS dead_link_rewrite,
            dp.kind AS dead_pdf_kind,
            REGEXP_EXTRACT(lower(COALESCE(lm.landing_page_url, '')), 'cabdirect[.]org/(?:cabdirect/)?abstracts?/([0-9]{11})', 1) AS cab_an
          FROM
            openalex.works.locations_mapped_curated lm
            LEFT JOIN openalex.works.dead_link_hosts dl
              ON dl.host = REGEXP_EXTRACT(lower(COALESCE(lm.landing_page_url, lm.pdf_url)), '^[a-z]+://(?:www[.])?([^/:?#]+)', 1)
              AND (dl.url_rlike IS NULL OR lower(COALESCE(lm.landing_page_url, lm.pdf_url)) RLIKE dl.url_rlike)
            LEFT JOIN openalex.works.dead_link_hosts dp
              ON dp.host = REGEXP_EXTRACT(lower(lm.pdf_url), '^[a-z]+://(?:www[.])?([^/:?#]+)', 1)
              AND (dp.url_rlike IS NULL OR lower(lm.pdf_url) RLIKE dp.url_rlike)
            LEFT JOIN openalex.works.zenodo_closed_records zc -- oxjob #1628 (one row per record, so no fan-out)
              ON zc.rec = CASE
                WHEN lm.provenance = 'repo'
                  THEN TRY_CAST(REGEXP_EXTRACT(lower(lm.native_id), '^oai:zenodo[.]org:([0-9]+)$', 1) AS BIGINT)
                WHEN lm.provenance = 'datacite'
                  THEN TRY_CAST(REGEXP_EXTRACT(lower(lm.native_id), '^10[.]5281/zenodo[.]([0-9]+)$', 1) AS BIGINT)
              END
        )
      ) a
      LEFT JOIN priority_table b USING (provenance)
      LEFT JOIN mat_sources s ON a.source_id = s.source_id
      -- journal-host endpoints (oxjob #1409); live registry rows only, unique by endpoint_id
      LEFT JOIN openalex.sources.endpoint_to_source jh
        ON jh.endpoint_id = a.endpoint_id AND jh.is_journal_host
      -- oxjob #1926: at most one row per location (DISTINCT on the join keys)
      LEFT JOIN rs_deposit_abstracts rsd
        ON a.provenance = 'crossref'
        AND a.abstract IS NOT NULL
        AND lower(a.native_id) = rsd.rs_doi
        AND xxhash64(a.abstract) = rsd.abstract_hash
        AND length(a.abstract) = rsd.abstract_len
      LEFT JOIN retraction_watch_dois rw ON lower(a.best_doi) = rw.doi
  ),
  -- oxjob #1941: works whose Crossref posted-content record was posted more than two years after other records show the
  -- work existed: a late re-post (a preprint server's or repository's copy of an old paper; a DOI re-registered years
  -- later, e.g. Firenze University Press's 2026 Crossref records titled '<journal> - contenuto non piu disponibile' over
  -- DOIs of 2009-2019 articles). "Other records show it existed" = a PubMed record dated then, or the earlier DataCite
  -- registration of the same DOI suffix, or two of {DataCite, MAG, repository} dated by then (one family alone is not
  -- enough: placeholder years such as Morressier aggregator copies dated 2017-01-01, AACR supplements dated with their
  -- article, a Zenodo deposit dated 2012 in 2023). A ResearchGate copy never counts (user-entered years, #1874). The re-post
  -- must be the same people's work: it shares an author surname with another record, or re-registers a DOI suffix another
  -- record holds, or no other record names authors (keeps a fused work's other publication from taking its identity).
  -- Works with a Crossref version of record are left alone (it already wins). One row per work (~7.2K), broadcast into
  -- base_with_abstract_lens. Bar and sizes: oxjob #1941 README. Revert: delete these three CTEs, the join and the
  -- is_late_repost / is_late_twin uses below.
  late_repost_records AS (
    SELECT
      work_id, provenance, version, published_date,
      provenance = 'crossref' AND raw_type = 'posted-content' AS is_pc,
      NULLIF(REGEXP_EXTRACT(LOWER(native_id), '^10[.][0-9]+/(.+)', 1), '') AS doi_suffix,
      -- the same ResearchGate test as base (oxjob #1874)
      COALESCE(
        (native_id_namespace = 'doi' AND lower(native_id) LIKE '10.13140/%')
        OR lower(COALESCE(pdf_url, landing_page_url, '')) RLIKE '^https?://([^/]*[.])?researchgate[.]net([/:?#]|$)',
        FALSE
      ) AS is_rg,
      COALESCE(FILTER(TRANSFORM(authors, a -> LOWER(REGEXP_REPLACE(
        CASE WHEN INSTR(a.name, ',') > 0 THEN SPLIT(a.name, ',')[0] ELSE ELEMENT_AT(SPLIT(TRIM(a.name), ' '), -1) END,
        '[^\\p{L}]', ''))), s -> LENGTH(s) >= 2), ARRAY()) AS surnames
    FROM openalex.works.locations_mapped_curated
    WHERE work_id IN (SELECT work_id FROM openalex.works.locations_mapped_curated WHERE provenance = 'crossref' AND raw_type = 'posted-content')
  ),
  late_repost_work_dates AS (
    SELECT
      work_id,
      MAX(CASE WHEN provenance = 'crossref' AND version = 'publishedVersion' THEN 1 ELSE 0 END) AS has_vor,
      MIN(CASE WHEN provenance = 'pubmed' THEN published_date END) AS pubmed_min,
      MIN(CASE WHEN provenance = 'datacite' AND NOT is_rg THEN published_date END) AS datacite_min,
      MIN(CASE WHEN provenance = 'mag' THEN published_date END) AS mag_min,
      MIN(CASE WHEN provenance IN ('repo', 'repo_backfill') AND NOT is_rg THEN published_date END) AS repo_min,
      ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(CASE WHEN NOT is_pc THEN surnames END))) AS other_surnames,
      COLLECT_LIST(CASE WHEN provenance IN ('crossref', 'datacite') AND NOT is_pc AND NOT is_rg
                        THEN NAMED_STRUCT('doi_suffix', doi_suffix, 'published_date', published_date) END) AS doi_records
    FROM late_repost_records
    GROUP BY work_id
  ),
  late_reposts AS (
    SELECT work_id, COLLECT_SET(doi_suffix) AS late_suffixes, MIN(corroborated_date) AS late_corroborated_date
    FROM (
      SELECT
        r.work_id, r.doi_suffix, r.published_date,
        LEAST(
          d.pubmed_min,
          ARRAY_MIN(TRANSFORM(FILTER(d.doi_records, x -> x.doi_suffix = r.doi_suffix), x -> x.published_date)),
          TRY_ELEMENT_AT(ARRAY_SORT(FILTER(ARRAY(d.datacite_min, d.mag_min, d.repo_min), x -> x IS NOT NULL)), 2)
        ) AS corroborated_date,
        ARRAYS_OVERLAP(r.surnames, d.other_surnames) OR EXISTS(d.doi_records, x -> x.doi_suffix = r.doi_suffix)
          OR SIZE(d.other_surnames) = 0 AS same_people
      FROM late_repost_records r JOIN late_repost_work_dates d USING (work_id)
      WHERE r.is_pc AND d.has_vor = 0 AND LENGTH(r.doi_suffix) >= 6
    )
    WHERE DATEDIFF(published_date, corroborated_date) > 730 AND same_people
    GROUP BY work_id
  ),
  -- oxjob #1545: a whole abstract supplement (Crossref record with no authors, 50+ pages, 'abstract(s)' in its title)
  -- takes no byline from the single-abstract records that cite its DOI. The publisher deposits no authors for these
  -- books (e.g. Brain Injury 37(8) 758-1039 'ABSTRACTS', EJI 46 S1 1-1274 'Abstracts of ICI 2016'); byline_pick used
  -- to fall through to one abstract's repository or MAG record and credit its authors, institutions and profiles with
  -- the whole book. Size and bar: oxjob #1545 README. Revert: delete this CTE and the anti-join in byline_pick.
  whole_abstract_supplements AS (
    SELECT DISTINCT work_id
    FROM openalex.works.locations_mapped_curated
    WHERE provenance = 'crossref' AND COALESCE(SIZE(authors), 0) = 0
      AND TRY_CAST(last_page AS INT) - TRY_CAST(first_page AS INT) >= 49
      AND LOWER(title) RLIKE '(^|[^a-z])abstracts?([^a-z]|$)'
  ),
  -- authors start
  byline_pick AS (
    SELECT
        work_id,
        authors,
        provenance AS byline_provenance,
        native_id AS byline_native_id
    FROM openalex.works.locations_mapped_curated
    LEFT JOIN openalex.system.priority_table USING (provenance)
    LEFT ANTI JOIN whole_abstract_supplements USING (work_id)
    WHERE authors IS NOT NULL
        AND SIZE(authors) > 0
    -- Deterministic tie-break (oxjob #608): same-priority locations (e.g. multiple
    -- datacite deposit copies) used to be ordered by shuffle luck, flapping the author
    -- list run-over-run and re-corrupting position-bound author_ids. authors_exist DESC
    -- keeps landing-page-derived lists (rows whose native record had no authors) from
    -- outranking native lists by size; SIZE DESC prefers full rosters over 1-author
    -- collective entries; native_id, then updated_date and a content hash (duplicate
    -- (provenance, native_id) harvest copies flap otherwise, oxjob #649) make it total.
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY work_id
        ORDER BY priority ASC, authors_exist DESC NULLS LAST, SIZE(authors) DESC, native_id ASC,
            updated_date DESC NULLS LAST, xxhash64(to_json(authors)) ASC
    ) = 1
  ),
  -- oxjob #1158: a Crossref byline that is a truncated prefix of the work's own PubMed record takes PubMed's remaining
  -- authors after its own (openalex.works.pubmed_fuller_bylines, cell above: ~270K rows, broadcast). Crossref's seats are
  -- kept as they are; only seats are added. The row applies only to the Crossref record and list size it was built from.
  deduplicated_authors_source AS (
    SELECT
        bp.work_id,
        bp.authors,
        EXISTS(bp.authors.is_corresponding, x -> x = TRUE) AS is_corresponding_exists,
        ARRAY_SIZE(bp.authors) AS author_count,
        bp.byline_provenance,
        bp.byline_native_id  -- oxjob #2028: the Crossref DOI a group-member seat is keyed on (authors_with_corresponding)
    FROM (
      SELECT /*+ BROADCAST(fb) */ p.work_id, IF(fb.work_id IS NOT NULL, CONCAT(p.authors, fb.authors_tail), p.authors) AS authors,
             p.byline_provenance, p.byline_native_id
      FROM byline_pick p
      LEFT JOIN openalex.works.pubmed_fuller_bylines fb
        ON fb.work_id = p.work_id AND p.byline_provenance = 'crossref'
       AND fb.cr_native_id = p.byline_native_id AND fb.cr_n = SIZE(p.authors)
    ) bp
  ),
  exploded_authors_src AS (
      SELECT
          work_id,
          author_count,
          is_corresponding_exists,
          POSEXPLODE(authors) AS (original_author_order, author_data),
          TRIM(author_data.name) AS raw_author_name,
          -- oxjob #1715 gap D: a byline taken from MAG carries no ORCID. MAG printed none: its seat ORCIDs are each seat's
          -- legacy profile ORCID copied back (CreateAuthorObservedOrcids already drops them as provenance_mag, circular), so
          -- unbound MAG seats showed another person's ORCID (oxjob #1715 tickets 26937, 27308). Sized 17 Nov 2026: 20,649,702
          -- works / 34,353,573 seats lose raw_orcid; 1,724 seats (0.005%) carry a value a repository record of the same work
          -- also prints. author.orcid of a bound seat still comes from its profile. Revert: git revert of this commit.
          IF(byline_provenance = 'mag', CAST(NULL AS STRING), author_data.orcid) AS orcid,
          author_data.affiliations.name AS raw_affiliation_strings,
          author_data.is_corresponding AS is_corresponding_from_source,
          byline_provenance, byline_native_id  -- oxjob #2028
      FROM deduplicated_authors_source
  ),
  -- oxjob #583 (Case B): a byline of ONE entry whose name holds the whole author list becomes one seat per name.
  -- Harvard Dataverse 'Ratkovic, Marc T. (University Of Wisconsin, Madison)\x3B Eng, Kevin H. (...)', AJMC Crossref
  -- 'Joao A. de Andrade, MD\x3BGary M. Owens, MD\x3B...', DOAJ 'Hartmut BOESCH,Yi LIU,Paul I PALMER,...', Zenodo creators,
  -- OSTI 'Subhodeep Banerjee [NETL ...]\x3B Mehrdad Shahnam [NETL]\x3B ...', Turkish Studies 'Yilmaz MUTLU- Demet DENIZ - Semra POLAT'.
  -- Only one-entry bylines, so no later seat moves (work_authors binds ids by seat number, #1893).
  -- Rule (reference + tests: oxjob #583 work/split_rule_v3.py.txt): no organisation word outside brackets, then either
  -- a semicolon list of 2+ names ('Given Family' or 'Family, Given'), or a comma / ' and ' list of 3+ 'Given Family' names
  -- of 2-6 tokens, or (#1981) a list joined by a dash with a space after it, alone or with commas, of 2+ such names
  -- (a space before the dash only is a hyphenated given name: 'Anthony W. -C. Lun').
  -- Two-name comma strings stay whole ('Garcia Lopez, Maria Jose' is one person).
  -- A byline within 5 characters of the ingest cap (MAX_AUTHOR_NAME_LENGTH 500, landing pages 1000) whose last part
  -- isn't a name lost the rest of that name at ingest ('... Tauxe, Cameron Sigler [LANL]\x3B Orba'): that part is dropped.
  -- A '(...)' or '[...]' after a name is that seat's affiliation, unless it is only 'Place, ST' (a patent inventor's town).
  -- The entry's own affiliations, ORCID and legacy rows go
  -- to no seat: we can't tell whose they are (#726: a seat takes only what its own record lists at it).
  -- API sample 15 Nov 2026: ~0.24% of single-author works (~300K), Harvard Dataverse 19%, Zenodo 1%, no false split read
  -- (416 read)
  -- #1981 scan 16 Nov: ~870 dash lists in Turkish Studies journals, 0 in 8,000 random single-author works.
  -- Full-scale bar 16 Nov (Databricks, every byline, this SQL): 375,264 works split into 1,497,403 seats (repo 179K,
  -- DataCite 138K, Crossref 58K), 47 of 50 read all-right, 3 with one non-person seat beside the people ('Bellew, Frank\x3B
  -- Vanity Fair'), each of which was one non-person seat before. v3 keeps whole a 'Person\x3B Universidad X' string
  -- (non-English organisation words, ~3K works),
  -- reads an upper-case AND as a joiner, strips every leading title ('Assistant Professor. Dr.'), needs a space after a
  -- dash (~460 works), and keeps German organisation compounds ('Volkshochschule Braunschweig') and job titles whole.
  -- v4 (17 Nov, after night 2026-11-17's 368,530 splits): French / Italian / Dutch / Indonesian organisation words keep a
  -- string whole ('Ministère de l’Agriculture ..., Direction de la Conservation Foncière ...' had become 4 seats, W6927756631),
  -- and so does a suspended hyphen ('Koninklijke Academie voor Nederlandse Taal- en Letterkunde' had become 2 seats).
  -- v5 (18 Nov): in a semicolon list a spaced hyphen inside one name closes up ('Patchett, John M [LANL]\x3B Lo, Li - Ta [LANL]\x3B ...'
  -- stayed one seat because 'Li - Ta' failed the name test). Only parts that failed it change, so no earlier split's names move.
  -- Revert: delete combined_split_src, combined_split_lists, combined_split_trunc, combined_split_parts and
  -- exploded_authors, rename exploded_authors_src back to exploded_authors, drop the two `AND NOT e.is_split`.
  -- Name tokens use p{L} / p{M} / p{N} classes, never the U flag with w: on Databricks (Photon) w stayed ASCII, so 'Munoz Nieves'
  -- failed the name test and no byline holding an accented name split (found on the full-scale bar, 16 Nov).
  -- One pass, no join: each seat row explodes to itself, or (a split one-entry byline) to its names.
  combined_split_src AS (
      SELECT e.*,
          -- the candidate gate: one-entry byline, a list separator, no organisation word outside brackets
          (e.author_count = 1
           AND e.raw_author_name RLIKE '[,&\\x3B]|\\sand\\s|\\S\\s*[-\\x{2013}\\x{2014}]\\s+\\S'
           AND NOT COALESCE(REGEXP_REPLACE(e.raw_author_name, '\\s*[(\\[][^)\\]]*[)\\]]', '') RLIKE
               '(?iu)(\\x{00FC}niversite|universite|fak\\x{00FC}lte|enstit\\x{00FC}|hastane|bakanl\\x{0131}|derne[g\\x{011F}]i|vakf[\\x{0131}i]|m\\x{00FC}d\\x{00FC}rl\\x{00FC}|ba\\x{015F}kanl\\x{0131}|okulu|lisesi)|\\b(universit|institut|ministry|department|dept\\b|society|association|committee|group|cent(er|re)\\b|council|agency|laborator|hospital|foundation|inc\\b|ltd\\b|corporation|office|bureau|organi[sz]ation|network|consortium|project|team|survey|service|library|museum|archive|board|commission|federation|union|school|college|academy|company|gmbh|trust|program|faculty|division|unit\\b|workshop|conference|editor|staff|anonymous|et al)|\\b(universidad|facult|faculdad|facolt|istituto|centro\\b|escuela|escola\\b|[e\\x{00E9}]cole\\b|col[e\\x{00E9}]gio|departament|dipartiment|minist[e\\x{00E9}]rio|fundaci[o\\x{00F3}]n|funda[c\\x{00E7}][a\\x{00E3}]o|cl[i\\x{00ED}]nica\\b|secretar[i\\x{00ED}]a\\b|servicio|servi[c\\x{00E7}]o\\b|sede\\b|n[u\\x{00FA}]cleo|consejo|conselho|asociaci[o\\x{00F3}]n|associa[c\\x{00E7}][a\\x{00E3}]o|sociedad|empresa\\b|gobierno|governo\\b|prefeitura|municipalidad|ayuntamiento|polit[e\\x{00E9}]cnic|politecnic|hochschule|klinik|zentrum|stiftung|editora\\b|biblioteca|museo\\b|museu\\b|observatori|agencia|ag[e\\x{00EA}]ncia|cooperativa|programa\\b|grupo\\b|pontif[i\\x{00ED}]cia|imprimerie|scholar\\b|lecturer\\b|assistant professor|associate professor)|(hochschule|schule\\b|gemeinschaft|gesellschaft|verein\\b|verband\\b|anstalt\\b|bibliothek|ministerium)|\\b(minist[e\\x{00E8}\\x{00E9}]re|ministero|ministerie|kementerian|direktorat|lembaga\\b|d[e\\x{00E9}]partement\\b|direction\\b|soci[e\\x{00E9}]t[e\\x{00E9}]|agence\\b|conseil\\b|mairie\\b|pr[e\\x{00E9}]fecture|r[e\\x{00E9}]publique|syndicat|compagnie|banque\\b|caisse\\b|comit[e\\x{00E9}]|comitato|groupement|observatoire|biblioth[e\\x{00E8}]que|acad[e\\x{00E9}]mie\\b|servizio\\b|ufficio\\b|autorit[a\\x{00E0}]|azienda\\b|regione\\b|provincia\\b|provincie\\b|gemeente\\b|comune\\b|instituut|hogeschool|stichting|vereniging|genootschap|koninklijke)', FALSE)
           -- v4 (17 Nov): a suspended hyphen ('Taal- en Letterkunde', 'Lehr- und ...') is a compound word, not a dash list
           AND NOT COALESCE(e.raw_author_name RLIKE '(?i)\\p{L}-\\s+(and|en|und|oder|och|og|of|or|et|e|y)\\s', FALSE)) AS split_candidate,
          (CHAR_LENGTH(e.raw_author_name) BETWEEN 495 AND 500 OR CHAR_LENGTH(e.raw_author_name) BETWEEN 995 AND 1000) AS near_cap
      FROM exploded_authors_src e
  ),
  combined_split_lists AS (
      SELECT c.*,
          -- every part with its name test: ok = 'Given Family' of 2-6 tokens, or (semicolon lists) 'Family, Given'
          CASE WHEN c.split_candidate AND c.raw_author_name RLIKE '\\x3B' THEN TRANSFORM(FILTER(
              TRANSFORM(SPLIT(c.raw_author_name, '\\x3B'), p -> NAMED_STRUCT(
                  'name', TRIM(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(
                      -- v5 (18 Nov): a spaced hyphen inside one name closes up ('Lo, Li - Ta' -> 'Lo, Li-Ta', OSTI W4391684925, 'Kuhn, P. -H.',
                      -- 'Arpaci - Dusseau, Remzi'): only between two single words, the first opening the part or following its comma, the
                      -- second ending the part or followed by a comma, and never after 'Dipl.' / 'Dr.' ('Dipl. -Ing.' is a German title).
                      -- Not after two capitals either ('CONICET - IIGG', 'MBAL - Varna': acronyms). 'Person\x3B Institution - Place' and
                      -- 'Name, Given - Role' parts keep failing the name test, so those bylines stay whole.
                      REGEXP_REPLACE(REGEXP_REPLACE(p, '\\s*[(\\[][^)\\]]*[)\\]]', ''), '(?<=(?:^|,)\\s{0,3}\\p{L}[\\p{L}\\p{M}\\x27.]{0,30})(?<!\\b(?i:dipl|dr|chem|ing)\\.)(?<!\\p{Lu}\\p{Lu})\\s+-\\s*(?=\\p{L}[\\p{L}\\p{M}\\x27.]{0,30}\\s*(?:,|$))', '-'),
                      '[,\\s]+(MD|M\\.D\\.|PhD|Ph\\.D\\.|MPH|MS|MSc|M\\.Sc\\.|BSc|MA|MBA|RN|DO|PharmD|DDS|DMD|FACP|FACS|FRCP|FRCS|MBBS|BS|BA|Dr|Prof|Jr|Sr|II|III|IV)\\.?\\s*$', ''),
                      '\\s+', ' '),
                      '^[\\s,\\x3B]+|[\\s,\\x3B]+$', '')),
                  -- a bracket that holds only a place ('Blackfoot, ID', 'Oak Ridge, TN': OSTI patent inventors) is no affiliation
                  'affiliation', CASE WHEN TRIM(REGEXP_EXTRACT(p, '[(\\[]([^)\\]]*)[)\\]]', 1)) RLIKE '^[^,]+,\\s*[A-Z]{2}[0-9]?$' THEN CAST(NULL AS STRING)
                                  ELSE NULLIF(TRIM(REGEXP_EXTRACT(p, '[(\\[]([^)\\]]*)[)\\]]', 1)), '') END
              )),
              x -> x.name != ''),
              x -> NAMED_STRUCT('name', x.name, 'affiliation', x.affiliation, 'ok',
                  x.name RLIKE '(^|\\s)\\p{Lu}' AND (
                  x.name RLIKE '^\\p{L}[\\p{L}\\p{M}\\p{N}_\\x27\\x{2019}.\\-]*( \\p{L}[\\p{L}\\p{M}\\p{N}_\\x27\\x{2019}.\\-]*){1,5}$'
                  OR x.name RLIKE '^\\p{L}[\\p{L}\\p{M}\\p{N}_\\x27\\x{2019}.\\-]*( \\p{L}[\\p{L}\\p{M}\\p{N}_\\x27\\x{2019}.\\-]*){0,5},\\s*\\p{L}[\\p{L}\\p{M}\\p{N}_\\x27\\x{2019}.\\-]*( \\p{L}[\\p{L}\\p{M}\\p{N}_\\x27\\x{2019}.\\-]*){0,5}$')))
          WHEN c.split_candidate THEN TRANSFORM(FILTER(
              TRANSFORM(
                  SPLIT(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(c.raw_author_name, '\\s*[(\\[][^)\\]]*[)\\]]', ''),
                                       '\\s+(and|And|AND|&|y|und|et|e)\\s+', ','),
                                       '\\s*[-\\x{2013}\\x{2014}]\\s+', ','), ','),
                  p -> NAMED_STRUCT(
                      'name', TRIM(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(p,
                          '[,\\s]+(MD|M\\.D\\.|PhD|Ph\\.D\\.|MPH|MS|MSc|M\\.Sc\\.|BSc|MA|MBA|RN|DO|PharmD|DDS|DMD|FACP|FACS|FRCP|FRCS|MBBS|BS|BA|Dr|Prof|Jr|Sr|II|III|IV)\\.?\\s*$', ''),
                          '\\s+', ' '),
                          '^[\\s,\\x3B]+|[\\s,\\x3B]+$', '')),
                      'affiliation', CAST(NULL AS STRING))),
              x -> x.name != ''),
              x -> NAMED_STRUCT('name', x.name, 'affiliation', x.affiliation, 'ok',
                  x.name RLIKE '(^|\\s)\\p{Lu}'
                  AND x.name RLIKE '^\\p{L}[\\p{L}\\p{M}\\p{N}_\\x27\\x{2019}.\\-]*( \\p{L}[\\p{L}\\p{M}\\p{N}_\\x27\\x{2019}.\\-]*){1,5}$'))
          END AS list_parts,
          -- a semicolon or dash list needs 2 names, a comma list 3
          IF(c.raw_author_name RLIKE '\\x3B|\\S\\s*[-\\x{2013}\\x{2014}]\\s+\\S', 2, 3) AS min_parts
      FROM combined_split_src c
  ),
  combined_split_trunc AS (
      SELECT l.*,
          -- a name cut at the ingest cap: drop the byline's last part when it isn't a name
          IF(l.near_cap AND SIZE(l.list_parts) >= 1 AND NOT COALESCE(TRY_ELEMENT_AT(l.list_parts, -1).ok, TRUE),
             SLICE(l.list_parts, 1, SIZE(l.list_parts) - 1), l.list_parts) AS kept_parts
      FROM combined_split_lists l
  ),
  combined_split_parts AS (
      SELECT t.*,
          -- every part must still hold a name once its titles are stripped ('Assistant Professor' is a role, not a person)
          IF(SIZE(t.kept_parts) >= t.min_parts AND FORALL(t.kept_parts, x -> x.ok
                 AND REGEXP_REPLACE(x.name, '(?i)^((and|dr|mr|mrs|ms|prof|professor|profesor|assistant|associate)(\\.\\s*|\\s+))+', '') != ''),
             -- a leading title is not part of the name ('Dr. Bilal Mukhtar', 'Ms. Riya K Prakash')
             TRANSFORM(t.kept_parts, x -> NAMED_STRUCT('name', REGEXP_REPLACE(x.name, '(?i)^((and|dr|mr|mrs|ms|prof|professor|profesor|assistant|associate)(\\.\\s*|\\s+))+', ''),
                                                       'affiliation', x.affiliation)),
             NULL) AS split_parts
      FROM combined_split_trunc t
  ),
  exploded_authors AS (
      SELECT
          s.work_id,
          IF(s.split_parts IS NULL, s.author_count, SIZE(s.split_parts)) AS author_count,
          IF(s.split_parts IS NULL, s.is_corresponding_exists, FALSE) AS is_corresponding_exists,
          IF(s.split_parts IS NULL, s.original_author_order, x.pos) AS original_author_order,
          IF(s.split_parts IS NULL, s.raw_author_name, x.part.name) AS raw_author_name,
          IF(s.split_parts IS NULL, s.orcid, CAST(NULL AS STRING)) AS orcid,
          IF(s.split_parts IS NULL, s.raw_affiliation_strings,
             IF(x.part.affiliation IS NULL, CAST(ARRAY() AS ARRAY<STRING>), ARRAY(x.part.affiliation))) AS raw_affiliation_strings,
          IF(s.split_parts IS NULL, s.is_corresponding_from_source, CAST(NULL AS BOOLEAN)) AS is_corresponding_from_source,
          s.split_parts IS NOT NULL AS is_split,
          s.byline_provenance, s.byline_native_id  -- oxjob #2028
      FROM combined_split_parts s
      LATERAL VIEW POSEXPLODE(COALESCE(s.split_parts,
          ARRAY(NAMED_STRUCT('name', CAST(NULL AS STRING), 'affiliation', CAST(NULL AS STRING))))) x AS pos, part
  ),
  -- Legacy affiliations from works_legacy.raw_affiliation_strings table
  -- oxjob #801: this table is the SECOND input to authorships[].raw_affiliation_strings
  -- (see the CONCAT in authors_with_corresponding below). The map-time mojibake repair in
  -- CreateLocationsMapped does NOT reach it, so without this CTE the legacy copy re-adds the
  -- mojibake variant one step underneath the fix -- measured 2026-08-17, that left 102,495 of
  -- the 237,229 affected works still broken. Repaired at READ time because this table is a
  -- frozen 850M-row / 44GB snapshot (last write 2026-01-07); rewriting it would be a far
  -- larger and riskier operation for the same result.
  --
  -- The RLIKE gate below is byte-identical to the one in CreateLocationsMapped and MUST STAY
  -- THAT WAY. Raw affiliation strings are exact-match keys (ras_curations,
  -- affiliation_strings_lookup), so if the two inputs disagree about which strings get
  -- repaired, a work ends up carrying both variants and a curation keyed on the repaired
  -- string silently applies to only one of them.
  -- oxjob #2107: decode HTML entities in the legacy copy too, before the mojibake repair, exactly as CreateLocationsMapped
  -- does for every provenance: 718,005 legacy works carry an IEEE-style escaped string (Technische Universit + a hex
  -- reference + t), and without this step a work would show both spellings (the CONCAT below). Gate in a WHERE, UDF in
  -- the projection above it (never inside a CASE: Spark hoists Python UDFs out of CASE and runs them on every row).
  -- The gate is byte-identical to CreateLocationsMapped's and uses \\x3B, never a literal semicolon.
  -- Revert: delete this CTE and read openalex.works_legacy.raw_affiliation_strings in legacy_affiliations_repaired again.
  legacy_affiliations_decoded AS (
      SELECT work_id, author_sequence, raw_affiliation_string
      FROM openalex.works_legacy.raw_affiliation_strings
      WHERE raw_affiliation_string IS NOT NULL
        AND NOT COALESCE(raw_affiliation_string RLIKE '&(#[0-9]{1,7}|#[xX][0-9a-fA-F]{1,6}|[A-Za-z][A-Za-z0-9]{1,31})\\x3B', FALSE)
      UNION ALL
      SELECT work_id, author_sequence, openalex.common.decode_affiliation_entities(raw_affiliation_string) AS raw_affiliation_string
      FROM openalex.works_legacy.raw_affiliation_strings
      WHERE raw_affiliation_string RLIKE '&(#[0-9]{1,7}|#[xX][0-9a-fA-F]{1,6}|[A-Za-z][A-Za-z0-9]{1,31})\\x3B'
  ),
  legacy_affiliations_repaired AS (
      SELECT
          work_id,
          author_sequence,
          CASE
              WHEN raw_affiliation_string RLIKE '[\\u00C2-\\u00DF\\u00E0-\\u00EF][\\u0080-\\u00BF\\u20AC\\u201A\\u0192\\u201E\\u2026\\u2020\\u2021\\u02C6\\u2030\\u0160\\u2039\\u0152\\u017D\\u2018\\u2019\\u201C\\u201D\\u2022\\u2013\\u2014\\u02DC\\u2122\\u0161\\u203A\\u0153\\u017E\\u0178]|[\\u0102\\u0139]'
              THEN openalex.common.repair_mojibake(raw_affiliation_string)
              ELSE raw_affiliation_string
          END AS raw_affiliation_string
      FROM legacy_affiliations_decoded
      WHERE raw_affiliation_string IS NOT NULL
  ),
  legacy_affiliations AS (
      SELECT
          work_id,
          author_sequence,
          -- ARRAY_SORT: legacy table has no ordinal; unordered COLLECT_LIST reshuffles per
          -- scan and churns the enriched content hash nightly (wship full rebuild, oxjob #660)
          ARRAY_SORT(COLLECT_LIST(raw_affiliation_string)) AS raw_affiliation_strings
      FROM (
          -- oxjob #1594: a work whose Crossref initials-keyed block CreateCrossrefSuperAuthorships paired takes its legacy
          -- rows from openalex.works.legacy_initials_keyed_split (cell above: the old system's copy of that block paired
          -- the same way, leftover copies dropped, each seat's own strings kept, decoded and repaired as above). Every
          -- other work reads the frozen table as before. ~103K work ids, broadcast. Here, after the repair, so this
          -- statement calls no more Python functions (a SQL warehouse allows 5 per statement).
          -- Revert: read FROM legacy_affiliations_repaired again (drop this subquery) and delete the cell above.
          SELECT /*+ BROADCAST(k) */ r.work_id, r.author_sequence, r.raw_affiliation_string
          FROM legacy_affiliations_repaired r
          LEFT ANTI JOIN openalex.works.legacy_initials_keyed_split k ON r.work_id = k.work_id
          UNION ALL
          SELECT k.work_id, x.author_sequence, x.raw_affiliation_string
          FROM openalex.works.legacy_initials_keyed_split k
          LATERAL VIEW EXPLODE(k.legacy) t AS x
          UNION ALL
          -- oxjob #2378: a dissertation's degree-granting institution (repository feeds #2377, thesis parser #2367) as a seat
          -- string, one row per string in openalex.works.thesis_grantor_affiliations (provenance, version). Rows are written
          -- only for seats of works that named no institution when built (fill-only at build time), already in the served
          -- form (decoded, single spaces, no trailing '.'), so they join here like the frozen per-seat strings and are
          -- appended to the seat's own. Revert: delete this branch, or DELETE a version's rows and run oxjob #2378's
          -- work/revert.sql (work_authors keeps added strings otherwise).
          SELECT g.work_id, g.author_sequence, g.raw_affiliation_string
          FROM openalex.works.thesis_grantor_affiliations g
      )
      GROUP BY work_id, author_sequence
  ),
  -- oxjob #2275: a work whose legacy rows flag EVERY one of 3 or more seats corresponding carries no legacy evidence (the
  -- old parser's guess, e.g. an e-mail shown for every author). ~202K visible works showed every author as corresponding
  -- from it while their pages name one: of 28 seeded pages, the 24 that name anyone all name a subset (2-seat rows stay:
  -- 2 of 2 co-corresponding is common). Their legacy rows are left out here, so they show no one unless a source or PMC
  -- flags someone (CreateCrossrefSuperAuthorships drops the same works from its PMC block). Frozen table, so the set is
  -- fixed (~670K work ids, broadcast). Revert: delete legacy_all_corresponding_works and the anti join below.
  legacy_all_corresponding_works AS (
      SELECT work_id
      FROM openalex.works_legacy.work_authors
      GROUP BY work_id
      HAVING COUNT(*) >= 3 AND COUNT_IF(is_corresponding) = COUNT(*)
  ),
  -- Legacy work_authors for is_corresponding values
  legacy_work_authors AS (
      SELECT /*+ BROADCAST(x) */
          lwa.work_id,
          lwa.author_sequence,
          lwa.is_corresponding AS legacy_is_corresponding
      FROM openalex.works_legacy.work_authors lwa
      LEFT ANTI JOIN legacy_all_corresponding_works x ON x.work_id = lwa.work_id
  ),
  authors_with_corresponding AS (
      SELECT /*+ BROADCAST(cas) */
          e.work_id,
          e.original_author_order,
          e.author_count,
          e.raw_author_name,
          e.orcid,
          e.is_corresponding_from_source,
          e.is_corresponding_exists,
          -- Combine source and legacy affiliations, clean up, deduplicate, and filter out garbage
          ARRAY_DISTINCT(
              FILTER(
                  TRANSFORM(
                      ARRAY_COMPACT(
                          -- oxjob #1569: a site curator's list for this seat (users-api raw_affiliation_strings["<served
                          -- name>"]) takes the place of what the sources give; it is cleaned and filtered like them.
                          -- Revert: delete this CASE (keep the CONCAT) and the cas join below.
                          CASE WHEN cas.raw_affiliation_strings IS NOT NULL THEN cas.raw_affiliation_strings
                          ELSE CONCAT(
                              COALESCE(e.raw_affiliation_strings, ARRAY()),
                              -- oxjob #1242: a legacy string the source now carries with its Crossref place appended (same seat) is the same
                              -- affiliation; keep only the source's fuller string. Revert: put back COALESCE(l.raw_affiliation_strings, ARRAY()).
                              -- oxjob #2028: nor the legacy copy of a collaboration-group member's one-for-everyone parse
                              -- affiliation (gm below; the same letters-only test as CreateCrossrefSuperAuthorships).
                              COALESCE(FILTER(l.raw_affiliation_strings, ls -> NOT EXISTS(COALESCE(e.raw_affiliation_strings, ARRAY()),
                                  s -> s <> ls AND LEFT(s, LENGTH(REGEXP_REPLACE(ls, '[\\s,;]+$', '')) + 2)
                                                   = CONCAT(REGEXP_REPLACE(ls, '[\\s,;]+$', ''), ', '))
                                  AND (gm.doi IS NULL
                                       OR NOT ARRAY_CONTAINS(gm.broadcast_letters, LOWER(REGEXP_REPLACE(ls, '[^\\p{L}]', ''))))), ARRAY())
                          ) END
                      ),
                      s -> TRIM(TRAILING '.' FROM TRIM(REPLACE(s, '\\n', '')))
                  ),
                  -- oxjob #726: a string with no letter names nothing ('', ',', '-', '?', page, phone and postcode digits):
                  -- 1,119,745 works carried one on 1,977,257 seats (15 Nov 2026), 148 of them matched to an institution.
                  -- Revert: delete this comment and the \p{L} condition below.
                  s -> LENGTH(s) <= 2000 AND NOT s RLIKE '<(div|span|option|select|textarea|td|tr)' AND s RLIKE '\\p{L}'
                       -- oxjob #1063: a string that starts with a library catalogue notice is a book's front matter, not an affiliation:
                       -- the German National Library's CIP notice ('Bibliografische Information der Deutschen Nationalbibliothek ...
                       -- verzeichnet diese Publikation', 'Die Deutsche Bibliothek - CIP-Einheitsaufnahme', the English 'Bibliographic
                       -- information published by ...') and the Library of Congress / British Library cataloguing-in-publication heading.
                       -- The matcher links them to the Deutsche Nationalbibliothek (I4210110517) and the Library of Congress, and string
                       -- curations don't hold: each new copy of the notice is matched again (ZD 8515). Only strings that START with the
                       -- notice: one that follows a real affiliation ('Universitaet Bremen Bibliografische Information ...') is kept.
                       -- Measured 18 Nov 2026: 966 strings on 5,910 works; 755 works lose an institution link (DNB 729, LoC 21, 5 others),
                       -- 338 works lose their last string. Revert: delete these #1063 lines and the condition below.
                       AND NOT s RLIKE '(?i)^\\W{0,3}(bibliogra\\S{0,8}\\s?\\S{0,8}\\s*informatione?n?\\s*(der|published\\s*by\\s*(the|die)?)\\s*deutschen?|(die\\s*|the\\s*)?deutschen?\\s*(national)?bibliothek\\s*(die\\s*deutsche\\s*(national)?bibliothek\\s*)?(verzeichnet|lists\\s*this)|(der\\s*)?deutschen?\\s*national-?\\s*bibliogra(f|ph)|(die\\s*)?deutsche\\s*bibliothek\\s*-?\\s*cip|cip-?\\s*(einheits|kurztitel|titel)aufnahme|(library\\s*of\\s*congress|british\\s*library)\\s*catalogu?ing[\\s-]*in\\b|a\\s*catalogue\\s*record\\s*for\\s*this\\s*(book|title|publication)\\s*is\\s*available)'
              )
          ) AS raw_affiliation_strings,
          -- Legacy is_corresponding value (NULL if no legacy record exists)
          lw.legacy_is_corresponding,
          -- Check if legacy has any corresponding author for this work
          MAX(CASE WHEN lw.legacy_is_corresponding = TRUE THEN 1 ELSE 0 END)
              OVER (PARTITION BY e.work_id) AS legacy_has_corresponding,
          CASE
              WHEN e.is_corresponding_exists THEN
                  MAX(CASE WHEN e.is_corresponding_from_source = TRUE THEN 1 ELSE 0 END)
                      OVER (PARTITION BY e.work_id)
              ELSE 0
          END AS work_has_corresponding
      FROM exploded_authors e
      LEFT JOIN legacy_affiliations l
          ON e.work_id = l.work_id
          AND e.original_author_order = l.author_sequence
          AND NOT e.is_split  -- oxjob #583: a split seat takes no legacy row of the combined seat
      -- oxjob #2028: collaboration-group member seats (CreateCrossrefSuperAuthorships), one row per seat, ~87K rows
      LEFT JOIN openalex.works.group_member_seats gm
          ON e.byline_provenance = 'crossref'
          AND gm.doi = e.byline_native_id
          AND gm.author_position = e.original_author_order
          AND NOT e.is_split
      LEFT JOIN legacy_work_authors lw
          ON e.work_id = lw.work_id
          AND e.original_author_order = lw.author_sequence
          AND NOT e.is_split  -- oxjob #583
      -- oxjob #1569: seat affiliation curations (SyncWorkFieldCurations), keyed on the name as raw_authorships serves it
      LEFT JOIN openalex.works.work_curated_affiliation_strings cas
          ON e.work_id = cas.work_id
          AND TRIM(REPLACE(e.raw_author_name, '\\n', '')) = cas.raw_author_name
    WHERE e.raw_author_name IS NOT NULL AND TRIM(e.raw_author_name) != ''
  ),
  raw_authorships AS (
      SELECT
          work_id,
          TRANSFORM(
              ARRAY_SORT(
                  COLLECT_LIST(
                      STRUCT(
                          original_author_order,
                          STRUCT(
                              CAST(ARRAY() AS ARRAY<STRUCT<institution_ids: ARRAY<STRING>, raw_affiliation_string: STRING>>) AS affiliations,
                              NAMED_STRUCT(
                                  'display_name', TRIM(REPLACE(raw_author_name, '\\n', '')),
                                  'id', CAST(NULL AS STRING),
                                  'orcid', CAST(NULL AS STRING),
                                  -- every ORCID observed on the resolved profile, primary first;
                                  -- filled by CreateWorkAuthorships (oxjob #1340)
                                  'observed_orcids', CAST(ARRAY() AS ARRAY<STRING>),
                                  -- oxjob #2437: the seat's author type ('human' | 'organization' | 'software-agent'), NULL =
                                  -- not stated; filled by CreateWorkAuthorships from openalex.works.work_author_types. Seeded
                                  -- here so the build's authorships carry the field when work_authorships is merged in.
                                  'type', CAST(NULL AS STRING)
                              ) AS author,
                              CASE
                                  WHEN original_author_order = 0 THEN 'first'
                                  WHEN original_author_order + 1 = author_count THEN 'last'
                                  ELSE 'middle'
                              END AS author_position,
                              original_author_order AS author_order_number,
                              CAST(ARRAY() AS ARRAY<STRING>) AS countries,
                              CAST(ARRAY() AS ARRAY<STRUCT<country_code: STRING, display_name: STRING, id: STRING, lineage: ARRAY<STRING>, ror: STRING, type: STRING>>) AS institutions,
                              CASE
                                  -- Source says this author is corresponding
                                  WHEN is_corresponding_from_source = TRUE THEN TRUE
                                  -- Source has corresponding info and someone else is corresponding
                                  WHEN work_has_corresponding = 1 THEN FALSE
                                  -- No source info, use legacy is_corresponding if available
                                  WHEN legacy_is_corresponding = TRUE THEN TRUE
                                  -- Legacy has corresponding info and someone else is corresponding
                                  WHEN legacy_has_corresponding = 1 THEN FALSE
                                  -- No source/legacy evidence: only a sole author defaults to corresponding (oxjob #401)
                                  WHEN author_count = 1 THEN TRUE
                                  ELSE FALSE
                              END AS is_corresponding,
                              -- oxjob #1063: a copyright line ('Copyright of Chulalongkorn University', '© Pharmacotherapy Group, Faculty of Pharmacy,
                              -- University of Benin', '©1992 Brown University') is the journal's or publisher's notice, not the author's affiliation,
                              -- and the matcher links it to the holder. Drop a string that starts with '©' or 'Copyright' when the same seat keeps
                              -- another string; a seat with nothing else keeps it (a thesis or an institution's own report: the holder is often the
                              -- author's institution, 'Copyright by Mahidol University Fac. of Grad. Studies'). '(c)' is an affiliation label, not
                              -- copyright. Measured 18 Nov 2026: 17,009 works change; 6,869 lose a link (7,166 links: CNRS 1,294, Nigeria 438, ASEE
                              -- 312, ...); 40/40 sampled lost links were the holder, not the author's institution. Revert: put back the bare
                              -- raw_affiliation_strings line and delete these #1063 lines.
                              FILTER(raw_affiliation_strings, s -> NOT s RLIKE '(?i)^\\W{0,3}(\\x{00A9}|copyright\\b)'
                                  OR FORALL(raw_affiliation_strings, o -> o RLIKE '(?i)^\\W{0,3}(\\x{00A9}|copyright\\b)')) AS raw_affiliation_strings,
                              TRIM(REPLACE(raw_author_name, '\\n', '')) AS raw_author_name,
                              CASE
                                  WHEN orcid IS NULL OR orcid = '' THEN NULL
                                  WHEN orcid LIKE 'https://orcid.org/____-____-____-____' THEN orcid
                                  WHEN orcid RLIKE '^[0-9]{4}-[0-9]{4}-[0-9]{4}-[0-9]{3}[0-9Xx]$' THEN CONCAT('https://orcid.org/', orcid)
                                  ELSE NULL
                              END AS raw_orcid
                          ) AS authorship
                      )
                  ),
                  (left, right) -> CASE
                      WHEN left.original_author_order < right.original_author_order THEN -1
                      WHEN left.original_author_order > right.original_author_order THEN 1
                      ELSE 0
                  END
              ),
              x -> x.authorship
          ) AS authorships
      FROM authors_with_corresponding
      GROUP BY work_id
  ),
  -- oxjob #1815: a curated byline fills a work whose records name no author (users-api POST /curations: entity works,
  -- property byline, value a JSON list [{"name": ..., "orcid": ...}] as printed on the work's own page or PDF; site
  -- curators only; SyncWorkFieldCurations). Fill-only (#1164): set_fields takes it only where raw_authorships has no row,
  -- so a record that names anyone wins. One row per curated work (a few rows). Seats get authors through MatchAuthors'
  -- size_grew rematch; claims bind them (ApplyWorkAuthorCurations). The struct mirrors raw_authorships' field for field.
  -- Revert: delete this CTE, its LEFT JOIN and the COALESCE on the authorships line in set_fields.
  raw_authorships_curated AS (
      SELECT
          work_id,
          TRANSFORM(
              ARRAY_SORT(
                  COLLECT_LIST(
                      STRUCT(
                          original_author_order,
                          STRUCT(
                              CAST(ARRAY() AS ARRAY<STRUCT<institution_ids: ARRAY<STRING>, raw_affiliation_string: STRING>>) AS affiliations,
                              NAMED_STRUCT(
                                  'display_name', raw_author_name,
                                  'id', CAST(NULL AS STRING),
                                  'orcid', CAST(NULL AS STRING),
                                  'observed_orcids', CAST(ARRAY() AS ARRAY<STRING>),
                                  'type', CAST(NULL AS STRING) -- oxjob #2437, as raw_authorships
                              ) AS author,
                              CASE
                                  WHEN original_author_order = 0 THEN 'first'
                                  WHEN original_author_order + 1 = author_count THEN 'last'
                                  ELSE 'middle'
                              END AS author_position,
                              original_author_order AS author_order_number,
                              CAST(ARRAY() AS ARRAY<STRING>) AS countries,
                              CAST(ARRAY() AS ARRAY<STRUCT<country_code: STRING, display_name: STRING, id: STRING, lineage: ARRAY<STRING>, ror: STRING, type: STRING>>) AS institutions,
                              author_count = 1 AND NOT editor_fill AS is_corresponding, -- as raw_authorships with no source evidence: only a sole author (#401); never an editor (#2075)
                              curated_affiliations AS raw_affiliation_strings, -- oxjob #2133: the seat's #1569 curation, if any
                              raw_author_name,
                              CASE
                                  WHEN orcid RLIKE '^[0-9]{4}-[0-9]{4}-[0-9]{4}-[0-9]{3}[0-9Xx]$' THEN CONCAT('https://orcid.org/', orcid)
                                  ELSE NULL
                              END AS raw_orcid
                          ) AS authorship
                      )
                  ),
                  (left, right) -> CASE
                      WHEN left.original_author_order < right.original_author_order THEN -1
                      WHEN left.original_author_order > right.original_author_order THEN 1
                      ELSE 0
                  END
              ),
              x -> x.authorship
          ) AS curated_authorships,
          BOOL_OR(editor_fill) AS editor_fill -- oxjob #2075 D11: the byline is a book's Crossref editors
      FROM (
          -- oxjob #2133: a curated seat takes its raw_affiliation_strings["<name>"] curation (#1569's view, keyed by
          -- work and served name), cleaned as raw_authorships cleans the sources' strings. Before, #1569 reached record
          -- seats only, so an added or filled seat could never show the affiliation printed for it (ZD 24778).
          -- Revert: delete this wrapper and its join (keep the inner SELECT) and put back an empty array above.
          SELECT p.*,
                 ARRAY_DISTINCT(FILTER(
                     TRANSFORM(COALESCE(cas.raw_affiliation_strings, CAST(ARRAY() AS ARRAY<STRING>)),
                               s -> TRIM(TRAILING '.' FROM TRIM(REPLACE(s, '\\n', '')))),
                     s -> LENGTH(s) <= 2000 AND s RLIKE '\\p{L}')) AS curated_affiliations
          FROM (
              SELECT c.work_id, t.original_author_order, SIZE(c.authors) AS author_count,
                     TRIM(REPLACE(t.a.name, '\\n', '')) AS raw_author_name, t.a.orcid AS orcid, c.editor_fill
              FROM (
                  SELECT work_id, FROM_JSON(byline, 'ARRAY<STRUCT<name: STRING, orcid: STRING>>') AS authors, FALSE AS editor_fill
                  FROM openalex.works.work_curated_fields
                  WHERE byline IS NOT NULL
                  UNION ALL
                  -- oxjob #2075 D11: a book whose Crossref record lists only editors, and that no record names anyone on,
                  -- takes its editors as the byline (openalex.works.crossref_book_contributors, built in a cell above; a
                  -- curated byline wins). CreateWorksEnriched gives these seats the contributor role editor.
                  SELECT e.work_id, TRANSFORM(e.people, x -> NAMED_STRUCT('name', x.name, 'orcid', x.orcid)) AS authors,
                         TRUE AS editor_fill
                  FROM openalex.works.crossref_book_contributors e
                  LEFT ANTI JOIN (SELECT work_id FROM openalex.works.work_curated_fields WHERE byline IS NOT NULL) cb
                      ON cb.work_id = e.work_id
                  WHERE e.role = 'editor' AND e.fill
              ) c
              LATERAL VIEW POSEXPLODE(c.authors) t AS original_author_order, a
          ) p
          LEFT JOIN openalex.works.work_curated_affiliation_strings cas
              ON cas.work_id = p.work_id AND cas.raw_author_name = p.raw_author_name
      )
      WHERE raw_author_name IS NOT NULL AND raw_author_name <> ''
      GROUP BY work_id
  ),
  -- oxjob #2133 (D26, Kyle 18 Nov 2026): a curated byline (users-api works property `byline`, the byline printed on the
  -- work's own PDF or page, read by a site curator) also ADDS the authors a record's byline leaves out. "When the
  -- publisher sent no data (missing authors), we fix it ourselves; when it sent data and it's wrong, the publisher fixes
  -- it." So every seat the record gives stays, with its affiliations and ORCID; the curated list only adds seats around
  -- them, in printed order. Walking the curated names in order against the record's seats:
  --   - a name that matches the record's next seat takes that seat (same letters; or 'Family, Given' read as 'Given
  --     Family'; or the same last word and first initial: 'J. Grobler' / 'J.D. Grobler');
  --   - names whose letters join up to the record's next seat split it, one seat per person (#583's rule: 'Janna
  --     Hastings, Egon Willighagen' is two people, and so is 'Egon Willighagen and Michel Ballings': the letters
  --     key drops a standalone 'and', ZD 24482, CRAN record W4399586924);
  --   - any other name is a new seat (the printed author the record leaves out).
  -- It applies only when every record seat is taken or split in order, so a curated list can never rename, drop or
  -- reorder a person the record names. One exception: a record seat that is an organisation (university, institute,
  -- ministry, ...) and that the printed byline does not show is left out (Crossref 10.70634/reid.v2i13.174 names the
  -- university as sole author; the PDF prints two people). Works whose records name no one keep #1815's fill above.
  -- UpdateWorkAuthors moves the record seats' work_authors rows to their new seat numbers; MatchAuthors' size_grew
  -- rematch seats the new names (byline works first) and claims bind them. A few works change (one per curated work).
  -- One pass over raw_authorships (set_fields reads this CTE in its place, under its name): a second reference
  -- re-ran the whole authors aggregation (EXPLAIN, 20 Nov). Rows with no curated byline pass through unchanged.
  -- Revert: delete this CTE and point set_fields' LEFT JOIN back at raw_authorships.
  raw_authorships_extended AS (
    SELECT work_id,
           CASE WHEN ext IS NOT NULL AND NOT (TRANSFORM(ext, a -> a.raw_author_name) <=> TRANSFORM(rec_all, a -> a.raw_author_name))
                THEN ext ELSE rec_all END AS authorships
    FROM (
      SELECT work_id, rec_all,
             -- every record seat taken or split, none left half split
             CASE WHEN cur IS NOT NULL AND walk.j = SIZE(rec) + 1 AND walk.buf = '' THEN TRANSFORM(cur, (c, i) -> CASE WHEN ELEMENT_AT(walk.m, i + 1) > 0 THEN
                 STRUCT(
                     TRY_ELEMENT_AT(rec, GREATEST(ELEMENT_AT(walk.m, i + 1), 1)).affiliations AS affiliations,
                     TRY_ELEMENT_AT(rec, GREATEST(ELEMENT_AT(walk.m, i + 1), 1)).author AS author,
                     c.author_position AS author_position,
                     c.author_order_number AS author_order_number,
                     TRY_ELEMENT_AT(rec, GREATEST(ELEMENT_AT(walk.m, i + 1), 1)).countries AS countries,
                     TRY_ELEMENT_AT(rec, GREATEST(ELEMENT_AT(walk.m, i + 1), 1)).institutions AS institutions,
                     TRY_ELEMENT_AT(rec, GREATEST(ELEMENT_AT(walk.m, i + 1), 1)).is_corresponding AS is_corresponding,
                     TRY_ELEMENT_AT(rec, GREATEST(ELEMENT_AT(walk.m, i + 1), 1)).raw_affiliation_strings AS raw_affiliation_strings,
                     TRY_ELEMENT_AT(rec, GREATEST(ELEMENT_AT(walk.m, i + 1), 1)).raw_author_name AS raw_author_name,
                     TRY_ELEMENT_AT(rec, GREATEST(ELEMENT_AT(walk.m, i + 1), 1)).raw_orcid AS raw_orcid)
                 ELSE c END) END AS ext
      FROM (
          SELECT work_id, cur, rec, rec_all,
                 CASE WHEN cur IS NOT NULL THEN AGGREGATE(
                     SEQUENCE(1, SIZE(cur)),
                     NAMED_STRUCT('j', 1, 'buf', '', 'm', CAST(ARRAY() AS ARRAY<INT>)),
                     (acc, i) -> CASE
                         -- the record's next seat: same person
                         WHEN acc.buf = '' AND acc.j <= SIZE(rk) AND (
                                  ELEMENT_AT(ck, i).l = TRY_ELEMENT_AT(rk, acc.j).l
                               OR ARRAY_JOIN(ELEMENT_AT(ck, i).t, ' ') = ARRAY_JOIN(TRY_ELEMENT_AT(rk, acc.j).t, ' ')
                               OR (SIZE(ELEMENT_AT(ck, i).t) >= 2 AND SIZE(TRY_ELEMENT_AT(rk, acc.j).t) >= 2
                                   AND LENGTH(TRY_ELEMENT_AT(ELEMENT_AT(ck, i).t, -1)) >= 2
                                   AND TRY_ELEMENT_AT(ELEMENT_AT(ck, i).t, -1) = TRY_ELEMENT_AT(TRY_ELEMENT_AT(rk, acc.j).t, -1)
                                   AND LEFT(TRY_ELEMENT_AT(ELEMENT_AT(ck, i).t, 1), 1) = LEFT(TRY_ELEMENT_AT(TRY_ELEMENT_AT(rk, acc.j).t, 1), 1)))
                             THEN NAMED_STRUCT('j', acc.j + 1, 'buf', '', 'm', CONCAT(acc.m, ARRAY(acc.j)))
                         -- the last name of a split seat
                         WHEN acc.buf <> '' AND acc.j <= SIZE(rk) AND CONCAT(acc.buf, ELEMENT_AT(ck, i).l) = TRY_ELEMENT_AT(rk, acc.j).l
                             THEN NAMED_STRUCT('j', acc.j + 1, 'buf', '', 'm', CONCAT(acc.m, ARRAY(0)))
                         -- a name that starts (or continues) a split seat
                         WHEN acc.j <= SIZE(rk) AND ELEMENT_AT(ck, i).l <> ''
                              AND LENGTH(CONCAT(acc.buf, ELEMENT_AT(ck, i).l)) < LENGTH(TRY_ELEMENT_AT(rk, acc.j).l)
                              AND STARTSWITH(TRY_ELEMENT_AT(rk, acc.j).l, CONCAT(acc.buf, ELEMENT_AT(ck, i).l))
                             THEN NAMED_STRUCT('j', acc.j, 'buf', CONCAT(acc.buf, ELEMENT_AT(ck, i).l), 'm', CONCAT(acc.m, ARRAY(0)))
                         -- a printed author the record leaves out
                         ELSE NAMED_STRUCT('j', acc.j, 'buf', acc.buf, 'm', CONCAT(acc.m, ARRAY(0)))
                     END) END AS walk
          FROM (
              SELECT work_id, cur, rec, rec_all,
                     TRANSFORM(cur, a -> NAMED_STRUCT(
                         'l', REGEXP_REPLACE(TRANSLATE(REGEXP_REPLACE(LOWER(a.raw_author_name), '\\s+and\\s+', ' '), 'áàäâãåéèëêíìïîóòöôõøúùüûñçýćčšžłőűăşţșțđ', 'aaaaaaeeeeiiiioooooouuuuncyccszlouaststd'), '[^\\p{L}\\p{M}]', ''),
                         't', FILTER(SPLIT(TRIM(REGEXP_REPLACE(TRANSLATE(LOWER(CASE WHEN a.raw_author_name RLIKE '^[^,]+,[^,]+$' THEN CONCAT(TRIM(SUBSTRING_INDEX(a.raw_author_name, ',', -1)), ' ', TRIM(SUBSTRING_INDEX(a.raw_author_name, ',', 1))) ELSE a.raw_author_name END), 'áàäâãåéèëêíìïîóòöôõøúùüûñçýćčšžłőűăşţșțđ', 'aaaaaaeeeeiiiioooooouuuuncyccszlouaststd'), '[^\\p{L}\\p{M}]+', ' ')), ' '), t -> t <> ''))) AS ck,
                     TRANSFORM(rec, a -> NAMED_STRUCT(
                         'l', REGEXP_REPLACE(TRANSLATE(REGEXP_REPLACE(LOWER(a.raw_author_name), '\\s+and\\s+', ' '), 'áàäâãåéèëêíìïîóòöôõøúùüûñçýćčšžłőűăşţșțđ', 'aaaaaaeeeeiiiioooooouuuuncyccszlouaststd'), '[^\\p{L}\\p{M}]', ''),
                         't', FILTER(SPLIT(TRIM(REGEXP_REPLACE(TRANSLATE(LOWER(CASE WHEN a.raw_author_name RLIKE '^[^,]+,[^,]+$' THEN CONCAT(TRIM(SUBSTRING_INDEX(a.raw_author_name, ',', -1)), ' ', TRIM(SUBSTRING_INDEX(a.raw_author_name, ',', 1))) ELSE a.raw_author_name END), 'áàäâãåéèëêíìïîóòöôõøúùüûñçýćčšžłőűăşţșțđ', 'aaaaaaeeeeiiiioooooouuuuncyccszlouaststd'), '[^\\p{L}\\p{M}]+', ' ')), ' '), t -> t <> ''))) AS rk
              FROM (
                  SELECT /*+ BROADCAST(rac) */ r.work_id, rac.curated_authorships AS cur, r.authorships AS rec_all,
                         -- an organisation seat the printed byline doesn't show is left out
                         -- (oxjob #2133, 23 Nov: Spanish/Portuguese 'Corporación'/'Corporação', ZD 3116's Corpoica seat)
                         CASE WHEN rac.work_id IS NOT NULL THEN FILTER(r.authorships, a -> NOT (
                             COALESCE(a.raw_author_name RLIKE '(?iu)\\b(universi|institu|ministr|departament|department|facult|college|colegio|school|escola|escuela|hospital|centre\\b|center\\b|centro\\b|societ|sociedad|associa|asocia|council|conselho|foundation|funda[cç][aã]o|fundaci[oó]n|agency|company|corporation|corporaci[oó]n|corpora[cç][aã]o|laborator|ltd\\b|inc\\b|gmbh\\b)', FALSE)
                             AND NOT EXISTS(rac.curated_authorships, c ->
                                 LOWER(REGEXP_REPLACE(c.raw_author_name, '[^\\p{L}\\p{M}]', '')) = LOWER(REGEXP_REPLACE(a.raw_author_name, '[^\\p{L}\\p{M}]', ''))))) END AS rec
                  FROM raw_authorships r
                  LEFT JOIN raw_authorships_curated rac ON rac.work_id = r.work_id
                      AND NOT rac.editor_fill -- oxjob #2075: editors fill only an empty byline, never extend one
              )
          )
      )
    )
  ),
  -- end authors
  -- abstract lengths for the pubmed-vs-crossref abstract preference
  base_with_abstract_lens AS (
    SELECT /*+ BROADCAST(late_reposts) */
      *,
      LENGTH(abstract) AS abstract_len,
      MAX(CASE WHEN provenance = 'crossref' AND abstract IS NOT NULL THEN LENGTH(REGEXP_REPLACE(abstract, '<[^>]+>', '')) END) OVER (PARTITION BY work_id) AS crossref_abstract_len,
      -- oxjob #2142: the work holds a preprint record's abstract (a preprint merged with its published version)
      MAX(CASE WHEN type = 'preprint' AND abstract_inverted_index IS NOT NULL THEN 1 ELSE 0 END) OVER (PARTITION BY work_id) = 1 AS has_preprint_abstract,
      -- oxjob #1581: a DataCite version record whose concept DOI is also on this work
      COALESCE(ARRAY_CONTAINS(COLLECT_SET(CASE WHEN provenance = 'datacite' THEN LOWER(native_id) END) OVER (PARTITION BY work_id), version_of_doi), FALSE) AS is_concept_on_work,
      -- oxjob #1057: a DataCite dataset record that declares IsPartOf another DataCite dataset record on this work (a file
      -- merged into its dataset)
      COALESCE(ARRAYS_OVERLAP(COLLECT_SET(CASE WHEN provenance = 'datacite' AND type = 'dataset' THEN LOWER(native_id) END) OVER (PARTITION BY work_id), part_of_dois), FALSE) AS is_parent_on_work,
      -- oxjob #1941: the late re-post itself (late_reposts above), and its twins: the same deposit under a DOI with the same
      -- suffix (Firenze University Press's earlier DataCite registration of the very DOI, dated at registration; OSF's 10.17605
      -- node DOI of a 10.312xx preprint), or a DataCite record itself posted 2+ years after the work existed (an arXiv re-post).
      -- The re-post ranks after every record, a twin just before it. Both stay locations.
      COALESCE(provenance = 'crossref' AND raw_type = 'posted-content'
        AND ARRAY_CONTAINS(late_suffixes, NULLIF(REGEXP_EXTRACT(LOWER(native_id), '^10[.][0-9]+/(.+)', 1), '')), FALSE) AS is_late_repost,
      COALESCE(provenance IN ('crossref', 'datacite') AND NOT (provenance = 'crossref' AND raw_type = 'posted-content')
        AND (ARRAY_CONTAINS(late_suffixes, NULLIF(REGEXP_EXTRACT(LOWER(native_id), '^10[.][0-9]+/(.+)', 1), ''))
             OR (provenance = 'datacite' AND DATEDIFF(published_date, late_corroborated_date) > 730)), FALSE) AS is_late_twin
    FROM
      base
      LEFT JOIN late_reposts USING (work_id) -- oxjob #1941
  ),
  collect_all_values AS (
    select
      work_id,
      collect_list(struct(best_doi, doi_priority, cr_registered)) as best_dois,
      -- oxjob #1676: a blank title (no letter, digit, punctuation or symbol: '', spaces, tabs, no-break spaces, control
      -- characters) is no title. NULL drops it from the vote, so a real title on another location wins (~10K works), and a work
      -- with none gets NULL, not '', which sorted first under sort=display_name:asc (~627K works). Same classes the ES sync's sanitize_name keeps.
      -- oxjob #1852: a title carrying J-GLOBAL's machine-translation marker (【Powered by NICT】, 【JST・京大機械翻訳】) ranks after
      -- every other title (998, under get_highest_priority_value's 999), so it shows only when it is the work's only title.
      -- oxjob #1894: a version record whose concept record is on the work ranks after it for the title too, as for the DOI
      -- (doi_priority, #1581), so the title is the work's DOI record's. Zenodo keeps its concept DOI's DataCite record equal
      -- to the newest version, so the work shows the current title, not the alphabetically first of its versions' titles
      -- (6 of 60 sampled multi-version Zenodo works showed an old version's title). Dates, type and biblio keep cr_vor_priority.
      collect_list(struct(CASE WHEN title RLIKE '[\\p{L}\\p{N}\\p{P}\\p{S}]' THEN title END AS title,
        CASE WHEN title RLIKE '【(Powered by NICT|JST・京大機械翻訳)】\\s*$' THEN 998 ELSE doi_priority END AS title_priority)) as titles,
      collect_list(struct(publisher, cr_vor_priority)) as publishers,
      -- oxjob #1057: a file merged into its dataset gives no abstract (nor inverted index, nor language): a file's
      -- description (':unav', 'Codebook') does not describe the dataset, so a dataset without one keeps none.
      -- oxjob #2142: one record gives the work both its abstract and its inverted index. Picked apart, each tie went to the
      -- smaller string, so a merged preprint + article could show the preprint's abstract, or one record's index beside
      -- another's text. Order: a published version's abstract first (is_published_abstract), then provenance priority as
      -- before (PubMed first unless a Crossref abstract on the work is 20% longer), then the smaller index, as before, so a
      -- tie shows the same abstract it did, then native_id. A NULL key is no candidate (min_by skips it).
      min_by(
        CASE WHEN NOT is_parent_on_work AND abstract_inverted_index IS NOT NULL AND priority IS NOT NULL
          THEN named_struct('abstract', abstract, 'abstract_inverted_index', abstract_inverted_index) END,
        CASE WHEN NOT is_parent_on_work AND abstract_inverted_index IS NOT NULL AND priority IS NOT NULL
          THEN named_struct('not_published', NOT is_published_abstract, 'priority', CASE WHEN provenance = 'pubmed' AND crossref_abstract_len > abstract_len * 1.2 THEN priority WHEN provenance = 'pubmed' THEN 0 ELSE priority END,
                            -- oxjob #1894: the concept record's abstract (the newest version's) before its versions', as the title
                            'version_of_concept_on_work', is_concept_on_work,
                            'abstract_inverted_index', abstract_inverted_index, 'native_id', native_id) END
      ) AS location_abstract,
      array_distinct(flatten(collect_list(referenced_works))) as referenced_works,
      -- oxjob #1926: the work's Research Square preprint's own CC abstract, text and index from the same record (the
      -- latest version, then DOI, when the work holds several). NULL when it holds none (max_by skips rows whose key is
      -- NULL, so only those records compete). Used only under the takedown rule. The struct is built only for those
      -- records (~0.5M of every location): building it for every row copied each abstract and index into the
      -- aggregate a second time (a suspect in night 2026-10-30's Works_Base timeout).
      max_by(
        CASE WHEN is_preprint_own_cc_abstract AND abstract_inverted_index IS NOT NULL
          THEN named_struct('abstract', abstract, 'abstract_inverted_index', abstract_inverted_index) END,
        CASE WHEN is_preprint_own_cc_abstract AND abstract_inverted_index IS NOT NULL
          THEN concat(lpad(coalesce(nullif(regexp_extract(lower(native_id), '/v([0-9]+)$', 1), ''), '0'), 6, '0'), lower(native_id)) END
      ) AS preprint_own_abstract,
      -- oxjob #2495: each record's volume, issue and pages as one block, with the DOI the record is the record of (Crossref and
      -- DataCite only; a MAG, PubMed or repository record's DOI is only a claim), so set_fields picks a tie's biblio per record.
      collect_list(struct(cr_vor_priority AS p, IF(provenance IN ('crossref', 'datacite'), LOWER(native_id), NULL) AS doi,
        volume, issue, first_page, last_page)) AS biblio_records,
      -- oxjob #1098: the volumes and issues the work's repository citations state (repo_source_biblio); a volume 0 or a year
      -- as volume (19xx/20xx, DOAJ's stand-in for journals without volumes) is skipped with its issue.
      collect_set(CASE WHEN repo_source_biblio.volume <> '0' AND NOT repo_source_biblio.volume RLIKE '^(19|20)[0-9]{2}$'
        THEN repo_source_biblio.volume END) AS repo_source_volumes,
      collect_set(CASE WHEN repo_source_biblio.volume <> '0' AND NOT repo_source_biblio.volume RLIKE '^(19|20)[0-9]{2}$'
        THEN repo_source_biblio.issue END) AS repo_source_issues,
      collect_set(repo_source_pages) AS repo_source_page_ranges, -- oxjob #1098 step 4 (repo_source_pages, below)
      -- oxjob #1871: an NLM-declared article language outranks every copy's text classification (a Crossref copy's English
      -- title would otherwise win at priority 1)
      collect_list(struct(CASE WHEN NOT is_parent_on_work THEN language END AS language, CASE WHEN is_nlm_declared_language THEN 0 ELSE priority END AS priority)) as languages, -- oxjob #1057: as abstracts
      filter(
        collect_list(struct(published_date, cr_vor_priority)),
        -- oxjob #1644: skip a location date of year 10000+ (BORIS DataCite deposits carry Available +10000-01-01,
        -- which wins DataCite.py's string-earliest pick); it outranked the PubMed / MAG dates and the work served no year.
        x -> x.published_date is not null AND year(x.published_date) < 10000
      ) as published_dates,
      filter(
        collect_list(openalex_created_dt),
        x -> x is not null
      ) as openalex_created_dts,
      filter(
        collect_list(struct(openalex_updated_dt, priority)),
        x -> x.openalex_updated_dt is not null
      ) as openalex_updated_dts,
      ARRAY_SORT(
        filter(
          collect_set(struct(native_id_namespace, native_id)),
          -- oxjob #2065: nor is a minted data-series record's native id (namespace `series`) an id of the work
          x -> lower(x.native_id_namespace) NOT IN ('pmh', 'series')
        ),
        (left, right) -> CASE
          WHEN left.native_id_namespace < right.native_id_namespace THEN -1
          WHEN left.native_id_namespace > right.native_id_namespace THEN 1
          WHEN left.native_id < right.native_id THEN -1
          WHEN left.native_id > right.native_id THEN 1
          ELSE 0
        END
      ) as ids,
      -- oxjob #2155: the work's PubMed records (PMID, the record's own DOI, typed preprint), for the PMID pick in set_fields
      collect_set(CASE WHEN provenance = 'pubmed' AND native_id_namespace = 'pmid' THEN named_struct(
        'pmid', native_id, 'doi', lower(best_doi), 'is_preprint', COALESCE(type = 'preprint', FALSE)) END) AS pubmed_records,
      -- locations
      collect_set(
        struct(
          case
            when provenance = 'repo_backfill' then 'repo'
            else provenance
          end as provenance,
          native_id,
          -- oxjob #1573: DOI of the registration record behind this location (Crossref/DataCite).
          -- oxjob #1616: other provenances carry their record's DOI as a candidate; set_fields keeps it only when it is
          -- the work's DOI and no Crossref/DataCite location holds it.
          case
            when provenance in ('crossref', 'datacite')
              then case when native_id_namespace = 'doi' then concat('https://doi.org/', lower(native_id)) end
            when best_doi rlike '^10\\.' then concat('https://doi.org/', lower(best_doi))
          end as doi,
          case
            when is_dead_link then 12 -- oxjob #1006: a copy on a dead or squatted host ranks after every other location
            when is_late_repost then 11 -- oxjob #1941: a late re-post ranks after every other record
            when is_late_twin then 10 -- oxjob #1941: and its same-deposit twin just before it
            when provenance = 'crossref'
            and best_doi is not null
            and version = 'publishedVersion' then 0 -- crossref version of record wins primary (oxjob #589)
            when provenance = 'crossref'
            and best_doi is not null then 1 -- publisher with a doi
            when provenance = 'crossref' then 2 -- publisher without a doi
            when version = 'publishedVersion'
            and pdf_url is not null then 3 -- published version with a pdf url
            when version = 'publishedVersion' then 4 -- published version without a pdf url
            when version = 'acceptedVersion'
            and pdf_url is not null then 5 -- accepted version with a pdf url
            when version = 'acceptedVersion' then 6 -- accepted version without a pdf url
            when version = 'submittedVersion'
            and pdf_url is not null then 7 -- submitted version with a pdf url
            when version = 'submittedVersion' then 8 -- submitted version without a pdf url
            else 9
          end as sort_score,
          location_is_oa as is_oa, -- oxjob #2269: defined once, below, so served_license reads the same flag
          coalesce(
            landing_page_url,
            get(
              filter(urls, x -> x.content_type = "html").url,
              0
            )
          ) as landing_page_url,
          pdf_url,
          case
            when contains(
              coalesce(pdf_url, landing_page_url),
              "europepmc.org"
            ) then 1
            -- oxjob #2448: PMC's own URL form (pmc.ncbi.nlm.nih.gov/articles/PMC<n>/, #2112) ranks as '/pmc/' did
            when contains(coalesce(pdf_url, landing_page_url), "/pmc/")
              or contains(coalesce(pdf_url, landing_page_url), "pmc.ncbi.nlm.nih.gov") then 2
            when contains(coalesce(pdf_url, landing_page_url), "arxiv") then 3
            when contains(coalesce(pdf_url, landing_page_url), ".edu") then 4
            else 5
          end as url_sort_score,
          oa_status,
          COALESCE(version = 'publishedVersion', FALSE) as is_published,
          COALESCE(version in ('acceptedVersion','publishedVersion'), FALSE) as is_accepted,
          CASE
            WHEN source_id IS NULL THEN NULL
            ELSE struct(
              concat("https://openalex.org/S", source_id) as id,
              display_name AS display_name,
              issn_l,
              issns as issn,
              COALESCE(source_is_oa, FALSE) as is_oa,
              COALESCE(is_in_doaj, FALSE) as is_in_doaj,
              COALESCE(is_core, FALSE) as is_core,
              -- external list membership; hash-excluded in CreateWorksEnriched (oxjob #1205)
              COALESCE(listed_in, ARRAY()) as listed_in,
              -- Same rule as CreateSourcesApi: publisher first, else institution (oxjob #1563:
              -- journals linked only by institution_id had a host org on /sources but not on works).
              -- oxjob #2453: a repository with no institution falls through to its publisher, as its lineage already does.
              case
                when source_type = 'repository' and institution_id IS NOT NULL then concat('https://openalex.org/I', institution_id)
                when publisher_id IS NOT NULL then concat('https://openalex.org/P', publisher_id)
                when institution_id IS NOT NULL then concat('https://openalex.org/I', institution_id)
              end as host_organization,
              case
                when source_type = 'repository' and institution_id IS NOT NULL then institution_name
                when publisher_id IS NOT NULL then publisher_name
                when institution_id IS NOT NULL then institution_name
              end as host_organization_name,
              IF ((source_type = 'repository' or source_type = 'metadata' or publisher_id IS NULL) and institution_id IS NOT NULL,
                ARRAY(CONCAT('https://openalex.org/I', institution_id)),
                -- oxjob #1813: the publisher's whole chain, as CreateSourcesApi builds it (was the publisher and its parent only)
                ARRAY_COMPACT(CONCAT(ARRAY(CONCAT('https://openalex.org/P', publisher_id)), COALESCE(publisher_ancestor_urls, ARRAY(parent_publisher.id))))
              ) AS host_organization_lineage,
              IF ((source_type = 'repository' or source_type = 'metadata' or publisher_id IS NULL) and institution_id IS NOT NULL,
                CAST(ARRAY() AS ARRAY<STRING>),
                ARRAY_COMPACT(CONCAT(ARRAY(publisher_name), COALESCE(publisher_ancestor_names, ARRAY(parent_publisher.display_name))))
              ) AS host_organization_lineage_names,
              source_type as type
            )
          END as source,
          -- repo dc:source can hold full citations/author lists; >500 chars is not a source name
          CASE
            WHEN LENGTH(TRIM(REGEXP_REPLACE(source_name, '\\s+', ' '))) <= 500
              THEN TRIM(REGEXP_REPLACE(source_name, '\\s+', ' '))
          END as raw_source_name,
          raw_type,
          apc_prices,
          apc_usd,
          served_license as license, -- oxjob #2269 (below)
          CASE WHEN served_license IS NOT NULL THEN CONCAT('https://openalex.org/licenses/', served_license) END as license_id,
          version,
          host_type,
          case
            when provenance in ('repo', 'repo_backfill') then endpoint_id
          end as endpoint_id,
          case
            when provenance in ('repo', 'repo_backfill') then native_id
          end as pmh_id,
          provenance = 'crossref' as is_unpaywall_record,
          pdf_s3_id,
          grobid_s3_id,
          type as location_type,
          cast(openalex_updated_dt as timestamp) as updated,
          is_parent_on_work, -- oxjob #1057: read by the locations_sorted sort only
          cr_registered -- oxjob #1902 F2: read by the locations_sorted sort only
        )
      ) as locations,
      exists(collect_set(is_retracted), x -> x = True) as is_retracted,
      array_contains(collect_set(provenance), 'crossref') as indexed_in_crossref
    from
      (
        -- oxjob #589: when a work has >1 crossref DOI (e.g. a preprint DOI and the
        -- published DOI), the published (publishedVersion) crossref DOI must win
        -- canonical doi/type/publisher/biblio. Encode as a x10 rescale + a +1
        -- penalty on non-publishedVersion crossref locations, so the reordering is
        -- contained WITHIN crossref and provenance precedence is untouched.
        select
          *,
          -- oxjob #1098: a repository record's citation (OAI dc:source, kept as source_name) states the volume and issue the
          -- record has no field for: OJS "Journal; Vol 9 No 3 (2019): ...", DOAJ "Vol 53, Iss 3, Pp 4-10 (2023)", Pure
          -- "vol. 26, no. 4, pp. ...". Read only from a citation with exactly one volume + issue pair; supplements ("Iss 4(S)",
          -- "106-Ek", "Suppl", "5(4)") don't match; leading zeros dropped; issue 0 is no issue. Used fill-only in set_fields.
          CASE WHEN provenance IN ('repo', 'repo_backfill') AND source_name ILIKE '%vol%'
              AND SIZE(REGEXP_EXTRACT_ALL(source_name, '(?i)(?<![\\p{L}\\p{N}])vol(?:ume)?\\.?\\s*([0-9]{1,5})\\s*[,;:]?\\s*(?:no|nr|n|num|iss|issue)\\.?\\s*([0-9]{1,5}(?:\\s*[-/]\\s*[0-9]{1,5}|\\.[0-9]{1,3})?)(?![0-9\\p{L}])(?!\\([0-9])(?!\\s*[-(]\\s*(?:s|si|sp|ek)\\b)(?!\\s*[-(]?\\s*(?:suppl?|supl|supp|spec|special|esp|especial)\\b)', 1)) = 1
            THEN NAMED_STRUCT(
              'volume', REGEXP_REPLACE(REGEXP_EXTRACT(source_name, '(?i)(?<![\\p{L}\\p{N}])vol(?:ume)?\\.?\\s*([0-9]{1,5})\\s*[,;:]?\\s*(?:no|nr|n|num|iss|issue)\\.?\\s*([0-9]{1,5}(?:\\s*[-/]\\s*[0-9]{1,5}|\\.[0-9]{1,3})?)(?![0-9\\p{L}])(?!\\([0-9])(?!\\s*[-(]\\s*(?:s|si|sp|ek)\\b)(?!\\s*[-(]?\\s*(?:suppl?|supl|supp|spec|special|esp|especial)\\b)', 1), '^0+(?=[0-9])', ''),
              'issue', NULLIF(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_EXTRACT(source_name, '(?i)(?<![\\p{L}\\p{N}])vol(?:ume)?\\.?\\s*([0-9]{1,5})\\s*[,;:]?\\s*(?:no|nr|n|num|iss|issue)\\.?\\s*([0-9]{1,5}(?:\\s*[-/]\\s*[0-9]{1,5}|\\.[0-9]{1,3})?)(?![0-9\\p{L}])(?!\\([0-9])(?!\\s*[-(]\\s*(?:s|si|sp|ek)\\b)(?!\\s*[-(]?\\s*(?:suppl?|supl|supp|spec|special|esp|especial)\\b)', 2), '\\s+', ''), '^0+(?=[0-9])', ''), '0'))
          END AS repo_source_biblio,
          -- oxjob #1098 step 4: the same citation's page range (DOAJ "Pp 4-10", Pure "pp. 61-72"), read only when it states
          -- exactly one; leading zeros dropped. A range starting at page 1 (or 0) is skipped: DOAJ cites a continuous journal's
          -- article by its PDF's span (BMC "Pp 1-8" for article 180), 1.03M of the 1.29M disagreements beside a record's page.
          -- So is a range that runs backwards or over 500 pages. A 'p' inside a word, URL or DOI ('P11-1021') is not a page.
          CASE WHEN provenance IN ('repo', 'repo_backfill') AND source_name ILIKE '%p%'
              AND SIZE(REGEXP_EXTRACT_ALL(source_name, '(?i)(?<![\\p{L}\\p{N}/._-])pp?\\.?\\s*([0-9]{1,6})\\s*[-\\x{2013}]\\s*([0-9]{1,6})(?![0-9\\p{L}-])', 1)) = 1
              AND REGEXP_REPLACE(REGEXP_EXTRACT(source_name, '(?i)(?<![\\p{L}\\p{N}/._-])pp?\\.?\\s*([0-9]{1,6})\\s*[-\\x{2013}]\\s*([0-9]{1,6})(?![0-9\\p{L}-])', 1), '^0+(?=[0-9])', '') NOT IN ('0', '1')
              AND TRY_CAST(REGEXP_EXTRACT(source_name, '(?i)(?<![\\p{L}\\p{N}/._-])pp?\\.?\\s*([0-9]{1,6})\\s*[-\\x{2013}]\\s*([0-9]{1,6})(?![0-9\\p{L}-])', 2) AS INT) >= TRY_CAST(REGEXP_EXTRACT(source_name, '(?i)(?<![\\p{L}\\p{N}/._-])pp?\\.?\\s*([0-9]{1,6})\\s*[-\\x{2013}]\\s*([0-9]{1,6})(?![0-9\\p{L}-])', 1) AS INT)
              AND TRY_CAST(REGEXP_EXTRACT(source_name, '(?i)(?<![\\p{L}\\p{N}/._-])pp?\\.?\\s*([0-9]{1,6})\\s*[-\\x{2013}]\\s*([0-9]{1,6})(?![0-9\\p{L}-])', 2) AS INT) - TRY_CAST(REGEXP_EXTRACT(source_name, '(?i)(?<![\\p{L}\\p{N}/._-])pp?\\.?\\s*([0-9]{1,6})\\s*[-\\x{2013}]\\s*([0-9]{1,6})(?![0-9\\p{L}-])', 1) AS INT) <= 500
            THEN NAMED_STRUCT(
              'first_page', REGEXP_REPLACE(REGEXP_EXTRACT(source_name, '(?i)(?<![\\p{L}\\p{N}/._-])pp?\\.?\\s*([0-9]{1,6})\\s*[-\\x{2013}]\\s*([0-9]{1,6})(?![0-9\\p{L}-])', 1), '^0+(?=[0-9])', ''),
              'last_page', REGEXP_REPLACE(REGEXP_EXTRACT(source_name, '(?i)(?<![\\p{L}\\p{N}/._-])pp?\\.?\\s*([0-9]{1,6})\\s*[-\\x{2013}]\\s*([0-9]{1,6})(?![0-9\\p{L}-])', 2), '^0+(?=[0-9])', ''))
          END AS repo_source_pages,
          -- oxjob #1941: a late re-post ranks after every record, its twin just before it (both under get_highest_priority_value's 999)
          if(is_late_repost, 990, if(is_late_twin, 980, priority * 10
            + if(provenance = 'crossref'
                 and version is distinct from 'publishedVersion', 1, 0)
            -- oxjob #1057: a DataCite record whose IsPartOf parent is on this work (a Dataverse file merged into its dataset)
            -- ranks after the parent, still inside DataCite's band, so the dataset's title, publisher, biblio and dates win.
            + if(is_parent_on_work, 5, 0))) as cr_vor_priority,
          -- oxjob #1581 (Casey 2026-10-07): the concept DOI is the work's DOI. A version DOI whose concept is on the same work
          -- ranks after it, still inside its provenance's band; DOI and title (#1894) only, so type/biblio/dates keep cr_vor_priority.
          -- oxjob #1057: a file's DOI ranks after its dataset's the same way; one +5 for either, so DataCite stays in its band.
          -- oxjob #1941: as cr_vor_priority (a late re-post's DOI still wins when no other record has one).
          if(is_late_repost, 990, if(is_late_twin, 980, priority * 10
            + if(provenance = 'crossref'
                 and version is distinct from 'publishedVersion', 1, 0)
            + if(is_concept_on_work OR is_parent_on_work, 5, 0))) as doi_priority,
          -- oxjob #2142: on a work holding a preprint's abstract, a published version's abstract outranks it (location_abstract),
          -- unless it is a fragment: under 100 characters (a cover-page note) or a PubMed text that does not begin with a capital
          -- letter (the PubMed parser cut it at an inline tag, #182). Other works rank as before.
          COALESCE(has_preprint_abstract AND is_published_record AND LENGTH(abstract) >= 100
            AND (provenance <> 'pubmed' OR abstract RLIKE '^\\p{Lu}'), FALSE) AS is_published_abstract,
          -- the location's served open flag (the struct's is_oa above): a repository copy also opens on its own record's flag
          case
            when host_type = 'repository' then (
              is_oa_raw
              OR composite_is_oa
            )
            else composite_is_oa
          end as location_is_oa,
          -- temp license filter, need to update normalize_license and re-run all DLTs to remove these upstream
          -- oxjob #2269 (ZD 24580): a location serves its licence only while it is open, or when a curation set it. The
          -- licence is "the license this copy is available under" (help.openalex.org, Locations); on closed copies it was a
          -- page's figure-credit or table-of-contents CC link, a "public domain in the US" government-work notice, or the
          -- rights field of a repository record with no file, so `is_oa:false` + `locations.license:cc-by` returned ~101K
          -- works, each closed or open elsewhere. Open copies keep theirs: best_oa_location, oa_status and the Unpaywall
          -- feed (open copies only) don't move. Revert: drop `AND (COALESCE(location_is_oa, FALSE) OR is_license_curated)`.
          CASE
            WHEN license IS NOT NULL AND license NOT IN ('False', 'unspecified-oa', 'gpl', 'gpl-2', 'gpl-3', 'apache-2.0')
              AND (COALESCE(location_is_oa, FALSE) OR is_license_curated)
              THEN license
          END AS served_license
        from base_with_abstract_lens
      )
    group by
      work_id
  ),
  abstracts_backfill AS (
    SELECT
      bf.work_id,
      -- Garbled-text overlay (oxjob #807). abstracts_backfill is a static legacy snapshot with
      -- no producer, so neither the at-ingest cleaner nor a full Silver reparse ever cleans it,
      -- yet ~104.9M works publish their abstract from here and 3.79M of those carry undecoded
      -- HTML entities, raw tags, or mojibake. `abstracts_backfill_repair` (built once by
      -- notebooks/maintenance/CleanAbstractsBackfill through the SAME dlt_utils cleaner the
      -- ingest path uses) holds only the rows whose text changed. Swap the pair TOGETHER --
      -- a cleaned abstract beside an uncleaned index would break the #191.1 invariant that the
      -- searched text and the inverted index derive from byte-identical input.
      CASE WHEN r.work_id IS NOT NULL THEN r.abstract ELSE bf.abstract END AS abstract,
      CASE WHEN r.work_id IS NOT NULL THEN r.abstract_inverted_index ELSE bf.abstract_inverted_index END AS abstract_inverted_index,
      CAST(NULL AS BIGINT) AS replace_hash
    FROM
      openalex.abstracts.abstracts_backfill bf
      LEFT JOIN openalex.abstracts.abstracts_backfill_repair r ON bf.work_id = r.work_id
      -- Not-an-abstract filter (oxjob #1800): the junk Parseland no longer emits (site blurbs, "No
      -- abstract available.", SEO citation lines, CABI datasheet lines) sits in this snapshot too, and
      -- came straight back through the fallback once the landing page dropped it (642K works, 25 Oct
      -- 2026). The work_ids are precomputed with parseland-lib's is_not_abstract() on the text above
      -- (repair overlay applied) by notebooks/maintenance/FlagBackfillNotAbstracts.
      -- Revert: delete this join (or TRUNCATE the table; never DROP it while this reads it).
      LEFT ANTI JOIN openalex.abstracts.abstracts_backfill_not_abstract na ON bf.work_id = na.work_id
    WHERE
      -- Junk-abstract denylist (oxjob 682): this legacy table holds ~1.04M parser-boilerplate
      -- rows (CUP preview notices, T&F zoom-UI); without this filter they re-enter works via
      -- the priority-997 fallback whenever every other abstract source is empty.
      -- Matched against the ORIGINAL text: the denylist patterns are what the raw snapshot says.
      NOT (bf.abstract LIKE 'Click to increase image size%'
        OR bf.abstract LIKE 'An abstract is not available for this content so a preview%'
        OR bf.abstract LIKE '%A summary is not available for this content so a preview%'
        OR bf.abstract LIKE 'Advertisements that appeared within the print issues of Chem. Eng. News%')
    -- oxjob #1800: works with no other abstract take a repository record's later description that a record reader
    -- accepted (notebooks/maintenance/BuildRepoDescriptionFill), at this fallback's priority (997), so it shows only
    -- where no location has one. Its works never have a row in abstracts_backfill (the builder excludes them, and that
    -- snapshot gains no rows), so the union keeps one row per work. A union, not its own join: the join added ~10 min
    -- and 3x spill to this statement (shadow build, 9 Nov). Revert: TRUNCATE openalex.abstracts.repo_description_fill.
    -- oxjob #1800 lead 2: a row with replace_hash also replaces the record's short first description (a cataloguing note,
    -- "Orientador: ...") where the work shows exactly that text; see set_fields. Revert those rows alone:
    -- DELETE FROM openalex.abstracts.repo_description_fill WHERE replace_hash IS NOT NULL.
    UNION ALL
    SELECT rdf.work_id, rdf.fill_abstract, rdf.fill_abstract_inverted_index, rdf.replace_hash
    FROM openalex.abstracts.repo_description_fill rdf
  ),
  set_fields AS (
    SELECT
      collect_all_values.work_id as id,
      -- oxjob #1162: a title curation from users-api (SyncWorkFieldCurations) wins over the records' title, here, so
      -- every later use of the title reads it (control-character cleanup, topics_key, PMC matching, type rules).
      COALESCE(wcf.wcf_title, get_highest_priority_value(titles, titles.title)) AS title,
      -- get_highest_priority_value, with the Crossref registration date between doi_priority and the alphabetical tie-break
      -- (oxjob #1902 F2)
      GET(ARRAY_SORT(
        FILTER(best_dois, x -> x.best_doi IS NOT NULL AND x.doi_priority < 999),
        (x, y) -> CASE
          WHEN x.doi_priority <> y.doi_priority THEN IF(x.doi_priority < y.doi_priority, -1, 1)
          WHEN NOT (x.cr_registered <=> y.cr_registered)
            THEN IF(COALESCE(x.cr_registered, DATE'0001-01-01') > COALESCE(y.cr_registered, DATE'0001-01-01'), -1, 1)
          WHEN x.best_doi <> y.best_doi THEN IF(x.best_doi < y.best_doi, -1, 1)
          ELSE 0
        END), 0).best_doi AS raw_best_doi,
      CONCAT('https://doi.org/', raw_best_doi) AS best_doi,
      get_highest_priority_value(publishers, publishers.publisher) AS publisher,
      -- Springer/Elsevier takedown rule (commit 9fda36ac): no abstract index for these works unless an OA copy is
      -- diamond/gold/hybrid or has no source. Read by abstract and abstract_inverted_index below (oxjob #1926).
      (
          -- Springer/Elsevier by publisher string or host org id
          (
            publisher IS NOT NULL
            AND lower(publisher) RLIKE '(springer|elsevier)'
            -- oxjob #2351 (17 Nov 2026): Springer Publishing Company (New York; DOI prefix 10.1891; publisher P4310320499,
            -- ROR 03629ds51) is not Springer Nature; the name matched 'springer' and hid 32,821 works' own abstracts.
            -- Revert: delete this line.
            AND NOT lower(trim(publisher)) RLIKE '^springer publishing company[.]?$'
          )
          -- oxjob #2351 (17 Nov 2026): the host-organization arm read split index 4, which is always NULL
          -- ('https://openalex.org/P4310319965' splits into 4 parts; the P id is index 3), so it never fired and the name arm
          -- alone hid abstracts. Fixed as the rule was written: it also hides works on Elsevier- and Springer Nature-hosted
          -- sources whose Crossref publisher name says neither (society journals such as Blood and Journal of Dairy Science;
          -- works with no Crossref publisher). A source's host must be current for this: a title that moved to another
          -- publisher is re-hosted to its DOI registrant in the sources registry. Springer Publishing Company (P4310320499,
          -- not Springer Nature) stays out under this arm too, as under the name arm.
          -- Revert: index back to 4 and drop the publisher line (git revert).
          OR (
            exists(
              locations,
              x -> x.source.host_organization IS NOT NULL
              AND get(split(x.source.host_organization, '/'), 3) IN ('P4310320990', 'P4310319965')
              -- oxjob #2453: a repository copy never triggers this arm (repositories had no publisher host before #2453;
              -- Scopus (Elsevier) S4306400063 has one since, and 7,673 works holding a Scopus copy keep their abstract).
              AND NOT coalesce(x.source.type = 'repository', FALSE)
            )
            AND NOT coalesce(lower(trim(publisher)) RLIKE '^springer publishing company[.]?$', FALSE)
          )
        ) -- Allow only if any OA location is diamond/gold/hybrid (1/2/3)
        -- oxjob #1867: or has no source. Those copies were labelled gold until #1867 and opened this gate; they still do,
        -- so the label fix moves no abstract. A CC-licensed copy does not open it; Jason (29 Oct 2026, oxjob #1926) allowed
        -- only a Research Square preprint's own deposit abstract (preprint_own_abstract), never another record's.
        AND NOT exists(
          filter(locations, x -> x.is_oa),
          y -> y.oa_status IN (1, 2, 3) OR y.host_type IS NULL
        ) AS is_abstract_takedown,
      -- oxjob #1926: under the takedown rule the searched text is the preprint's own abstract when the work holds one, so
      -- it is the text of the index shown (#191.1); otherwise unchanged.
      -- oxjob #1103: a site curator's remove (users-api property abstract, remove only; SyncWorkFieldCurations): none shown
      CASE WHEN array_contains(wcf.wcf_curated_fields, 'abstract') THEN NULL
      WHEN is_abstract_takedown AND preprint_own_abstract IS NOT NULL THEN preprint_own_abstract.abstract
      -- oxjob #1800 lead 2: the work shows a repository record's short first description (the fill row's replace_hash) and
      -- a record reader accepted that record set's later description: show the later one. Any other text the work shows wins.
      WHEN bf.replace_hash IS NOT NULL AND location_abstract IS NOT NULL
        AND xxhash64(location_abstract.abstract) = bf.replace_hash THEN bf.abstract
      -- oxjob #2142: the record collect_all_values chose, else the backfill row (only when no location has an abstract)
      WHEN location_abstract IS NOT NULL THEN location_abstract.abstract
      ELSE bf.abstract END AS abstract,
      SIZE(referenced_works) AS referenced_works_count,
      referenced_works,
      CASE
        WHEN array_contains(wcf.wcf_curated_fields, 'abstract') THEN NULL -- oxjob #1103: removed by a curation, as above
        WHEN is_abstract_takedown THEN preprint_own_abstract.abstract_inverted_index -- oxjob #1926: NULL unless the preprint's own
        WHEN bf.replace_hash IS NOT NULL AND location_abstract IS NOT NULL
          AND xxhash64(location_abstract.abstract) = bf.replace_hash THEN bf.abstract_inverted_index -- oxjob #1800 lead 2
        WHEN location_abstract IS NOT NULL THEN location_abstract.abstract_inverted_index -- oxjob #2142: the text's record
        ELSE bf.abstract_inverted_index
      END AS abstract_inverted_index,
      -- oxjob #2495: volume, issue and pages from the records at the best priority, without mixing them. Two records of one
      -- priority (two Crossref records: a JSTOR 10.2307 mirror beside the publisher's DOI, an alias DOI; two PubMed or MAG
      -- records of one article) tied, and the smallest string won each field on its own, so a record stating only the start
      -- page set the last page: 380K works served 1105-1105 where their own DOI's record says 1105-1117 (W2164351958).
      -- Precedence between priorities is unchanged. Within a tie the work's own DOI record (Crossref or DataCite,
      -- raw_best_doi) wins each field, then the smallest string as before; the last page comes from a record that states the
      -- chosen first page (else as before). No preference for a range over a lone page: two PubMed records of a letter and
      -- its reply (558 and 558-562) tie, and PubMed's record of the work gives the lone page (oxjob #2495, 43 of 60).
      -- Revert: get_highest_priority_value(volumes / issues / first_pages / last_pages) back in the fields below, their four
      -- collect_lists back in collect_all_values in place of biblio_records, and delete the four bib_* columns.
      GET(ARRAY_SORT(FILTER(biblio_records, r -> r.volume IS NOT NULL AND r.p < 999), (x, y) -> CASE
          WHEN x.p <> y.p THEN IF(x.p < y.p, -1, 1)
          WHEN (x.doi <=> LOWER(raw_best_doi)) <> (y.doi <=> LOWER(raw_best_doi)) THEN IF(x.doi <=> LOWER(raw_best_doi), -1, 1)
          WHEN x.volume <> y.volume THEN IF(x.volume < y.volume, -1, 1)
          ELSE 0 END), 0).volume AS bib_volume,
      GET(ARRAY_SORT(FILTER(biblio_records, r -> r.issue IS NOT NULL AND r.p < 999), (x, y) -> CASE
          WHEN x.p <> y.p THEN IF(x.p < y.p, -1, 1)
          WHEN (x.doi <=> LOWER(raw_best_doi)) <> (y.doi <=> LOWER(raw_best_doi)) THEN IF(x.doi <=> LOWER(raw_best_doi), -1, 1)
          WHEN x.issue <> y.issue THEN IF(x.issue < y.issue, -1, 1)
          ELSE 0 END), 0).issue AS bib_issue,
      GET(ARRAY_SORT(FILTER(biblio_records, r -> r.first_page IS NOT NULL AND r.p < 999), (x, y) -> CASE
          WHEN x.p <> y.p THEN IF(x.p < y.p, -1, 1)
          WHEN (x.doi <=> LOWER(raw_best_doi)) <> (y.doi <=> LOWER(raw_best_doi)) THEN IF(x.doi <=> LOWER(raw_best_doi), -1, 1)
          WHEN x.first_page <> y.first_page THEN IF(x.first_page < y.first_page, -1, 1)
          ELSE 0 END), 0).first_page AS bib_first_page,
      COALESCE(
        GET(ARRAY_SORT(FILTER(biblio_records, r -> r.last_page IS NOT NULL AND r.p < 999 AND TRIM(r.first_page) = TRIM(bib_first_page)),
          (x, y) -> CASE
            WHEN x.p <> y.p THEN IF(x.p < y.p, -1, 1)
            WHEN (x.doi <=> LOWER(raw_best_doi)) <> (y.doi <=> LOWER(raw_best_doi)) THEN IF(x.doi <=> LOWER(raw_best_doi), -1, 1)
            WHEN x.last_page <> y.last_page THEN IF(x.last_page < y.last_page, -1, 1)
            ELSE 0 END), 0).last_page,
        GET(ARRAY_SORT(FILTER(biblio_records, r -> r.last_page IS NOT NULL AND r.p < 999), (x, y) -> CASE
            WHEN x.p <> y.p THEN IF(x.p < y.p, -1, 1)
            WHEN x.last_page <> y.last_page THEN IF(x.last_page < y.last_page, -1, 1)
            ELSE 0 END), 0).last_page) AS bib_last_page,
      -- oxjob #1103: a page-field curation (users-api property biblio.<field>; SyncWorkFieldCurations) wins: a replace's
      -- value, NULL for a remove (as ApplyLocationCurations serves location curations)
      -- oxjob #1098: a work no record gives a volume takes the one volume its repository citations state, unless the work
      -- has a different issue; it takes their one issue only together with that volume (DOAJ puts article numbers in "Iss"
      -- for continuous journals, wrong beside a Crossref volume). Fill-only: a record's value or a curation always wins.
      -- Revert: the two COALESCE fills below, repo_source_volumes/issues in collect_all_values and repo_source_biblio.
      IF(array_contains(wcf.wcf_curated_fields, 'biblio.volume'), wcf.wcf_biblio_volume,
        COALESCE(bib_volume,
          IF(SIZE(repo_source_volumes) = 1
              AND (IF(array_contains(wcf.wcf_curated_fields, 'biblio.issue'), wcf.wcf_biblio_issue,
                    bib_issue) IS NULL
                OR (SIZE(repo_source_issues) = 1
                  AND IF(array_contains(wcf.wcf_curated_fields, 'biblio.issue'), wcf.wcf_biblio_issue,
                    bib_issue) = GET(repo_source_issues, 0))),
            GET(repo_source_volumes, 0), NULL))) AS volume,
      IF(array_contains(wcf.wcf_curated_fields, 'biblio.issue'), wcf.wcf_biblio_issue,
        COALESCE(bib_issue,
          IF(NOT COALESCE(array_contains(wcf.wcf_curated_fields, 'biblio.volume'), FALSE)
              AND bib_volume IS NULL
              AND SIZE(repo_source_volumes) = 1 AND SIZE(repo_source_issues) = 1,
            GET(repo_source_issues, 0), NULL))) AS issue,
      IF(array_contains(wcf.wcf_curated_fields, 'biblio.first_page'), wcf.wcf_biblio_first_page,
        bib_first_page) AS first_page,
      IF(array_contains(wcf.wcf_curated_fields, 'biblio.last_page'), wcf.wcf_biblio_last_page,
        bib_last_page) AS last_page,
      -- oxjob #1703: a language curation (users-api POST /curations, entity works, property language; SyncWorkFieldCurations)
      -- wins over every copy's declared or detected language.
      COALESCE(wcf.wcf_language, get_highest_priority_value(languages, languages.language)) AS language,
      TRY_CAST(
        ARRAY_MIN(openalex_created_dts) AS DATE
      ) AS created_date,
      -- oxjob #1125: an is_xpac curation (users-api POST /curations, entity works; SyncWorkFieldCurations) wins over the
      -- frozen xpac_work_ids list, e.g. a DOI-less conference paper the list put in the expansion corpus.
      COALESCE(wcf.wcf_is_xpac, xpac.work_id IS NOT NULL) AS is_xpac,
      -- oxjob #1098 step 4: the one page range the work's repository citations state (repo_source_pages), unless a page
      -- curation exists; read only by biblio below, where it fills pages no record gives.
      IF(NOT COALESCE(array_contains(wcf.wcf_curated_fields, 'biblio.first_page')
                      OR array_contains(wcf.wcf_curated_fields, 'biblio.last_page'), FALSE)
          AND SIZE(repo_source_page_ranges) = 1,
        GET(repo_source_page_ranges, 0), NULL) AS repo_cited_pages,
      CAST(
        get_highest_priority_value(
          openalex_updated_dts,
          openalex_updated_dts.openalex_updated_dt
        ) AS TIMESTAMP
      ) AS updated_date,
      -- oxjob #1644: where no location gives a date, the one-time date fill (maintenance/BuildPublicationDateFill.sql):
      -- the year a record reader (ResolvePublicationYears, Jev jev-v1, confidence >= 0.95) found in the work's own
      -- repository / DataCite records, audited at ~1.5% off by > 1 year. New records keep today's parse.
      -- Revert: empty openalex.works.publication_date_fill (the next nightly takes the years back out).
      -- Its column is fill_publication_date, NOT publication_date: YEAR(publication_date) below must stay this alias.
      -- oxjob #1865 (Jason, 27 Oct 2026): a date more than 12 months from today is an error (Zenodo 2035, Mendeley
      -- embargoes to 2035, Liverpool 2050); the work takes the day it first appeared (created_date) instead, which is
      -- stable night to night. Dates within the next 12 months stay (embargoes, forthcoming issues), so the year shown
      -- is always this year or next. Years 2400+ (Buddhist-era, 9999) stay with #1541. Revert: keep only the ELSE branch.
      -- oxjob #2241 part 2: the picked date, except that a work still dated the 1 January padded from its Crossref record's
      -- bare year takes its own PubMed record's later date that year (ypd, built in the cell above). No year changes.
      -- oxjob #1154: a work whose pick is a placeholder a record carries instead of a date (1970-01-01 epoch, 1899-12-30,
      -- a 1400/1500/1900 year before its journal existed) takes its PubMed / DataCite / MAG date, else none (ppd, built in
      -- the cell above; NULL falls through to the #1644 fill below, else the work shows no date).
      -- oxjob #2383: a work dated by its MAG record whose repository records all state one year, earlier than MAG's by the
      -- cell's gap, takes the repository date (mod, built in its cell; MAG ranks last only where the check says so).
      -- oxjob #2097: a work dated by its DataCite preprint record that holds the journal's own published copy takes that
      -- copy's date (pod, built in its cell: the published side survives).
      -- oxjob #1154 (repec_year): a work dated only by its MAG record whose RePEc records all state one year at least 6
      -- years earlier takes 1 January of that year (rpd, built in its cell: RePEc records carry only the handle's year).
      -- oxjob #1154 step 6: a work still dated a 1 January padded from one Crossref record's bare year takes the earliest
      -- later date another Crossref record of the work states that year (pyd, built in its cell). No year changes.
      CASE WHEN TRY_CAST(get_highest_priority_value(published_dates, published_dates.published_date) AS DATE) = ypd.padded_date
           THEN ypd.pubmed_date
           WHEN TRY_CAST(get_highest_priority_value(published_dates, published_dates.published_date) AS DATE) = ppd.placeholder_date
           THEN ppd.new_date
           -- oxjob #2383: a MAG-dated work whose repository records all state one earlier year takes it (mod, built above)
           WHEN TRY_CAST(get_highest_priority_value(published_dates, published_dates.published_date) AS DATE) = mod.mag_date
           THEN mod.repo_date
           -- oxjob #2097: a preprint-dated work holding its journal's own published copy takes that copy's date (pod, built above)
           WHEN TRY_CAST(get_highest_priority_value(published_dates, published_dates.published_date) AS DATE) = pod.preprint_date
           THEN pod.journal_date
           -- oxjob #1154: a MAG-only work whose RePEc handles state one year 6+ years earlier takes it (rpd, built above)
           WHEN TRY_CAST(get_highest_priority_value(published_dates, published_dates.published_date) AS DATE) = rpd.mag_date
           THEN rpd.repec_date
           -- oxjob #1154 step 6: a 1 January padded from one Crossref record's bare year takes the later date another
           -- Crossref record of the work states that year (pyd, built in its cell; no year changes)
           WHEN TRY_CAST(get_highest_priority_value(published_dates, published_dates.published_date) AS DATE) = pyd.padded_date
           THEN pyd.doi_date
           ELSE TRY_CAST(get_highest_priority_value(published_dates, published_dates.published_date) AS DATE)
      END AS date_pick,
      -- oxjob #1132: a publication_date curation from users-api (SyncWorkFieldCurations) wins over the picked date: the
      -- date the work's own source states, where every record we read gives a registration or harvest day instead (an
      -- OSTI record dated the day OSTI entered it, a DataCite deposit dated its minting year). publication_year follows.
      COALESCE(TRY_CAST(wcf.wcf_publication_date AS DATE), CASE
        WHEN COALESCE(date_pick, dfill.fill_publication_date) > ADD_MONTHS(CURRENT_DATE(), 12)
         AND YEAR(COALESCE(date_pick, dfill.fill_publication_date)) < 2400
        THEN COALESCE(TRY_CAST(ARRAY_MIN(openalex_created_dts) AS DATE), date_pick, dfill.fill_publication_date)
        ELSE COALESCE(date_pick, dfill.fill_publication_date)
      END) AS publication_date,
      YEAR(publication_date) AS publication_year,
      -- oxjob #2155: a work holding several PubMed records serves the one whose own DOI is the work's DOI, else one not
      -- typed preprint (NIH preprint pilot), else the smallest PMID string as before; ids.pmid, and through it ids.pmcid
      -- and MeSH, follow it. The smallest string alone served another record's PMID on 62,259 works (56/80 their preprint's).
      LOWER(COALESCE(raw_best_doi, GET(FILTER(collect_all_values.ids, i -> i.native_id_namespace = 'doi'), 0).native_id)) AS pubmed_pick_doi,
      CASE WHEN SIZE(pubmed_records) > 1 THEN GET(ARRAY_SORT(pubmed_records, (a, b) -> CASE
          WHEN COALESCE(a.doi = pubmed_pick_doi, FALSE) <> COALESCE(b.doi = pubmed_pick_doi, FALSE)
            THEN IF(COALESCE(a.doi = pubmed_pick_doi, FALSE), -1, 1)
          WHEN a.is_preprint <> b.is_preprint THEN IF(a.is_preprint, 1, -1)
          WHEN a.pmid < b.pmid THEN -1
          WHEN a.pmid > b.pmid THEN 1
          ELSE 0
        END), 0).pmid END AS pubmed_pick,
      MAP_FROM_ENTRIES(
        AGGREGATE(
          TRANSFORM(ids, x -> IF(x.native_id_namespace = 'pmid' AND pubmed_pick IS NOT NULL, -- oxjob #2155
            NAMED_STRUCT('native_id_namespace', x.native_id_namespace, 'native_id', pubmed_pick), x)),
          ARRAY(
            NAMED_STRUCT(
              'native_id_namespace',
              'openalex',
              'native_id',
              CONCAT(
                'https://openalex.org/W',
                CAST(collect_all_values.work_id AS STRING)
              )
            )
          ),
          (acc, x) -> CASE
            WHEN SIZE(
              FILTER(
                acc,
                y -> y.native_id_namespace = x.native_id_namespace
              )
            ) = 0 THEN acc || ARRAY(x)
            ELSE acc
          END
        )
      ) AS ids,
      -- oxjob #1057: sort the collected locations, then rebuild each one. The sort keys are the same fields either way;
      -- sorting first lets the sort read is_parent_on_work while locations_sorted keeps its struct, field for field.
      TRANSFORM(
        ARRAY_SORT(
          locations,
          (x, y) -> CASE
            WHEN STRUCT(
              -- oxjob #1057: a file whose dataset's record is on this work never leads, so the dataset is the
              -- primary location (and gives the work its type)
              CASE WHEN x.is_parent_on_work THEN 1 ELSE 0 END,
              x.sort_score,
              x.url_sort_score,
              -- Publisher-hosted beats repository-hosted (oxjob #805). Sits below the
              -- version/url terms, above provenance/native_id: it only decides ties that
              -- alphabetical native_id used to settle arbitrarily (measured 08-21: 8,873
              -- works change, all toward publisher). Non-publisher incl. sourceless NULL
              -- share rank 1 -- ranking NULL last instead moved 1.13M works to repository.
              CASE WHEN x.host_type = 'publisher' THEN 0 ELSE 1 END,
              x.provenance,
              -DATEDIFF(COALESCE(x.cr_registered, DATE'0001-01-01'), DATE'0001-01-01'),  -- oxjob #1902 F2: newest Crossref registration first
              x.native_id,
              COALESCE(x.landing_page_url, ''),
              COALESCE(x.pdf_url, '')
            ) < STRUCT(
              CASE WHEN y.is_parent_on_work THEN 1 ELSE 0 END,
              y.sort_score,
              y.url_sort_score,
              CASE WHEN y.host_type = 'publisher' THEN 0 ELSE 1 END,
              y.provenance,
              -DATEDIFF(COALESCE(y.cr_registered, DATE'0001-01-01'), DATE'0001-01-01'),
              y.native_id,
              COALESCE(y.landing_page_url, ''),
              COALESCE(y.pdf_url, '')
            ) THEN -1
            WHEN STRUCT(
              CASE WHEN x.is_parent_on_work THEN 1 ELSE 0 END,
              x.sort_score,
              x.url_sort_score,
              -- Publisher-hosted beats repository-hosted (oxjob #805). Sits below the
              -- version/url terms, above provenance/native_id: it only decides ties that
              -- alphabetical native_id used to settle arbitrarily (measured 08-21: 8,873
              -- works change, all toward publisher). Non-publisher incl. sourceless NULL
              -- share rank 1 -- ranking NULL last instead moved 1.13M works to repository.
              CASE WHEN x.host_type = 'publisher' THEN 0 ELSE 1 END,
              x.provenance,
              -DATEDIFF(COALESCE(x.cr_registered, DATE'0001-01-01'), DATE'0001-01-01'),  -- oxjob #1902 F2: newest Crossref registration first
              x.native_id,
              COALESCE(x.landing_page_url, ''),
              COALESCE(x.pdf_url, '')
            ) > STRUCT(
              CASE WHEN y.is_parent_on_work THEN 1 ELSE 0 END,
              y.sort_score,
              y.url_sort_score,
              CASE WHEN y.host_type = 'publisher' THEN 0 ELSE 1 END,
              y.provenance,
              -DATEDIFF(COALESCE(y.cr_registered, DATE'0001-01-01'), DATE'0001-01-01'),
              y.native_id,
              COALESCE(y.landing_page_url, ''),
              COALESCE(y.pdf_url, '')
            ) THEN 1
            ELSE 0
          END
        ),
        -- oxjob #1616: the work's DOI goes on its own locations when no Crossref/DataCite location holds it (~7M works
        -- whose DOI comes from a repo, MAG or PubMed record); other candidates are dropped. Same struct, field for field.
        x -> STRUCT(
          x.provenance,
          x.native_id,
          CASE WHEN x.provenance IN ('crossref', 'datacite')
                 OR (x.doi = LOWER(best_doi)
                     AND NOT EXISTS(locations, y -> y.provenance IN ('crossref', 'datacite') AND y.doi = LOWER(best_doi)))
               THEN x.doi END AS doi,
          x.sort_score, x.is_oa, x.landing_page_url, x.pdf_url, x.url_sort_score, x.oa_status, x.is_published,
          x.is_accepted, x.source, x.raw_source_name, x.raw_type, x.apc_prices, x.apc_usd, x.license, x.license_id,
          x.version, x.host_type, x.endpoint_id, x.pmh_id, x.is_unpaywall_record, x.pdf_s3_id, x.grobid_s3_id,
          x.location_type, x.updated
        )
      ) AS locations_sorted,
      -- oxjob #2509 (ruling D35: a work never holds two locations from one provider): each sorted location's arXiv paper
      -- id, when the location is arXiv's own (arXiv's source, its OAI record oai:arXiv.org:<id>, or its DataCite DOI record
      -- 10.48550/arxiv.<id>), versionless, lower case, an old-style subject class dropped (math.GT/0309136 -> math/0309136)
      -- as ids.arxiv takes it (#2177); NULL for every other location. Another repository's copy that links to arxiv.org is
      -- another provider, so it keeps its own location.
      TRANSFORM(locations_sorted, l -> CASE
        WHEN l.source.id = 'https://openalex.org/S4306400194'
          OR LOWER(COALESCE(l.native_id, '')) LIKE 'oai:arxiv.org:%'
          OR (l.provenance = 'datacite' AND LOWER(COALESCE(l.native_id, '')) LIKE '10.48550/arxiv.%')
        THEN LOWER(REGEXP_REPLACE(NULLIF(REGEXP_EXTRACT(
               REGEXP_REPLACE(REGEXP_REPLACE(COALESCE(
                 NULLIF(REGEXP_EXTRACT(COALESCE(l.native_id, ''), '(?i)^oai:arxiv\\.org:(.+)$', 1), ''),
                 NULLIF(REGEXP_EXTRACT(LOWER(COALESCE(l.native_id, '')), '^10\\.48550/arxiv\\.(.+)$', 1), ''),
                 NULLIF(REGEXP_EXTRACT(LOWER(COALESCE(l.doi, '')), '^(?:https?://(?:dx\\.)?doi\\.org/)?10\\.48550/arxiv\\.(.+)$', 1), ''),
                 NULLIF(REGEXP_EXTRACT(COALESCE(l.landing_page_url, ''), '(?i)^https?://(?:www\\.|export\\.)?arxiv\\.org/(?:abs|pdf)/(?:arxiv:)?([^?#]+?)(?:\\.pdf)?/?$', 1), ''),
                 NULLIF(REGEXP_EXTRACT(COALESCE(l.pdf_url, ''), '(?i)^https?://(?:www\\.|export\\.)?arxiv\\.org/(?:abs|pdf)/(?:arxiv:)?([^?#]+?)(?:\\.pdf)?/?$', 1), '')),
               '(?i)v[0-9]+$', ''), '^(?i)arxiv:', ''),
             '^([0-9]{4}\\.[0-9]{4,5}|[A-Za-z-]+(?:\\.[A-Za-z-]+)?/[0-9]{7})$', 1), ''), '\\.[A-Za-z-]+/', '/'))
      END) AS arxiv_keys,
      -- oxjob #2509: the locations the work serves. arXiv's OAI record, its DataCite DOI record and MAG's copies of one arXiv
      -- paper were 2-4 locations on 3,232,775 works (works_base of night 2026-11-22: 13.30M arXiv-touching locations, 9.61M
      -- after). They are now ONE location, in the place of the first of them in the sort. It is the first open one of them
      -- that is not MAG's (MAG ranks last, D16), else an open one on an arxiv.org page (so ids.arxiv still finds it), else
      -- the first open one, else the first: arXiv's OAI record on all but ~4K works, so primary_location and
      -- best_oa_location keep their id, links and version. It takes arXiv's DOI from the DataCite record as its doi, so
      -- /works/doi:10.48550/arxiv.<id>, filter=doi and ids.arxiv still find the work (dry run: 0 of 3,261,537 arXiv DOIs
      -- lost or moved to another work), and a licence, version or stored file it lacks from the others (dry run: 766 best
      -- open copies gain a licence, 73 a PDF link, none lose one). Everything else (type, dates, ids, has_content,
      -- any_repository_has_fulltext, the PMCID rule, ...) keeps reading locations_sorted, which still holds every record;
      -- only first_oa and the final SELECT's locations, locations_count, primary_location and best_oa_location read this.
      -- Two different arXiv papers on one work stay two locations (an over-merge, #1161). Readers of a DataCite location
      -- also take an arXiv-sourced location carrying arXiv's DOI: CreateWorkAuthorships deposit_byline,
      -- BuildLakebaseWorksDocs EXT_ID_CLAIMS. Revert: drop arxiv_keys and locations_served; first_oa and the final
      -- SELECT read s.locations_sorted again; drop those two readers' arXiv clause.
      IF(SIZE(FILTER(arxiv_keys, k -> k IS NOT NULL)) = SIZE(ARRAY_DISTINCT(FILTER(arxiv_keys, k -> k IS NOT NULL))),
        locations_sorted,
        TRANSFORM(
          FILTER(SEQUENCE(0, SIZE(locations_sorted) - 1),
                 i -> arxiv_keys[i] IS NULL OR ARRAY_POSITION(arxiv_keys, arxiv_keys[i]) = i + 1),
          i -> IF(arxiv_keys[i] IS NULL, locations_sorted[i],
            -- g: the paper's locations in sort order; m: the picks over g (a one-element TRANSFORM binds them once)
            GET(TRANSFORM(ARRAY(FILTER(locations_sorted, (y, j) -> arxiv_keys[j] = arxiv_keys[i])), g ->
              GET(TRANSFORM(ARRAY(NAMED_STRUCT(
                  'lead', COALESCE(
                    GET(FILTER(g, y -> COALESCE(y.is_oa, FALSE) AND y.provenance <> 'mag'), 0),
                    GET(FILTER(g, y -> COALESCE(y.is_oa, FALSE)
                                       AND COALESCE(y.landing_page_url, '') RLIKE '^https?://arxiv\\.org/(abs|pdf)/'), 0),
                    GET(FILTER(g, y -> COALESCE(y.is_oa, FALSE)), 0),
                    g[0]),
                  'dc_doi', GET(FILTER(g, y -> y.provenance = 'datacite'), 0).doi,
                  'lic', GET(FILTER(g, y -> y.license IS NOT NULL), 0),
                  'any_version', GET(FILTER(g, y -> y.version IS NOT NULL), 0).version,
                  'g', g)), m -> STRUCT(
                m.lead.provenance, m.lead.native_id,
                COALESCE(m.dc_doi, m.lead.doi) AS doi,
                m.lead.sort_score, m.lead.is_oa, m.lead.landing_page_url, m.lead.pdf_url, m.lead.url_sort_score,
                m.lead.oa_status,
                COALESCE(COALESCE(m.lead.version, m.any_version) = 'publishedVersion', FALSE) AS is_published,
                COALESCE(COALESCE(m.lead.version, m.any_version) IN ('acceptedVersion', 'publishedVersion'), FALSE) AS is_accepted,
                m.lead.source, m.lead.raw_source_name, m.lead.raw_type, m.lead.apc_prices, m.lead.apc_usd,
                IF(m.lead.license IS NOT NULL, m.lead.license, m.lic.license) AS license,
                IF(m.lead.license IS NOT NULL, m.lead.license_id, m.lic.license_id) AS license_id,
                COALESCE(m.lead.version, m.any_version) AS version,
                m.lead.host_type, m.lead.endpoint_id, m.lead.pmh_id, m.lead.is_unpaywall_record,
                COALESCE(m.lead.pdf_s3_id,
                  GET(FILTER(m.g, y -> y.pdf_s3_id IS NOT NULL AND y.is_oa <=> m.lead.is_oa), 0).pdf_s3_id) AS pdf_s3_id,
                COALESCE(m.lead.grobid_s3_id,
                  GET(FILTER(m.g, y -> y.grobid_s3_id IS NOT NULL AND y.is_oa <=> m.lead.is_oa), 0).grobid_s3_id) AS grobid_s3_id,
                m.lead.location_type,
                ARRAY_MAX(TRANSFORM(m.g, y -> y.updated)) AS updated)), 0)), 0)))) AS locations_served,
      -- oxjob #2182: the first open location's deciding fields, read once here so the best_oa_location pick in the final
      -- SELECT compares plain columns (a lookup nested in its lambda cost +40% task-hours in the shadow build, 12 Nov).
      -- oxjob #2509: from locations_served, the list best_oa_location is picked from.
      NAMED_STRUCT(
        'has_pdf', GET(FILTER(locations_served, x -> x.is_oa), 0).pdf_url IS NOT NULL,
        'is_publisher', COALESCE(GET(FILTER(locations_served, x -> x.is_oa), 0).host_type = 'publisher', FALSE),
        'version', GET(FILTER(locations_served, x -> x.is_oa), 0).version,
        'oa_status', GET(FILTER(locations_served, x -> x.is_oa), 0).oa_status,
        'license', GET(FILTER(locations_served, x -> x.is_oa), 0).license) AS first_oa,
      -- Work type comes straight from the primary location (locations_sorted[0])
      -- instead of a cross-location priority vote. Preprint-server
      -- DOI-prefix rules live in the CreateLocationsWithTypes classifier.
      -- oxjob #2138: a repository or MAG record that only says 'article' does not outrank the DOI's own DataCite record. When
      -- the primary location is such a record (and not hosted by a journal) and the DataCite location holding the work's
      -- DOI is typed dissertation (or preprint, for an arXiv DOI), the work takes that type. Zenodo (10.5281) and German
      -- Medical Science (10.3205) DOIs never do. Report and conference-paper failed #2138's judged draws here.
      -- native_id is the bare DOI, so it is compared with raw_best_doi: best_doi is the 'https://doi.org/' form, and that
      -- comparison never matched on night 2026-11-11 (#2138).
      -- oxjob #1056: a type curation (users-api POST /curations, entity works, property type; SyncWorkFieldCurations)
      -- wins over the records and every rule here. The final CASE below only re-types 'article'/'other'/NULL, so a
      -- curated paratext stays; a curated 'article' can still meet those rules (preprint server, Dialnet, rdt).
      COALESCE(wcf.wcf_type,
      CASE WHEN GET(locations_sorted, 0).location_type = 'article'
                AND GET(locations_sorted, 0).provenance IN ('repo', 'repo_backfill', 'mag')
                AND COALESCE(GET(locations_sorted, 0).source.type, '') <> 'journal'
           THEN COALESCE(GET(FILTER(locations_sorted, y -> y.provenance = 'datacite' AND LOWER(y.native_id) = LOWER(raw_best_doi)
                              AND NOT (LOWER(y.native_id) LIKE '10.5281/%' OR LOWER(y.native_id) LIKE '10.3205/%')
                              AND (y.location_type = 'dissertation'
                                   OR (y.location_type = 'preprint' AND LOWER(y.native_id) LIKE '10.48550/%'))), 0).location_type,
                         'article')
           ELSE GET(locations_sorted, 0).location_type
      END) AS type,
      -- oxjob #1815: a curated byline only where no record names anyone (#2133: one that adds authors to a record byline is in
      -- raw_authorships, the LEFT JOIN below). oxjob #2512: a work whose Crossref record lists translators (crossref_translators,
      -- cell above) gets one seat per translator the byline doesn't already show, after its last seat: author_position NULL
      -- (first and last stay with the authors), author_order_number continuing the byline's, never corresponding, no
      -- affiliations, Crossref's ORCID as raw_orcid. A byline seat is that translator by the cell above's name rule. A work
      -- no record names anyone on gets its translators alone. Works with no translators are untouched.
      -- Revert: put back the bare COALESCE and delete the tr join.
      CASE WHEN tr.tr_people IS NULL THEN COALESCE(raw_authorships.authorships, rac.curated_authorships)
           WHEN COALESCE(raw_authorships.authorships, rac.curated_authorships) IS NULL THEN TRANSFORM(tr.tr_people,
            (p, k) -> NAMED_STRUCT(
              'affiliations', CAST(ARRAY() AS ARRAY<STRUCT<institution_ids: ARRAY<STRING>, raw_affiliation_string: STRING>>),
              'author', NAMED_STRUCT('display_name', p.name, 'id', CAST(NULL AS STRING), 'orcid', CAST(NULL AS STRING),
                                     'observed_orcids', CAST(ARRAY() AS ARRAY<STRING>), 'type', CAST(NULL AS STRING)),
              'author_position', CAST(NULL AS STRING),
              'author_order_number', k,
              'countries', CAST(ARRAY() AS ARRAY<STRING>),
              'institutions', CAST(ARRAY() AS ARRAY<STRUCT<country_code: STRING, display_name: STRING, id: STRING, lineage: ARRAY<STRING>, ror: STRING, type: STRING>>),
              'is_corresponding', FALSE,
              'raw_affiliation_strings', CAST(ARRAY() AS ARRAY<STRING>),
              'raw_author_name', p.name,
              'raw_orcid', IF(p.orcid IS NULL, CAST(NULL AS STRING), CONCAT('https://orcid.org/', p.orcid))))
           ELSE CONCAT(COALESCE(raw_authorships.authorships, rac.curated_authorships), TRANSFORM(FILTER(tr.tr_people, p -> NOT COALESCE(EXISTS(
              TRANSFORM(COALESCE(raw_authorships.authorships, rac.curated_authorships), a -> FILTER(SPLIT(TRIM(REGEXP_REPLACE(REGEXP_REPLACE(TRANSLATE(LOWER(
                  CASE WHEN a.raw_author_name RLIKE '^[^,]+,[^,]+$' THEN CONCAT(TRIM(SUBSTRING_INDEX(a.raw_author_name, ',', -1)), ' ', TRIM(SUBSTRING_INDEX(a.raw_author_name, ',', 1)))
                       ELSE a.raw_author_name END),
                  'áàäâãåéèëêíìïîóòöôõøúùüûñçýćčšžłőűăşţșțđığ', 'aaaaaaeeeeiiiioooooouuuuncyccszlouaststdig'), '\\p{M}', ''), '[^\\p{L}]+', ' ')), ' '), w -> w <> '')),
              t -> COALESCE(ARRAY_JOIN(ARRAY_SORT(t), ' ') = p.k
                   OR (p.last IS NOT NULL AND LENGTH(p.last) >= 2 AND SIZE(t) >= 2 AND TRY_ELEMENT_AT(t, -1) = p.last
                       AND (TRY_ELEMENT_AT(t, 1) = p.first
                            OR (LEFT(TRY_ELEMENT_AT(t, 1), 1) = p.ini AND (p.short OR LENGTH(TRY_ELEMENT_AT(t, 1)) = 1)))), FALSE)), FALSE)),
            (p, k) -> NAMED_STRUCT(
              'affiliations', CAST(ARRAY() AS ARRAY<STRUCT<institution_ids: ARRAY<STRING>, raw_affiliation_string: STRING>>),
              'author', NAMED_STRUCT('display_name', p.name, 'id', CAST(NULL AS STRING), 'orcid', CAST(NULL AS STRING),
                                     'observed_orcids', CAST(ARRAY() AS ARRAY<STRING>), 'type', CAST(NULL AS STRING)),
              'author_position', CAST(NULL AS STRING),
              'author_order_number', SIZE(COALESCE(raw_authorships.authorships, rac.curated_authorships)) + k,
              'countries', CAST(ARRAY() AS ARRAY<STRING>),
              'institutions', CAST(ARRAY() AS ARRAY<STRUCT<country_code: STRING, display_name: STRING, id: STRING, lineage: ARRAY<STRING>, ror: STRING, type: STRING>>),
              'is_corresponding', FALSE,
              'raw_affiliation_strings', CAST(ARRAY() AS ARRAY<STRING>),
              'raw_author_name', p.name,
              'raw_orcid', IF(p.orcid IS NULL, CAST(NULL AS STRING), CONCAT('https://orcid.org/', p.orcid))))) END AS authorships,
      -- oxjob #2242: a curation (users-api POST /curations, entity works, property is_retracted; SyncWorkFieldCurations) wins
      -- over the records and the title rule. It replaced #1500's per-work override table.
      COALESCE(wcf.wcf_is_retracted, (
        is_retracted
        -- oxjob #1676: COALESCE, or a NULL title makes the flag NULL (FALSE OR NULL is NULL), as it did on the 1.69M untitled
        -- works, which then matched neither is_retracted:true nor is_retracted:false.
        OR COALESCE(lower(title) like 'retracted article%' OR lower(title) like 'retracted: %', FALSE)
      )) AS is_retracted,
      -- oxjob #1500: only the title rule flags the work (no curation, no record); read by is_retracted in the final SELECT
      wcf.wcf_is_retracted IS NULL AND NOT COALESCE(collect_all_values.is_retracted, FALSE)
        AND COALESCE(lower(title) like 'retracted article%' OR lower(title) like 'retracted: %', FALSE) AS is_retracted_title_only,
      -- oxjob #2264: the is_retracted curation alone; read by is_pulled in the final SELECT (a curated true stays pulled)
      wcf.wcf_is_retracted AS is_retracted_curated,
      -- oxjob #1186: the is_pulled curation; read by is_pulled in the final SELECT, where it wins over every signal
      wcf.wcf_is_pulled AS is_pulled_curated,
      -- oxjob #1056: title classes that are paratext; the final SELECT reads this for type AND is_paratext, so the two agree
      -- (ZD 8841). (1) ZD 2347: a journal's classified-advertising section (JAMA, Military Medicine, AJHP, C&RL News ...,
      -- ~4.1K works). Exact titles only. Bar 15 Nov: 15 of 15 drawn from the riskiest part (an abstract or a byline) are
      -- ad pages. (2) ZD 8113, 1071: a Wiley cover record (10.1002 DOI) repeats the featured article under its own DOI; its
      -- title ends in the issue tag '(Adv. Energy Mater. 42/2023)' or is 'Cover Picture', 'Titelbild: ...', 'Inside Back
      -- Cover: ...' (~47K works served as article). Bar 16 Nov: 96% of the tagged ones have a twin with the same title (tag
      -- and label dropped) under another 10.1002 DOI; a random 50 of the class all have 0 references and no abstract at
      -- Crossref; 30 of 30 twinless ones carry Wiley's cover DOI series (YYYY7NNNN) or a cover title. Revert one class:
      -- delete its lines. (3) ZD 4137 (re-pointed from #940): an IEEE front-matter record (10.1109 DOI) whose whole title,
      -- brackets dropped, names a front-matter page: '[Front cover]', '[Back cover]', 'Blank Page', 'Breaker page',
      -- '[Copyright notice]', '[Title page iii]', '[Table of contents]', 'Publisher's Information', 'Staff Listing',
      -- 'Advertisement', 'Additional Reviewers', 'Brief Author Index', 'Roster Page' ... The location classifier's
      -- 'front cover%'-style tests miss the bracketed form. 73,290 works served as article (28,239) or conference-paper
      -- (45,051) on openalex_works_base 17 Nov, none cited. Bar 17 Nov: a random 50 of the class and 20 drawn from its
      -- riskiest part (an abstract or a byline) are all front matter at Crossref (cover or roman pages, 0 references).
      -- A cover that repeats the featured paper's byline (10.1109/tpds.2007.1017, page c1) is a cover, as Wiley's are.
      COALESCE(
        lower(trim(title)) IN ('classified advertising', 'classified advertisements',
             'classified advertisement', 'classified advertisments', 'classified advertising and meetings',
             'classified advertising and meeting', 'professional opportunities: classified advertising',
             'physician opportunities: classified advertising', 'professional opportunities classified advertising',
             'classified advertising section', 'the classified advertising')
        OR (lower(raw_best_doi) LIKE '10.1002/%'
            AND (trim(title) RLIKE '[(][^()]{2,60} [0-9]{1,3}([-‐–][0-9]{1,3})?/(19|20)[0-9]{2}[)]$'
                 -- 17 Nov (ZD 8113 tail): the label in any case, plural, after a dash, or before a bare issue tag ('Cover picture',
                 -- 'Cover Pictures', 'Cover Picture – Eur. J. Immunol. 10/2005', 'Titelbild Chem. Ing. Tech. 6/2014'): +545
                 -- works; Crossref bar 20/20 cover records, 0 references.
                 -- 20 Nov (ZD 8113 tail 2): 'Inside Cover Picture' (Chin. J. Chem.), 'Inside Back Cover Picture', 'Outside Cover
                 -- Picture: ...' and the U+2010 hyphen ('Cover Picture ‐ J. Sep. Sci.'): +151 works; Crossref bar 15/15 covers.
                 OR lower(trim(title)) RLIKE '^(cover feature|front cover picture|titelbild|cover picture|inside cover|inside front cover|inside back cover|back cover|outside back cover|outside front cover|front cover|frontispiece|frontispiz|rücktitelbild|innentitelbild|innenrücktitelbild|cover image|(inside|outside|ouside)( front| back)? cover picture)s?( *[:–—‐-] *.*| [(][^()]*[0-9]{4}[)]| [^()]{2,80} [0-9]{1,3}(/[0-9]{1,3})?/(19|20)[0-9]{2})?$'))
        OR (lower(raw_best_doi) LIKE '10.1109/%'
            AND lower(trim(BOTH '[] ' FROM title)) RLIKE
                '^((inside |outside )?(front|back) cover|(front|back) inside cover( .blank.)?|front (and|&) back covers?|covers?|cover (page|art|image)s?|cover ?[1-4]|frontcover|backcover|front cover and ta(ble|bel) of contents|blank pages?|blank|breaker[- ]pages?|blank page.{0,4}back cover|back cover - advertisement|(ieee )?copyright (notice|form)|ieee copyright and consent form|([a-z0-9-]+ ){0,3}copyright page|title pages?( [ivx]+)?|half[- ]title( page)?|(ieee )?tables? of contents|front matter|back matter|publisher.?s? information|staff list(ing)?|magazine staff|advertisements?|advertisers?.? index|advertiser/product index|advertised products index|(additional |external |other |international |technical program )?reviewers( list)?|(conference |detailed |brief )?authors?.? index|keyword index|roster page)( - volume [0-9]+)?$')
        -- 18 Nov (ZD 4137): IEEE front matter whose title puts a journal, conference or session name before the label,
        -- after a bracket, a dash or a year ('IEEE Spectrum - Front cover', 'IECON 2020 Breaker Page', 'Session 9 ...
        -- [breaker page]', '[ICASI 2020 Copyright notice]'): 6,290 works article/conference-paper. Bar: Crossref random 50
        -- (17 Nov) and the riskiest 20 (most authors, most cited; 18 Nov) all front matter (roman/cover/breaker pages).
        -- The separator keeps real papers out ('... with metal back cover', 'On the Reading of Tables of Contents').
        OR (lower(raw_best_doi) LIKE '10.1109/%'
            AND lower(trim(title)) RLIKE '(\\[ ?| [-–—] |(19|20)[0-9]{2} )(front cover|back cover|inside front cover|inside back cover|blank pages?|breaker pages?|copyright notice|copyright page|tables? of contents)( ?\\])?$'),
        FALSE) AS is_paratext_title_class,
      indexed_in_crossref
    FROM
      collect_all_values
      LEFT JOIN raw_authorships_extended raw_authorships USING (work_id) -- oxjob #2133: raw_authorships with curated bylines' added authors
      LEFT JOIN raw_authorships_curated rac ON rac.work_id = collect_all_values.work_id -- oxjob #1815 (a few rows)
      LEFT JOIN (SELECT work_id AS tr_work_id, people AS tr_people FROM openalex.works.crossref_translators) tr -- oxjob #2512 (~52K rows)
        ON tr.tr_work_id = collect_all_values.work_id
      LEFT JOIN abstracts_backfill bf USING (work_id)
      LEFT JOIN openalex.works.xpac_work_ids xpac ON xpac.work_id = collect_all_values.work_id
      LEFT JOIN openalex.works.publication_date_fill dfill ON dfill.work_id = collect_all_values.work_id -- oxjob #1644
      LEFT JOIN openalex.works.crossref_year_only_pubmed_dates ypd ON ypd.work_id = collect_all_values.work_id -- oxjob #2241
      LEFT JOIN openalex.works.placeholder_publication_dates ppd ON ppd.work_id = collect_all_values.work_id -- oxjob #1154
      LEFT JOIN openalex.works.mag_outranked_publication_dates mod ON mod.work_id = collect_all_values.work_id -- oxjob #2383
      LEFT JOIN openalex.works.preprint_outranked_publication_dates pod ON pod.work_id = collect_all_values.work_id -- oxjob #2097
      LEFT JOIN openalex.works.repec_year_publication_dates rpd ON rpd.work_id = collect_all_values.work_id -- oxjob #1154 (repec_year)
      LEFT JOIN openalex.works.padded_year_doi_dates pyd ON pyd.work_id = collect_all_values.work_id -- oxjob #1154 (step 6)
      LEFT JOIN ( -- oxjob #2242: one row per curated work, the latest curation per field (SyncWorkFieldCurations)
        SELECT work_id AS wcf_work_id, is_retracted AS wcf_is_retracted, type AS wcf_type, -- type: oxjob #1056
          is_xpac AS wcf_is_xpac, -- oxjob #1125
          is_pulled AS wcf_is_pulled, -- oxjob #1186
          curated_fields AS wcf_curated_fields, biblio_volume AS wcf_biblio_volume, biblio_issue AS wcf_biblio_issue, -- oxjob #1103
          biblio_first_page AS wcf_biblio_first_page, biblio_last_page AS wcf_biblio_last_page
               , publication_date AS wcf_publication_date -- publication_date: oxjob #1132
               , title AS wcf_title -- title: oxjob #1162
               , language AS wcf_language -- language: oxjob #1703
        FROM openalex.works.work_curated_fields
      ) wcf ON wcf.wcf_work_id = collect_all_values.work_id
  )
  SELECT /*+ BROADCAST(tcf), BROADCAST(pcn), BROADCAST(jcp), BROADCAST(fpp), BROADCAST(rdt), BROADCAST(ctc), BROADCAST(rn), BROADCAST(cps), BROADCAST(rwx), BROADCAST(awd), BROADCAST(caw) */
    s.id,
    s.best_doi AS doi,
    -- oxjob #1910: a title holding a control character takes its clean form: the fix table's row (built above) where a
    -- byte stands for a real character, else the plain SQL drop. Titles without one are untouched. After the vote, so
    -- the winning location never changes.
    CASE WHEN s.title RLIKE '[\\x{0000}-\\x{001F}\\x{007F}-\\x{009F}]'
         THEN COALESCE(tcf.clean_title, openalex.common.drop_title_controls(s.title))
         ELSE s.title END AS title,
    COALESCE(s.authorships, ARRAY()) as authorships,
    SIZE(COALESCE(s.authorships, ARRAY())) AS authors_count,
    CAST(
      ARRAY() AS ARRAY < STRUCT <
      id STRING,
      display_name STRING,
      ror STRING,
      country_code STRING,
      type STRING,
      lineage ARRAY<STRING> > >
    ) AS institutions,
    CAST(0 AS INT) AS institutions_distinct_count,
    CAST(0 AS INT) AS countries_distinct_count,
    -- @TODO make changes later to calculate them upstream - they will have to rely on Walden logic
    CAST(ARRAY() AS ARRAY<STRING>) AS corresponding_author_ids,
    CAST(ARRAY() AS ARRAY<STRING>) AS corresponding_institution_ids,
    s.publication_date,
    s.publication_year,
    s.created_date,
    s.updated_date,
    s.abstract,
    s.abstract_inverted_index IS NOT NULL AS has_abstract,
    s.referenced_works_count,
    s.referenced_works,
    CAST(ARRAY() AS ARRAY < STRING >) AS related_works,
    s.abstract_inverted_index,
    CONCAT(
      'https://api.openalex.org/works?filter=cites:W',
      CAST(s.id AS STRING)
    ) AS cited_by_api_url,
    CAST(0 AS INT) as cited_by_count,
    CAST(
      ARRAY() AS ARRAY < STRUCT < year: INT,
      cited_by_count: INT > >
    ) AS counts_by_year,
    CAST(NULL AS DOUBLE) as fwci,
    CAST(
      NULL AS STRUCT < value DOUBLE,
      is_in_top_1_percent BOOLEAN,
      is_in_top_10_percent BOOLEAN >
    ) as citation_normalized_percentile,
    CAST(NULL AS STRUCT < min INT, max INT >) as cited_by_percentile_year,
    MAP_FILTER(
      TRANSFORM_VALUES(
        -- oxjob #1862: pmcid, from the PubMed record whose PMID the work serves (join `pmc` below). NULL drops out.
        MAP_CONCAT(CASE
          WHEN s.ids['doi'] IS NOT NULL OR s.best_doi IS NULL THEN
            MAP_CONCAT(
              MAP_FILTER(s.ids, (k, v) -> k != 'mag'),
              map('mag', COALESCE(
                IF(id < 4200000000, CAST(id AS STRING), NULL),
                s.ids['mag']
              ))
            )
          ELSE
            MAP_CONCAT(
              MAP_FILTER(s.ids, (k, v) -> k != 'mag'),
              map('mag', COALESCE(
                IF(id < 4200000000, CAST(id AS STRING), NULL),
                s.ids['mag']
              )),
              map('doi', regexp_replace(s.best_doi, '^https://doi.org/', ''))
            )
        -- oxjob #1163: no PMCID from the PubMed record? Then the PMCID of the work's own PMC copy, where NCBI's PMC-ids row ties it
        -- to this work: (A) NCBI lists the work's DOI or PMID; (B) NCBI lists neither, the work has neither, and the PMC record
        -- is the work's primary location (the PMC article defines the work); (D) NCBI lists neither, and NCBI's year, volume
        -- and first page (or the sub-item 718.3 on page 718) and the PMC title equal the work's. Never when NCBI names another
        -- DOI or PMID, when the PMCID sits on another work, or when two PMCIDs qualify (AGGREGATE returns '' then).
        -- Bar 10 Nov 2026 (seeds 11632, 11633): 400/400 vs NCBI's live records.
        END, map('pmcid', COALESCE(pmc.pmcid_digits, NULLIF(AGGREGATE(
          FILTER(pcn.copies, c ->
            NOT ((c.ncbi_doi IS NOT NULL
                      AND LOWER(COALESCE(REGEXP_REPLACE(s.best_doi, '^https://doi.org/', ''), s.ids['doi'])) <> c.ncbi_doi)
                  OR (c.ncbi_pmid IS NOT NULL AND s.ids['pmid'] <> c.ncbi_pmid))
            AND (COALESCE(LOWER(COALESCE(REGEXP_REPLACE(s.best_doi, '^https://doi.org/', ''), s.ids['doi'])) = c.ncbi_doi, FALSE)
              OR COALESCE(s.ids['pmid'] = c.ncbi_pmid, FALSE)
              OR (c.ncbi_doi IS NULL AND c.ncbi_pmid IS NULL
                  AND ((s.best_doi IS NULL AND s.ids['doi'] IS NULL AND s.ids['pmid'] IS NULL
                        AND CONCAT('oai:pubmedcentral.nih.gov:', c.pmc_digits) = GET(s.locations_sorted, 0).native_id)
                    OR COALESCE(c.title_hash = XXHASH64(LOWER(REGEXP_REPLACE(s.title, '[^\\p{L}\\p{N}]', '')))
                                AND c.ncbi_year = s.publication_year
                                AND LOWER(c.ncbi_volume) = LOWER(TRIM(s.volume))
                                -- (D2) a sub-page item: Crossref's 718.3 is the third item on NCBI's page 718
                                AND (LOWER(c.ncbi_page) = LOWER(TRIM(s.first_page))
                                  OR (TRIM(s.first_page) RLIKE '^[0-9]+[.][0-9]+$'
                                      AND c.ncbi_page = SUBSTRING_INDEX(TRIM(s.first_page), '.', 1))), FALSE))))),
          CAST(NULL AS STRING),
          (acc, c) -> CASE WHEN acc IS NULL OR acc = c.pmc_digits THEN c.pmc_digits ELSE '' END), ''))),
        -- oxjob #2177 (ruling D35, Jason 21 Nov 2026): arxiv, the work's arXiv id, one per work, versionless; NULL drops out.
        -- The work's own DOI when it is arXiv's (10.48550/arxiv.<id>), so ids.doi and ids.arxiv name one paper; else the
        -- first arXiv location in the work's location order: its OAI record (oai:arXiv.org:<id>), arXiv's DOI record, or an
        -- arxiv.org/abs|pdf URL. Kept only in arXiv's id grammar (YYMM.NNNN[N] or archive/YYMMNNN); an old-style
        -- subject class (math.GT/0309136) is dropped, as arXiv's canonical id does. Two different arXiv papers on one work
        -- are over-merges (#1161), and two records of one paper are #2509's.
        map('arxiv', LOWER(REGEXP_REPLACE(COALESCE(
          NULLIF(REGEXP_EXTRACT(LOWER(COALESCE(REGEXP_REPLACE(s.best_doi, '^https://doi.org/', ''), s.ids['doi'], '')),
            '^(?:https://doi\\.org/)?10\\.48550/arxiv\\.([0-9]{4}\\.[0-9]{4,5}|[a-z-]+(?:\\.[a-z-]+)?/[0-9]{7})(?:v[0-9]+)?$', 1), ''),
          GET(FILTER(TRANSFORM(
            FILTER(s.locations_sorted, l -> l.source.id = 'https://openalex.org/S4306400194'
                                            OR LOWER(COALESCE(l.doi, l.native_id, '')) RLIKE '10\\.48550/arxiv\\.'),
            l -> REGEXP_REPLACE(REGEXP_REPLACE(COALESCE(
                   NULLIF(REGEXP_EXTRACT(COALESCE(l.native_id, ''), '(?i)^oai:arxiv\\.org:(.+)$', 1), ''),
                   NULLIF(REGEXP_EXTRACT(LOWER(COALESCE(l.doi, l.native_id, '')),
                     '^(?:https?://(?:dx\\.)?doi\\.org/)?10\\.48550/arxiv\\.(.+)$', 1), ''),
                   NULLIF(REGEXP_EXTRACT(COALESCE(l.landing_page_url, ''),
                     '(?i)^https?://(?:www\\.|export\\.)?arxiv\\.org/(?:abs|pdf)/(?:arxiv:)?([^?#]+?)(?:\\.pdf)?/?$', 1), ''),
                   NULLIF(REGEXP_EXTRACT(COALESCE(l.pdf_url, ''),
                     '(?i)^https?://(?:www\\.|export\\.)?arxiv\\.org/(?:abs|pdf)/(?:arxiv:)?([^?#]+?)(?:\\.pdf)?/?$', 1), '')),
                 '(?i)v[0-9]+$', ''), '^(?i)arxiv:', '')),
            x -> x RLIKE '^([0-9]{4}\\.[0-9]{4,5}|[A-Za-z-]+(\\.[A-Za-z-]+)?/[0-9]{7})$'), 0)),
          '\\.[A-Za-z-]+/', '/')))),
        (k, v) -> CASE
          -- Override ids.doi value with canonical best_doi (preserves original key position).
          WHEN k = 'doi' AND s.best_doi IS NOT NULL THEN regexp_replace(s.best_doi, '^https://doi.org/', '')
          WHEN k = 'pmid' THEN CONCAT('https://pubmed.ncbi.nlm.nih.gov/', v)
          -- oxjob #1862: the legacy form (digits, no PMC prefix), which the API's pmcid: lookup builds; NCBI redirects it.
          WHEN k = 'pmcid' THEN CONCAT('https://www.ncbi.nlm.nih.gov/pmc/articles/', v)
          -- oxjob #2177: arXiv's abstract page, the URI form like pmid and pmcid.
          WHEN k = 'arxiv' THEN CONCAT('https://arxiv.org/abs/', v)
          ELSE v
        END
      ),
      (k, v) -> v IS NOT NULL
    ) as ids,
    -- transform_values(ids, (k, v) -> case when k = 'pmid' then CONCAT('https://pubmed.ncbi.nlm.nih.gov/', v) else v end)
    ARRAY_SORT(
      ARRAY_DISTINCT(
        ARRAY_COMPACT(
          FLATTEN(
            TRANSFORM(
              s.locations_sorted,
              loc -> CASE
                -- oxjob #2184: 'doaj' means DOAJ's article index holds this work (the oai:doaj.org record below),
                -- never that its journal is a DOAJ member; membership is source.is_in_doaj.
                WHEN loc.provenance IN ('crossref', 'pubmed', 'datacite') THEN array(loc.provenance)
                WHEN loc.provenance = 'repo'
                AND lower(loc.native_id) like 'oai:arxiv.org%' THEN array('arxiv')
                WHEN loc.provenance = 'repo'
                AND lower(loc.native_id) like 'oai:doaj.org/%' THEN array('doaj')
                WHEN loc.provenance = 'mag'
                AND lower(loc.source.display_name) = 'pubmed' THEN array('pubmed')
                ELSE array()
              END
            )
          )
        )
      )
    ) as indexed_in,
    s.language,
    s.publisher,
    STRUCT(
      -- oxjob #1098 step 2c: a work no record, curation or repository citation gives a volume takes the one the landing pages
      -- of its repository records state (rpv, cell above), and their issue with it when it has none; a page volume whose
      -- issue contradicts the work's own issue is not used. Beside a record's volume the pages agree 98.95% (22 Nov 2026).
      -- Revert: back to `STRUCT(s.volume, s.issue,` and drop the rpv join below.
      COALESCE(s.volume, IF(s.issue IS NULL OR rpv.issue IS NULL OR s.issue = rpv.issue, rpv.volume, NULL)) AS volume, -- oxjob #1098
      COALESCE(s.issue, IF(s.volume IS NULL, rpv.issue, NULL)) AS issue, -- oxjob #1098
      -- oxjob #1098 step 4: a work no record (or curation) gives a page takes the one range its repository citations state
      -- (s.repo_cited_pages); a Crossref article number still comes first. Beside a record's own first page these ranges
      -- agree 98.6% (8,415,653 of 8,537,540, 21 Nov 2026). Revert: these two COALESCEs back to
      -- `COALESCE(s.first_page, IF(s.last_page IS NULL, crn.article_number, NULL))` and `s.last_page`, then drop
      -- repo_cited_pages, repo_source_page_ranges and repo_source_pages.
      COALESCE(s.first_page, IF(s.last_page IS NULL, COALESCE(crn.article_number, s.repo_cited_pages.first_page), NULL)) AS first_page, -- oxjob #1098
      COALESCE(s.last_page, IF(s.first_page IS NULL AND crn.article_number IS NULL, s.repo_cited_pages.last_page, NULL)) AS last_page) AS biblio,
    -- volume (oxjob #1554): the work this work is part of, e.g. a chapter's book. A property of the work, learned from
    -- its Crossref record(s) in CreateWorkVolumes: the primary location's link when it has one, else the lowest book id.
    CASE WHEN wv.volumes IS NOT NULL THEN NAMED_STRUCT(
      'id', CONCAT('https://openalex.org/W', CAST(COALESCE(
        GET(FILTER(wv.volumes, v -> v.native_id = LOWER(GET(s.locations_sorted, 0).native_id)), 0),
        ARRAY_MIN(wv.volumes)).volume_work_id AS STRING)),
      'display_name', COALESCE(
        GET(FILTER(wv.volumes, v -> v.native_id = LOWER(GET(s.locations_sorted, 0).native_id)), 0),
        ARRAY_MIN(wv.volumes)).volume_display_name
    ) END AS volume,
    CASE
      /* Exact publication-year hit: keep the current Work currency when that
         publisher-original currency exists for the same year; otherwise use
         a coherent all-USD tuple. Zero is a real observed price. */
      WHEN pps.apc_prices_by_year IS NOT NULL
        AND TRY_ELEMENT_AT(pps.apc_usd_by_year, CAST(s.publication_year AS STRING)) >= 0 THEN
        COALESCE(
          GET(
            TRANSFORM(
              FILTER(
                TRY_ELEMENT_AT(pps.apc_prices_by_year, CAST(s.publication_year AS STRING)),
                p -> p.price >= 0
                  AND UPPER(p.currency) = UPPER(GET(GET(s.locations_sorted, 0).apc_prices, 0).currency)
              ),
              p -> STRUCT(
                CAST(BROUND(p.price) AS INT) AS value,
                p.currency AS currency,
                TRY_ELEMENT_AT(pps.apc_usd_by_year, CAST(s.publication_year AS STRING)) AS value_usd
              )
            ),
            0
          ),
          STRUCT(
            TRY_ELEMENT_AT(pps.apc_usd_by_year, CAST(s.publication_year AS STRING)) AS value,
            'USD' AS currency,
            TRY_ELEMENT_AT(pps.apc_usd_by_year, CAST(s.publication_year AS STRING)) AS value_usd
          )
        )
      /* No exact historical year: preserve today's complete list-price tuple. */
      WHEN GET(GET(s.locations_sorted, 0).apc_prices, 0).price IS NOT NULL THEN STRUCT(
        GET(GET(s.locations_sorted, 0).apc_prices, 0).price AS value,
        GET(GET(s.locations_sorted, 0).apc_prices, 0).currency AS currency,
        GET(s.locations_sorted, 0).apc_usd AS value_usd
      )
    END AS apc_list,
    CASE
      /* OpenAPC takes precedence whenever present */
      WHEN o.apc_in_euro IS NOT NULL OR o.apc_in_usd IS NOT NULL THEN STRUCT(
        o.apc_in_euro AS value,
        'EUR'         AS currency,
        o.apc_in_usd  AS value_usd
      )
      /* Otherwise: existing gold/hybrid estimate follows the resolved list price. */
      WHEN GET(FILTER(s.locations_sorted, x -> x.is_oa), 0).oa_status IN (2, 3)
        AND apc_list IS NOT NULL
        THEN apc_list
        ELSE NULL
    END AS apc_paid,
    CAST(
      NULL AS STRUCT < id STRING,
      display_name STRING,
      score FLOAT,
      subfield STRUCT < id STRING,
      display_name STRING >,
      field STRUCT < id STRING,
      display_name STRING >,
      domain STRUCT < id STRING,
      display_name STRING > >
    ) AS primary_topic,
    CAST(
      ARRAY() AS ARRAY < STRUCT < id STRING,
      display_name STRING,
      score FLOAT,
      subfield STRUCT < id STRING,
      display_name STRING >,
      field STRUCT < id STRING,
      display_name STRING >,
      domain STRUCT < id STRING,
      display_name STRING > > >
    ) AS topics,
    -- oxjob #1910: s.title (the raw winner), so removing control characters does not re-run topics on 865K works
    -- oxjob #2264: s.best_doi, not a bare doi: a joined table's doi column took the bare name (cps.doi on night 11-17,
    -- so the key lost the DOI) and then made it ambiguous. Same value as before night 11-17. Read nowhere, in no content hash.
    xxhash64(concat_ws('|', s.best_doi, s.title, abstract,
        COALESCE(
            GET(s.locations_sorted, 0).source.display_name,
            GET(s.locations_sorted, 0).raw_source_name
        )
    )) as topics_key,
    CAST(
      ARRAY() AS ARRAY < STRUCT < id STRING,
      display_name STRING,
      score FLOAT > >
    ) AS keywords,
    CAST(
      ARRAY() AS ARRAY < STRUCT < id BIGINT,
      wikidata STRING,
      display_name STRING,
      level INT,
      score FLOAT > >
    ) AS concepts,
    -- oxjob #2509: locations_count, locations, primary_location and best_oa_location read locations_served (one location
    -- per arXiv paper; see set_fields).
    SIZE(s.locations_served) AS locations_count,
    TRANSFORM(
      -- oxjob #2209: the journal copy served as primary_location (below, same CASE) leads the list; the rest keep the
      -- sort's order. A NULL pick leaves the order as it is (nothing equals NULL, so the first FILTER is empty).
      -- oxjob #1225: else the institution-portal copy of a Figshare item (below, same CASE).
      IF(jcp.work_id IS NULL AND fpp.work_id IS NULL, s.locations_served,
         CONCAT(
           FILTER(s.locations_served, x -> x <=> COALESCE(CASE WHEN jcp.work_id IS NOT NULL
                     AND COALESCE(GET(s.locations_served, 0).source.type, 'repository') IN ('repository', 'other')
                     AND NOT EXISTS(s.locations_served, y -> y.provenance = 'crossref')
                THEN GET(FILTER(s.locations_served, y -> y.source.type IN ('journal', 'conference')
                                  AND (y.provenance <> 'repo' OR y.host_type = 'publisher')
                                  AND ARRAY_CONTAINS(jcp.pick_keys, XXHASH64(y.provenance, y.native_id))), 0) END,
                CASE WHEN fpp.work_id IS NOT NULL
                     AND GET(s.locations_served, 0).source.id = 'https://openalex.org/S4377196282'
                     AND ARRAY_CONTAINS(fpp.figshare_native_ids, GET(s.locations_served, 0).native_id)
                THEN GET(FILTER(s.locations_served, y -> y.provenance = 'datacite'
                                  AND ARRAY_CONTAINS(fpp.pick_native_ids, y.native_id)), 0) END)),
           FILTER(s.locations_served, x -> NOT (x <=> COALESCE(CASE WHEN jcp.work_id IS NOT NULL
                     AND COALESCE(GET(s.locations_served, 0).source.type, 'repository') IN ('repository', 'other')
                     AND NOT EXISTS(s.locations_served, y -> y.provenance = 'crossref')
                THEN GET(FILTER(s.locations_served, y -> y.source.type IN ('journal', 'conference')
                                  AND (y.provenance <> 'repo' OR y.host_type = 'publisher')
                                  AND ARRAY_CONTAINS(jcp.pick_keys, XXHASH64(y.provenance, y.native_id))), 0) END,
                CASE WHEN fpp.work_id IS NOT NULL
                     AND GET(s.locations_served, 0).source.id = 'https://openalex.org/S4377196282'
                     AND ARRAY_CONTAINS(fpp.figshare_native_ids, GET(s.locations_served, 0).native_id)
                THEN GET(FILTER(s.locations_served, y -> y.provenance = 'datacite'
                                  AND ARRAY_CONTAINS(fpp.pick_native_ids, y.native_id)), 0) END))))),
      x -> STRUCT(
        COALESCE(x.is_oa, FALSE) AS is_oa,
        x.landing_page_url,
        x.pdf_url,
        COALESCE(x.is_published, FALSE) AS is_published,
        COALESCE(x.is_accepted, FALSE) AS is_accepted,
        x.source,
        x.raw_source_name,
        x.raw_type,
        x.license,
        x.license_id,
        x.version,
        x.oa_status,
        x.host_type,
        x.endpoint_id,
        x.pmh_id,
        COALESCE(x.is_unpaywall_record, FALSE) AS is_unpaywall_record,
        x.location_type,
        x.pdf_s3_id,
        x.grobid_s3_id,
        CAST(x.updated AS TIMESTAMP) AS updated,
        x.provenance,
        x.native_id,
        x.doi
      )
    ) AS locations,
    -- oxjob #2209: a work with no Crossref location whose sort put a repository / aggregator copy (or one with no source)
    -- first, though it holds its journal's own copy and another record on the work names that journal (picks table,
    -- built in the cell above, joined below), serves the journal copy as primary_location. Only primary_location and that
    -- location's place in `locations` move: type, best_oa_location, open_access, apc_list, topics_key, volume and the
    -- PMCID rule keep reading locations_sorted, so none of them changes (each needs its own bar).
    -- Revert: restore `GET(s.locations_sorted, 0) AS primary_location` and `s.locations_sorted` in the locations
    -- TRANSFORM, remove the jcp join and hint; drop the cell above and its table.
    COALESCE(
      CASE WHEN jcp.work_id IS NOT NULL
                AND COALESCE(GET(s.locations_served, 0).source.type, 'repository') IN ('repository', 'other')
                AND NOT EXISTS(s.locations_served, y -> y.provenance = 'crossref')
           -- oxjob #2494: a conference series' own copy too (same picks table; see its journals CTE).
           THEN GET(FILTER(s.locations_served, y -> y.source.type IN ('journal', 'conference')
                             AND (y.provenance <> 'repo' OR y.host_type = 'publisher')
                             AND ARRAY_CONTAINS(jcp.pick_keys, XXHASH64(y.provenance, y.native_id))), 0) END,
      -- oxjob #1225: an item on an institution's Figshare portal whose figshare.com record leads the sort (it states a
      -- version, the institution's DataCite copy none) serves the institution's copy instead (picks table, cell above).
      -- Same scope as #2209: only primary_location and its place in `locations` move.
      CASE WHEN fpp.work_id IS NOT NULL
                AND GET(s.locations_served, 0).source.id = 'https://openalex.org/S4377196282'
                AND ARRAY_CONTAINS(fpp.figshare_native_ids, GET(s.locations_served, 0).native_id)
           THEN GET(FILTER(s.locations_served, y -> y.provenance = 'datacite'
                             AND ARRAY_CONTAINS(fpp.pick_native_ids, y.native_id)), 0) END,
      GET(s.locations_served, 0)) AS primary_location,
    -- oxjob #2182: the best OA location is the first open location in the sort, with one exception. The sort ranks any
    -- Crossref copy first whatever its PDF link (#589), so 554K works served a best open copy with no pdf_url while an
    -- equal open copy had one (and Unpaywall's url_for_pdf was null). When the first open copy has no PDF link, the first
    -- open copy WITH one is served instead, if it has the same host kind (publisher or not), version and OA status, and
    -- the same licence (or the first copy states none): the docs' order is host kind, then version, then PDF link.
    -- So oa_status never changes and no licence is lost; primary_location and the locations order are untouched.
    -- Never another Crossref record: where a work holds two DOIs, which DOI is best is not this rule's call, and the bar
    -- sample (12 Nov, seed 2182) found those 'PDF links' are often pages (21K AACR figshare items).
    -- Revert: restore the one line `GET(FILTER(s.locations_sorted, x -> x.is_oa), 0) AS best_oa_location`.
    COALESCE(
      CASE WHEN NOT s.first_oa.has_pdf THEN
        GET(FILTER(s.locations_served, x -> x.is_oa AND x.pdf_url IS NOT NULL AND x.provenance <> 'crossref'
          AND COALESCE(x.host_type = 'publisher', FALSE) = s.first_oa.is_publisher
          AND x.version <=> s.first_oa.version
          AND x.oa_status <=> s.first_oa.oa_status
          AND (x.license <=> s.first_oa.license OR s.first_oa.license IS NULL)), 0)
      END,
      GET(FILTER(s.locations_served, x -> x.is_oa), 0)
    ) AS best_oa_location,
    -- oxjob #1987 (#1959 D6): a work has content when ANY open location holds the file, not only the best OA
    -- location. The Content API (D1 content_index) and the manifest already serve a file from any location, so
    -- the flag (and content_urls, built from it) hid files we serve: 2,729,584 works on 3 Nov 2026, most of them
    -- arXiv preprints whose best OA copy is the journal's page or arXiv's DataCite DOI record. None lose the flag.
    -- has_content is not in the content hash: the works that flip are re-stamped by flipping their stored
    -- content_hash before the night (#1987 work/d6_flip_hash.sql). Revert: restore the best-OA-only CASEs.
    STRUCT(
      -- oxjob #2430: only an open location's file counts; a closed location's file (#1006 dead host, a curation) moves to
      -- the private bucket and leaves the manifest and content API (utils/content_files.py). Was: dead-host files counted.
      COALESCE(EXISTS(s.locations_sorted, x -> COALESCE(x.is_oa, FALSE) AND x.pdf_s3_id IS NOT NULL), FALSE) AS pdf,
      COALESCE(EXISTS(s.locations_sorted, x -> COALESCE(x.is_oa, FALSE) AND x.grobid_s3_id IS NOT NULL), FALSE) AS `grobid_xml`
    ) AS has_content,
    CAST(
      ARRAY() AS ARRAY < STRUCT < id STRING,
      display_name STRING,
      score FLOAT > >
    ) AS sustainable_development_goals,
    -- study_designs (oxjob #1312): filled by CreateWorksEnriched from openalex.works.works_study_design
    CAST(
      ARRAY() AS ARRAY < STRUCT < id STRING,
      display_name STRING > >
    ) AS study_designs,
    CAST(
      ARRAY() AS ARRAY < STRUCT < id STRING,
      display_name STRING,
      funder_award_id STRING,
      funder_id STRING,
      funder_display_name STRING > >
    ) AS awards,
    CAST(
      ARRAY() AS ARRAY < STRUCT < id STRING,
      display_name STRING,
      ror STRING > >
    ) AS funders,
    STRUCT(
      best_oa_location IS NOT NULL AS is_oa,
      -- oxjob #2308: gold and diamond mean "published in a fully-OA journal" and green means "toll-access at the
      -- publisher, free in a repository" (help.openalex.org, Open access). best_oa_location ranks a repository copy with a
      -- version and a PDF (PubMed Central, a portal) above a journal copy without them, so works whose primary copy is open
      -- in a fully-OA journal read green (269K on night 2026-11-22). When the primary location is open, has a source, its
      -- own status is diamond or gold, the best OA copy's is not, and DOAJ lists that journal from the work's year on (the
      -- registry of fully-OA journals; the high-OA-rate flag alone misjudged subscription journals, and works before a
      -- journal's DOAJ start are hybrid or bronze by the docs: #2308 bars v1-v2), the work takes the primary's label.
      -- best_oa_location, oa_url, primary_location, apc_list and apc_paid do not change. The year lookup comes last, so it
      -- runs only on the few works the earlier terms let through. Revert: delete this WHEN.
      CASE
        WHEN COALESCE(primary_location.is_oa, FALSE) AND primary_location.source.id IS NOT NULL
             AND primary_location.oa_status IN (1, 2) AND NOT COALESCE(best_oa_location.oa_status IN (1, 2), FALSE)
             AND COALESCE(s.publication_year >= ELEMENT_AT(
               (SELECT MAP_FROM_ENTRIES(COLLECT_LIST(STRUCT(k, v))) FROM (
                  SELECT CONCAT('https://openalex.org/S', CAST(id AS STRING)) AS k, MAX(is_in_doaj_start_year) AS v
                  FROM openalex.sources.sources WHERE is_in_doaj AND is_in_doaj_start_year IS NOT NULL GROUP BY 1)),
               primary_location.source.id), FALSE)
          THEN IF(primary_location.oa_status = 1, 'diamond', 'gold')
        WHEN best_oa_location.oa_status = 1 THEN 'diamond'
        WHEN best_oa_location.oa_status = 2 THEN 'gold'
        WHEN best_oa_location.oa_status = 3 THEN 'hybrid'
        WHEN best_oa_location.oa_status = 4 THEN 'bronze'
        WHEN best_oa_location.oa_status = 5 THEN 'green'
        ELSE 'closed'
      END AS oa_status,
      COALESCE(
        best_oa_location.pdf_url,
        best_oa_location.landing_page_url
      ) AS oa_url,
      -- oxjob #1096 (ZD 256): a boolean, never null. EXISTS gave NULL when an open copy had no source (host_type NULL) and
      -- no repository copy was open: 9,871,084 works on 15 Nov 2026 (1,516,231 outside xpac; 2.04M green). A copy the
      -- status rule calls green (oa_status 5: a repository-type record or a preprint with no known journal, #1867) is a
      -- repository copy here too, so a green work always has repository full text. Only the NULLs change (pre-flight on
      -- openalex_works_base: every TRUE stays TRUE, every FALSE stays FALSE). Revert: the plain EXISTS on host_type.
      COALESCE(EXISTS(
        s.locations_sorted,
        x -> COALESCE(x.is_oa, FALSE)
             AND (COALESCE(x.host_type = 'repository', FALSE) OR COALESCE(x.oa_status = 5, FALSE))
      ), FALSE) AS any_repository_has_fulltext
    ) AS open_access,
    -- Override type to 'preprint' when the primary location's source is a preprint
    -- repository (openalex.sources.sources.is_preprint_repository), but only for
    -- ambiguous current types -- never overwrite dataset/book/dissertation/etc.
    CASE
      -- oxjob #1056 (ZD 2347, 8113, 1071, 4137): an article or conference paper in a paratext title class (is_paratext_title_class above) is paratext.
      -- First, so is_paratext below can read the same test. Revert: delete this WHEN and restore is_paratext below.
      WHEN s.type IN ('article', 'conference-paper') AND s.is_paratext_title_class
      THEN 'paratext'
      -- oxjob #1056 (ZD 27813): Üsküdar University registers every story of its news agency 'üha' (ISSN 2980-1974) as a
      -- Crossref journal-article under DOI 10.32739/uha.id.N: 7,320 news items served as article or conference-abstract.
      -- News items are type other (help.openalex.org/data/work-types; The Conversation's stories are other). Bar 19 Nov:
      -- a random 50 and the riskiest 20 (byline, references, most cited) all news or web articles, 0 references. The
      -- university's journal (10.32739/uha.jnbs.*) is not in the class. Revert: delete this WHEN.
      WHEN s.type IN ('article', 'conference-abstract') AND lower(s.raw_best_doi) LIKE '10.32739/uha.id.%'
      THEN 'other'
      WHEN COALESCE(pps.is_preprint_repository, FALSE)
           AND (s.type IN ('article', 'other') OR s.type IS NULL)
      THEN 'preprint'
      -- oxjob #2227: Dialnet's own servlet path names the form of a record that only says 'article' (all are MAG crawl records): /servlet/libro
      -- is a whole book, /servlet/tesis a doctoral thesis (164K and 113K works served as article; bar 14 Nov: a fresh random
      -- 100 vs Dialnet's own pages). Only when that record leads the sort, and never for a work #2209 hands to its journal
      -- copy (jcp), whose type that rule keeps. Revert: delete these two WHEN lines.
      WHEN s.type = 'article' AND jcp.work_id IS NULL
           AND GET(s.locations_sorted, 0).landing_page_url LIKE 'http%://dialnet.unirioja.es/servlet/libro?%'
      THEN 'book'
      WHEN s.type = 'article' AND jcp.work_id IS NULL
           AND GET(s.locations_sorted, 0).landing_page_url LIKE 'http%://dialnet.unirioja.es/servlet/tesis?%'
      THEN 'dissertation'
      -- oxjob #2227 phase 2: a Dialnet /servlet/articulo page is a journal article OR a chapter; Dialnet files a chapter as a
      -- part of a book (ISBN, no ISSN). Its MAG crawl record then names no venue, and the work has no volume and no issue
      -- (930K works). Bar 14 Nov: a fresh random 100 of them, 100 parts of books on Dialnet's own pages. Revert: delete this WHEN.
      WHEN s.type = 'article' AND jcp.work_id IS NULL AND s.volume IS NULL AND s.issue IS NULL
           AND GET(s.locations_sorted, 0).provenance = 'mag'
           AND COALESCE(GET(s.locations_sorted, 0).raw_source_name, '') = ''
           AND GET(s.locations_sorted, 0).landing_page_url LIKE 'http%://dialnet.unirioja.es/servlet/articulo?%'
      THEN 'book-chapter'
      -- oxjob #1160 (ZD 8479, 8482, 8483): MDZ digitizations of old dissertations (Bayerische Staatsbibliothek, landing pages
      -- urn:nbn:de:bvb:12-bsb...) are typed book or article: the BVB catalogue record we harvest says nothing more, and BSB marks
      -- a thesis only in its MARC note. The title says it: 'Inaug.-Diss.', 'Diss. inaug. med.', 'Diss. iur.', a title that starts
      -- 'Diss.' or 'Dissertatio'. 26,274 works on night 2026-11-21's openalex_works (23,147 book, 3,127 article). Bar 21 Nov: a
      -- fresh random 55 of the hits, 54 dissertations or academic disputations (one a faculty programme announcing one); a bare
      -- Latin 'dissertatio' elsewhere in the title (letters, treatises) and 'Habilitationsschrift' failed that draw and are left
      -- out. Only when the MDZ record leads the sort; never for a work #2209 hands to its journal copy (jcp), never over a type
      -- curation (ctc). Revert: delete this WHEN (keep the ctc join below: other type lines read it too).
      WHEN s.type IN ('article', 'book') AND jcp.work_id IS NULL AND ctc.ctc_work_id IS NULL
           AND GET(s.locations_sorted, 0).landing_page_url LIKE '%bvb:12-bsb%'
           AND s.title RLIKE '(?i)inaug(ural)?[ .-]*diss|\\bdiss(ertatio)?\\.? *(inaug|acad|med|iur|jur|phil|theol|polit|hist|phys|math|chem|bot|anat)|^ *diss(ertatio)?\\b'
      THEN 'dissertation'
      -- oxjob #1160 (ZD 5612, 8486; PLAN step 3): MAG crawls of CiNii Books pages (ci.nii.ac.jp/ncid/<NCID>) typed book or
      -- article whose CiNii record's note says the item is a thesis: 'Thesis (Ph. D.)--Yale University, 2006', 'Thesis
      -- (doctoral)--Universität Leipzig, 1903', 'Facsim. reprint of the author's thesis (Ph. D.)--...' (a UMI copy of the
      -- thesis itself). Rows: platform 'cinii' of record_declared_types (rdt). ~113K works on night 2026-11-22's openalex_works
      -- (83K book, 30K article). Bars: 21 Nov a random 50, 49 theses (the miss, 'Based on the author's thesis', is excluded);
      -- 22 Nov a fresh random 30 of the final rows, 30 theses. Unlike the generic rdt line below it also re-types book, and
      -- never over a type curation (ctc) or for a work #2209 hands to its journal copy (jcp). Revert: delete this WHEN (or
      -- the platform's rows).
      WHEN rdt.platform = 'cinii'
      THEN CASE WHEN s.type IN ('article', 'book') AND jcp.work_id IS NULL AND ctc.ctc_work_id IS NULL
                THEN rdt.type ELSE s.type END
      -- oxjob #2236: the record leading the sort says only 'article' (mostly a MAG crawl of a repository or catalogue page,
      -- which falls to the type cascade's default), but its own host declares another type: a row of
      -- openalex.works.record_declared_types (rdt; one platform per job, notebooks/maintenance/BuildRecordDeclaredTypes.sql).
      -- First platform: DiVA (diva-portal.org, ~50 Swedish universities), whose own publicationTypeCode says student and
      -- doctoral theses, conference papers, chapters, reports (~0.3M works served as article; bar 14 Nov: a fresh random 100
      -- vs DiVA's own record). Never for a work #2209 hands to its journal copy (jcp). Type only. Revert one platform: delete
      -- its rows. Revert the rule: delete this WHEN, the rdt join below and BROADCAST(rdt) in the hint.
      WHEN s.type = 'article' AND jcp.work_id IS NULL AND rdt.type IS NOT NULL
      THEN rdt.type
      -- oxjob #1160 (ZD 5175; PLAN step 4): Spanish journal items served as article whose title is a book citation or bare
      -- issue front matter. A book review is titled with the reviewed book's citation: 'ISBN' in the title (not at its
      -- start: some conference papers open with their proceedings' ISBN), or a title that ends with the page count
      -- (', 556 pp.', '. 302 págs.', ', 379 p.'; no leading zero, so Hispana's newspaper pages '0387 P.' stay out). A title
      -- that is only 'Introducción', 'Prólogo', 'Prefacio', 'Presentación' or 'Nota editorial' is the issue's editorial
      -- (an issue's 'Presentación' is editorial, ZD 5175, 16 Nov 2026), but only in a journal record (Crossref
      -- journal-article, an OJS /article/view/ page, Dialnet oaiart, Redalyc, DOAJ): in repositories it is often a book's
      -- prologue. 'Nota editorial' is editorial anywhere. Night 2026-11-22's openalex_works: 10,944 works to book-review,
      -- 4,983 to editorial. Bars 22 Nov: a random 50 per class chose these guards; a fresh draw read 30/30 ISBN titles and
      -- 49/50 page-count titles as book reviews, ~47/50 bare titles as front matter (the misses: introductions in
      -- monograph-series volumes). Not when a DataCite record leads (a ResearchGate upload of the booklet itself failed),
      -- never over a type curation (ctc). Unlike the URL lines above, a work #2209 hands to its journal copy (jcp) is in: the
      -- title decides here, not the leading record, and jcp means the work has its journal's own copy. Revert: delete these two WHENs.
      WHEN s.type = 'article' AND ctc.ctc_work_id IS NULL AND s.language = 'es'
           AND COALESCE(GET(s.locations_sorted, 0).provenance, '') <> 'datacite'
           AND ((lower(trim(s.title)) LIKE '%isbn%' AND lower(trim(s.title)) NOT RLIKE '^\\W*isbn')
                OR lower(trim(s.title)) RLIKE '[,.] *[1-9][0-9]{1,3} ?(pp|págs?|páginas|p)\\.?$')
      THEN 'book-review'
      WHEN s.type = 'article' AND ctc.ctc_work_id IS NULL AND s.language = 'es'
           AND COALESCE(GET(s.locations_sorted, 0).provenance, '') <> 'datacite'
           AND regexp_replace(lower(trim(s.title)), '[.:\\s]+$', '') IN ('introducción', 'introduccion', 'prólogo', 'prologo',
                 'prefacio', 'presentación', 'presentacion', 'nota editorial')
           AND (regexp_replace(lower(trim(s.title)), '[.:\\s]+$', '') = 'nota editorial' OR jcp.work_id IS NOT NULL
                OR (GET(s.locations_sorted, 0).provenance = 'crossref' AND GET(s.locations_sorted, 0).raw_type = 'journal-article')
                OR GET(s.locations_sorted, 0).landing_page_url RLIKE '/article/view/|dialnet\\.unirioja\\.es/servlet/oaiart|redalyc\\.org/articulo\\.oa|doaj\\.org/article/')
      THEN 'editorial'
      -- oxjob #2526: a repository or MAG copy that only says 'article' leads, but the work's own DOI is a DataCite DOI whose
      -- DataCite record declares a Dataset or Software (the DOI owner's own type; its DataCite location is typed so). The OSTI
      -- OAI copies of Materials Project and other DOE data, the DANS copies of Mendeley Data and Dryad packages lead their
      -- works and carry no type of their own: ~296K works served article (~79K outside xpac; 22 Nov 2026). Last, so every
      -- rule above keeps its work; never for a journal-hosted lead or a journal copy #2209 picks (jcp), never over a type
      -- curation (ctc). #2138's work-level rule (set_fields) does the same for dissertations and arXiv preprints. Prefixes
      -- left out on judged draws: Zenodo 10.5281 and GMS 10.3205 (as #2138), DANS 10.17026 (archaeology deposits are
      -- excavation reports), ZORA 10.5167 (essays), OSF 10.17605 (projects; worktype charter), University of Salento 10.1285
      -- (journal articles), figshare 10.6084 (papers, posters), Purdue PURR 10.4231 (lecture recordings), NETL EDX 10.18141
      -- (tools, manuals). Book, chapter and report fell short on their draws and are left out. Bar 22 Nov: a fresh random 60
      -- datasets 59/60 and 20 software 20/20 right by the DataCite record and the item. Revert: delete this WHEN.
      WHEN s.type = 'article' AND jcp.work_id IS NULL AND ctc.ctc_work_id IS NULL
           AND GET(s.locations_sorted, 0).provenance IN ('repo', 'repo_backfill', 'mag')
           AND COALESCE(GET(s.locations_sorted, 0).source.type, '') <> 'journal'
           AND split_part(LOWER(s.raw_best_doi), '/', 1) NOT IN ('10.5281', '10.3205', '10.17026', '10.5167', '10.17605', '10.1285', '10.6084', '10.4231', '10.18141')
           AND EXISTS(s.locations_sorted, y -> y.provenance = 'datacite' AND LOWER(y.native_id) = LOWER(s.raw_best_doi)
                                               AND y.location_type IN ('dataset', 'software'))
      THEN GET(FILTER(s.locations_sorted, y -> y.provenance = 'datacite' AND LOWER(y.native_id) = LOWER(s.raw_best_doi)
                                               AND y.location_type IN ('dataset', 'software')), 0).location_type
      ELSE s.type
    END AS type,
    -- type_crossref is deprecated and no longer served by the API; keep NULL
    -- until the column is dropped (raw Crossref type lives on the crossref
    -- location's raw_type if ever needed).
    CAST(NULL AS STRING) AS type_crossref,
    -- oxjob #1056 (ZD 8841): the same test as type above (no other WHEN there yields paratext but the rdt lines),
    -- so type:paratext and is_paratext agree. Before: s.type = 'paratext' alone, which left the class rule's 4,110 works
    -- type paratext with is_paratext false (night 2026-11-16).
    -- oxjob #2574: record_declared_types rows of type paratext (platform sciencedirect: items ScienceDirect labels Contents
    -- list, Editorial Board or Index, ~36K works) type a work paratext through the generic rdt WHEN, so they set is_paratext
    -- too. Same guards as that WHEN and the lines above it that can take an article first (CiNii, preprint servers, the
    -- uha news DOIs); the Dialnet and MDZ lines key on their own hosts' records, which hold no paratext rows. Every term is
    -- COALESCEd so a work with no rdt row keeps FALSE, never NULL. Revert: delete the third OR.
    (s.type = 'paratext' OR (s.type IN ('article', 'conference-paper') AND s.is_paratext_title_class)
     OR COALESCE(s.type = 'article' AND jcp.work_id IS NULL AND COALESCE(rdt.type, '') = 'paratext'
                 AND COALESCE(rdt.platform, '') <> 'cinii' AND NOT COALESCE(pps.is_preprint_repository, FALSE)
                 AND NOT LOWER(COALESCE(s.raw_best_doi, '')) LIKE '10.32739/uha.id.%', FALSE)) AS is_paratext,
    -- oxjob #1500 (Kyle 2026-10-02): a retraction notice is not retracted. Hindawi and MDPI title their notices "Retracted:
    -- <paper title>", so a work only the title rule flags loses the flag when its DOI is a retraction notice for another
    -- record and nothing retracts it (rn below). ~13.2K works on 2026-11-15. Revert: s.is_retracted, and drop the rn join.
    s.is_retracted AND NOT (s.is_retracted_title_only AND rn.rn_doi IS NOT NULL) AS is_retracted,
    -- oxjob #2264: pulled = retracted, withdrawn or removed by the publisher, host or authors (plans/retractions.md § 5).
    -- Three signals: is_retracted as served (curation, records, title rule, notices cleared); the work's own Crossref
    -- record declaring a withdrawal or removal (cps, cell above); a withdrawal or removal stamp in the title ("WITHDRAWN:
    -- <title>", "Withdrawn", "[Withdrawn] <title>", "Withdrawn article ...", "REMOVED: <title>", "Removed due to policy
    -- violations"), unless the work's DOI is a notice about another record. Left out on purpose: "Withdrawal: <title>"
    -- (JBC and AIAA title their notices so), "Withdrawn paper: <citation>" (notices), title-case "Removed: <title>" (a
    -- real title, "Removed: a novel"; publishers stamp in capitals). ~26.8K works newly pulled, ~118.7K in all; sizes and
    -- the 50-per-signal check in oxjob #2264 EXPLORE.md. Not hidden anywhere yet. Revert: drop this column, the cps
    -- join and its hint, and the crossref_pull_signals cell (grep 2264).
    -- oxjob #2264 (17 Nov, Cochrane gate): is_retracted pulls a work unless retraction_pull_exemptions (cell above) leaves
    -- it out (retract-and-replace, out of date, a Cochrane withdrawal with no problem reason; a Cochrane DOI with no row
    -- is left out too), except that a curated is_retracted or a "Retracted:" title keeps it pulled. ~759 works fewer.
    -- Revert: this term back to its first line, and drop the rwx join, its hint and the retraction_pull_exemptions cell.
    -- oxjob #1186 (18 Nov 2026): a curation (users-api POST /curations, entity works, property is_pulled, true/false;
    -- SyncWorkFieldCurations) wins over every signal below, as the is_retracted curation does for is_retracted: true
    -- for a withdrawal no source carries (ZD 26740: a publisher's withdrawal letter, nothing in Crossref), false for a
    -- wrong stamp (W2147670638, a database named WITHDRAWN). 0 curations when added, so 0 re-stamps. Revert: drop the
    -- COALESCE( line and its closing paren before AS is_pulled, is_pulled_curated, wcf_is_pulled and the view column.
    COALESCE(s.is_pulled_curated,
    (s.is_retracted AND NOT (s.is_retracted_title_only AND rn.rn_doi IS NOT NULL)
      AND (COALESCE(s.is_retracted_curated, FALSE)
           OR COALESCE(LOWER(s.title) LIKE 'retracted article%' OR LOWER(s.title) LIKE 'retracted: %', FALSE)
           OR NOT COALESCE(rwx.leaves_out,
                           regexp_replace(lower(COALESCE(s.raw_best_doi, '')), '[^a-zA-Z0-9./-]', '') LIKE '10.1002/14651858%')))
      OR COALESCE(cps.is_pulled_record, FALSE)
      OR (COALESCE(
            -- a bare word as the whole title; never a reference entry, where it is a dictionary headword
            -- (W4415863565 'removed'). 'withdraw': EWA's and others' withdrawn papers (#2264, 17 Nov)
            (LOWER(TRIM(s.title)) IN ('withdrawn', 'withdrawn.', '[withdrawn]', '(withdrawn)',
                                      'removed', 'removed.', '[removed]', '(removed)', 'withdraw', 'withdraw.')
              AND COALESCE(s.type, '') <> 'reference-entry')
            OR LOWER(TRIM(s.title)) RLIKE '^withdrawn\\s*[:\\-–—]'
            OR LOWER(TRIM(s.title)) RLIKE '^[\\[(](withdrawn|removed)[\\])]'
            OR LOWER(TRIM(s.title)) RLIKE '^withdrawn (article|preprint|abstract|manuscript|chapter)'
            OR TRIM(s.title) RLIKE '^REMOVED\\s*[:\\-–—]'
            OR LOWER(TRIM(s.title)) LIKE 'removed due to policy violation%'
            -- oxjob #2264 (17 Nov 2026): stamps the RetractBASE comparison found (#1771 EXPLORE.md § RetractBASE
            -- comparison), each read by hand on 50 random works (#2264 EXPLORE.md § More title stamps). ~14K works:
            -- withdrawn conference abstracts ('Abstract withdrawn', 'This abstract has been withdrawn', '393EMF WITHDRAWN',
            -- 'P2-109: Withdrawn', '<title> (WITHDRAWN)', but not Cochrane's
            -- reviews withdrawn as out of date (plans/retractions.md § 5 decision 1)); '[RETRACTED] <title>' (protocols.io pages
            -- the host pulled, Zenodo records their authors retracted), except Spandidos (10.3892), whose '[Retracted]'
            -- works are its retraction notices; Retos' 'Article RETRACTED due to ...'; and IEEE's in-place stamps on its own
            -- DOIs, 'Notice of Retraction: <title>' and 'Notice of Removal: <title>' (on other prefixes these are mostly
            -- notices, so they stay out). The notice exclusion below applies to all of them.
            OR LOWER(TRIM(s.title)) RLIKE '^(abstract withdrawn|this abstract (has been|was|is) withdrawn)'
            OR LOWER(TRIM(s.title)) RLIKE '^[a-z]{0,6}[0-9][a-z0-9.\\-]*\\s*[:\\-–—.]?\\s*withdrawn\\.?$'
            OR (LOWER(TRIM(s.title)) RLIKE '[\\[(]withdrawn[\\])]\\.?$' AND regexp_replace(lower(COALESCE(s.raw_best_doi, '')), '[^a-zA-Z0-9./-]', '') NOT LIKE '10.1002/14651858%')
            OR (LOWER(TRIM(s.title)) RLIKE '^[\\[(]retracted[\\])]' AND regexp_replace(lower(COALESCE(s.raw_best_doi, '')), '[^a-zA-Z0-9./-]', '') NOT LIKE '10.3892/%')
            OR LOWER(TRIM(s.title)) RLIKE '^article retracted due to'
            OR (regexp_replace(lower(COALESCE(s.raw_best_doi, '')), '[^a-zA-Z0-9./-]', '') LIKE '10.1109/%' AND LOWER(TRIM(s.title)) RLIKE '^notice of (retraction|removal)\\b'
                AND LOWER(TRIM(s.title)) NOT RLIKE '^notice of (retraction|removal)[\\s.:]*$'), FALSE)
          AND NOT COALESCE(cps.is_pull_notice, FALSE) AND rn.rn_doi IS NULL)
      -- oxjob #2102 (ZD 27233): an arXiv preprint its authors or arXiv withdrew (arxiv_withdrawals: DataCite's Withdrawn date
      -- at the current version, built by CreateSuperLocations), on the work's own DOI only, so a journal article or another
      -- preprint whose arXiv copy was withdrawn keeps its own DOI and is not pulled; nor is a work with a journal,
      -- conference or book-series copy beside it (MAG's venue guesses aside). ~14.2K works on 17 Nov 2026.
      -- Revert: drop this term, the awd join and its hint (grep 2102).
      OR (awd.awd_doi IS NOT NULL
          AND NOT COALESCE(EXISTS(s.locations_sorted, x -> COALESCE(x.provenance <> 'mag'
                AND x.source.type IN ('journal', 'conference', 'book series'), FALSE)), FALSE))
      -- oxjob #2264 / #1186: a Copernicus meeting abstract (EGU, EMS, EPSC...) its organisers withdrew: the abstract's own
      -- page says so and nothing else does (copernicus_abstract_withdrawals, loaded from a checked crawl). On the work's
      -- own DOI. Revert: drop this term, the caw join and its hint.
      OR caw.caw_doi IS NOT NULL) AS is_pulled,
    s.indexed_in_crossref,
    s.is_xpac,
    m.mesh_formatted AS mesh,
    CAST(NULL AS STRING) AS fulltext
  FROM
    set_fields s
    LEFT JOIN (
      SELECT id AS preprint_src_id, is_preprint_repository, apc_usd_by_year, apc_prices_by_year
      FROM openalex.sources.sources
    ) pps
      ON pps.preprint_src_id = TRY_CAST(REGEXP_EXTRACT(GET(s.locations_sorted, 0).source.id, '/S([0-9]+)', 1) AS BIGINT)
    LEFT JOIN openapc_paid o ON s.id = o.work_id
    -- oxjob #1920: MeSH from each PMID's latest PubMed record, built in the cell above.
    LEFT JOIN openalex.works.pmid_mesh_latest_record m ON s.ids.pmid = m.pmid
    LEFT JOIN (
      SELECT work_id,
        COLLECT_LIST(NAMED_STRUCT('volume_work_id', volume_work_id, 'volume_display_name', volume_display_name,
                                  'native_id', native_id)) AS volumes
      FROM openalex.works.work_volumes
      GROUP BY work_id
    ) wv ON wv.work_id = s.id
    -- oxjob #1862: the PMCID the work's own PubMed record states (ArticleIdList, IdType pmc), keyed by the PMID the
    -- work serves, so ids.pmid and ids.pmcid name one NCBI record (checked against NCBI's PMC-ids.csv: 99.995% agree).
    -- PMC123.2 keeps its digits; non-PMC values (wst_2023_002) are dropped. GROUP BY keeps the join one row per PMID.
    LEFT JOIN (
      SELECT native_id AS pmcid_pmid,
        MIN(NULLIF(REGEXP_EXTRACT(UPPER(TRIM(GET(FILTER(ids, x -> x.namespace = 'pmcid'), 0).id)),
                                  '^PMC([0-9]+)([.][0-9]+)?$', 1), '')) AS pmcid_digits
      FROM openalex.pubmed.pubmed_works
      WHERE EXISTS(ids, x -> x.namespace = 'pmcid')
      GROUP BY native_id
    ) pmc ON pmc.pmcid_pmid = s.ids['pmid']
    -- oxjob #1163: the work's PMC copies with NCBI's ids (cell above, ~0.9M rows, broadcast: hint above); the ids build
    -- picks one only where NCBI agrees
    LEFT JOIN openalex.works.pmc_copy_ncbi_ids pcn ON pcn.work_id = s.id
    -- oxjob #2209: works whose journal copy another record on the work names (cell above, ~2.8M rows of two columns,
    -- broadcast: hint above); read by primary_location and locations, and by #2227's Dialnet type lines (skipped there)
    LEFT JOIN openalex.works.journal_copy_primary_picks jcp ON jcp.work_id = s.id
    -- oxjob #1225: works whose institution-portal copy of a Figshare item should lead (cell above, ~0.4M rows of two small
    -- arrays, broadcast: hint above); read only by primary_location and the locations reorder
    LEFT JOIN openalex.works.figshare_portal_primary_picks fpp ON fpp.work_id = s.id
    -- oxjob #1160: works with a type curation (one row per work, ~200 rows on 21 Nov 2026, broadcast: hint above); read only by
    -- the type CASE lines that must never beat a curated type: the MDZ and CiNii dissertation lines, #1160's Spanish book
    -- reviews and bare front matter (ZD 5175) and #2526's DataCite datasets; drop it and BROADCAST(ctc) only with the last one
    LEFT JOIN (SELECT work_id AS ctc_work_id FROM openalex.works.work_curated_fields WHERE type IS NOT NULL) ctc ON ctc.ctc_work_id = s.id
    -- oxjob #2236: the type the leading record's own host declares (~0.3M rows, broadcast: hint above); read only by the type CASE
    LEFT JOIN openalex.works.record_declared_types rdt
      ON rdt.provenance = GET(s.locations_sorted, 0).provenance AND rdt.native_id = GET(s.locations_sorted, 0).native_id
    -- oxjob #1910: ~28K rows, broadcast (hint above); only titles holding a control character can match
    LEFT JOIN openalex.works.work_title_control_fixes tcf ON tcf.raw_title = s.title
    -- oxjob #1500: DOIs of retraction notices that nothing retracts, one row each (notice_link_pairs is rebuilt nightly by
    -- Map_Work_Ids; ~0.1M rows, broadcast: hint above). Cleaned the way notice_link_pairs cleans DOIs.
    LEFT JOIN (
      SELECT DISTINCT n.notice_id AS rn_doi
      FROM openalex.works.notice_link_pairs n
      LEFT ANTI JOIN (
        SELECT DISTINCT target_id FROM openalex.works.notice_link_pairs WHERE target_ns = 'doi' AND notice_type = 'retraction'
      ) rt ON rt.target_id = n.notice_id
      WHERE n.notice_ns = 'doi' AND n.notice_type = 'retraction'
    ) rn ON rn.rn_doi = regexp_replace(lower(s.raw_best_doi), '[^a-zA-Z0-9./-]', '')
    -- oxjob #2264: the work's Crossref pull signals (crossref_pull_signals, cell above; ~59K rows, broadcast: hint
    -- above), keyed on the DOI cleaned the same way as rn's
    LEFT JOIN openalex.works.crossref_pull_signals cps
      ON cps.doi = regexp_replace(lower(s.raw_best_doi), '[^a-zA-Z0-9./-]', '')
    -- oxjob #2264: retractions that don't pull the work (retraction_pull_exemptions, cell above; ~1.3K rows, broadcast)
    -- Its DOI column is renamed in a subquery: a second FROM column named doi made topics_key's bare doi ambiguous
    -- (AMBIGUOUS_REFERENCE cps.doi / rwx.doi: the statement failed to plan). topics_key now names s.best_doi.
    LEFT JOIN (SELECT doi AS rwx_doi, leaves_out FROM openalex.works.retraction_pull_exemptions) rwx
      ON rwx.rwx_doi = regexp_replace(lower(s.raw_best_doi), '[^a-zA-Z0-9./-]', '')
    -- oxjob #2102: withdrawn arXiv papers (arxiv_withdrawals, built by CreateSuperLocations; ~17K rows, broadcast: hint
    -- above). Its DOI column is renamed in a subquery, like rwx's, so no FROM column named doi meets a bare doi.
    LEFT JOIN (SELECT doi AS awd_doi FROM openalex.works.arxiv_withdrawals) awd
      ON awd.awd_doi = regexp_replace(lower(s.raw_best_doi), '[^a-zA-Z0-9./-]', '')
    -- oxjob #2264 / #1186: withdrawn Copernicus meeting abstracts (~4K rows, broadcast: hint above); doi renamed like rwx's
    LEFT JOIN (SELECT doi AS caw_doi FROM openalex.works.copernicus_abstract_withdrawals) caw
      ON caw.caw_doi = regexp_replace(lower(s.raw_best_doi), '[^a-zA-Z0-9./-]', '')
    -- oxjob #1098: Crossref article numbers (cell above); read only by biblio.first_page
    LEFT JOIN openalex.works.crossref_article_numbers crn ON crn.work_id = s.id
    -- oxjob #1098 step 2c: volume and issue from the landing pages of the work's repository records (cell above); read only by biblio
    LEFT JOIN openalex.works.repo_page_volumes rpv ON rpv.work_id = s.id
  WHERE s.id > 0
);